# BotV1.4.3 — Outreach + Gate horario + Similitud productos + Extractor Acumulativo + Coalescencia inbound + Robustez tras reinicio

**Cambios v1.4.3 sobre v1.4.2 — robustez tras reinicio del kernel:**
- ✅ `guardar_mensaje_usuario` ahora hace SELECT de verificación post-commit. Si la BD no muestra la fila tras commit (caso de conexiones zombi en el pool tras reinicio), lanza excepción explícita y el retry hace su trabajo. Cierra el bug 'UUID en memoria pero fila inexistente en BD'.
- ✅ `actualizar_respuesta_ia` ahora acepta fallbacks (`mensaje_usuario_fallback`, `id_ferreteria_fallback`). Si la fila no existe, hace INSERT con el MISMO UUID en lugar de crear duplicado. Elimina los warnings 'interacción no existe; fallback a INSERT' y evita filas zombi con UUID nuevo.
- ✅ Nuevo `DatabaseManager.obtener_inbounds_pendientes(ventana_minutos)`: lista filas con `respuesta_ia=NULL` recientes (excluyendo outreach-templates).
- ✅ Nuevo `DatabaseManager.marcar_inbound_perdido(id)`: marca con placeholder `[RESPUESTA PERDIDA POR REINICIO DEL BOT]` para que dejen de aparecer NULL en reportes.
- ✅ Nuevo `MessageProcessor.rehidratar_inbounds_huerfanos(...)`: al arranque, busca mensajes inbound de los últimos 15 min sin respuesta y los reagenda en el dispatcher (el cliente SÍ recibirá respuesta, con algo de retraso por el reinicio). Los de 15-120 min los marca como perdidos.
- ✅ Bootstrap llama `rehidratar_inbounds_huerfanos` justo tras `dispatcher.start()`, antes de levantar el servidor Flask. Falla aislada: si la rehidratación lanza, no bloquea el arranque del bot.

**Cambios v1.4.2 sobre v1.4.1 — coalescencia inbound:**
- ✅ Si la ferretería envía varios mensajes seguidos, el bot espera una ventana de "escucha activa" (`listen_window_*`, default 2–5 min) y los reúne en UNA sola respuesta. Cada mensaje nuevo REINICIA el timer (debounce).
- ✅ La respuesta sale inmediatamente al vencer el timer (sin doble delay). Más humano y ahorra tokens.
- ✅ Nuevos métodos `DatabaseManager.guardar_mensaje_usuario` y `actualizar_respuesta_ia`.
- ✅ `MessageDispatcher` con tres `kind`s de payload: `text`, `template`, `inbound_intent`.
- ✅ `MessageProcessor` con buffer in-memory por recipient (`_pending_messages`).
- ✅ Adjuntos (imagen/PDF) NO se coalescen.
- ✅ Outreach NO se coalesce. Usa `outreach_delay_*`.

**Cambios v1.1 (heredados):**
- ✅ `_handle_ai_flow` polimórfico (`outreach` / `inbound`).
- ✅ `OperatingHoursGate`: ventana horaria por día.
- ✅ Webhook entrante: descarta fuera de ventana.
- ✅ Cron de outreach: aborta fuera de ventana.
- ✅ Dispatcher: requeue de envíos fuera de ventana.
- ✅ `BroadcastScheduler` delega en `processor._handle_ai_flow(modo="outreach")`.

**Heredado de versiones anteriores:**
- SDK `anthropic`, modelos `claude-opus-4-7`, `claude-opus-4-6`, `claude-sonnet-4-6`, `claude-haiku-4-5-20251001`
- Estructura IA: BASE + REGION + ESTADO + HISTORIAL → CONTEXTO DINÁMICO → PROMPT FINAL
- Idempotencia persistente del webhook (tabla `webhook_events_processed`)
- Extracción de cotizaciones desde texto, PDF e imagen


## CELDA 1: INSTALACIÓN DE DEPENDENCIAS

In [ ]:
!pip install requests flask flask-cors pyngrok anthropic sqlalchemy psycopg2-binary python-dotenv apscheduler pymc scikit-learn pandas numpy fuzzywuzzy python-Levenshtein rapidfuzz -q
print("✅ Dependencias instaladas correctamente")

## CELDA 2: CARGAR SECRETOS DE COLAB

In [ ]:
from google.colab import userdata
import os

SECRETS = {
    'WA_TOKEN': userdata.get('WA_TOKEN'),
    'WA_PHONE_ID': userdata.get('WA_PHONE_ID'),
    'WEBHOOK_VERIFY_TOKEN': userdata.get('WEBHOOK_VERIFY_TOKEN'),
    'ANTHROPIC_API_KEY': userdata.get('ANTHROPIC_API_KEY'),  # ✅ NUEVO: Reemplaza GOOGLE_API_KEY
    'NGROK_AUTH_TOKEN': userdata.get('NGROK_AUTH_TOKEN'),
    'DB_USER': userdata.get('DB_USER'),
    'DB_PASSWORD': userdata.get('DB_PASSWORD'),
    'DB_HOST': userdata.get('DB_HOST'),
    'DB_NAME': userdata.get('DB_NAME'),
}

missing = [k for k, v in SECRETS.items() if not v]
if missing:
    raise RuntimeError(f"❌ Secretos faltantes: {missing}")

# Parsear host y puerto
if ':' in SECRETS['DB_HOST']:
    db_host, db_port = SECRETS['DB_HOST'].split(':')
else:
    db_host = SECRETS['DB_HOST']
    db_port = '5432'

DB_HOST_WITH_PORT = f"{db_host}:{db_port}"
print("✅ Todos los secretos cargados correctamente")

## CELDA 3: IMPORTACIONES Y CONFIGURACIÓN DE LOGGING

In [ ]:
import os
import json
import logging
import time
import uuid
import unicodedata
import re
import xml.etree.ElementTree as ET
from typing import Dict, Optional, List, Tuple, Any
from dataclasses import dataclass
from enum import Enum
from functools import wraps
from pathlib import Path
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from contextlib import contextmanager

import requests
import psycopg2
from flask import Flask, request, jsonify
from pyngrok import ngrok
from google.colab import drive
import anthropic  # ✅ Cliente de Anthropic
from sqlalchemy import (
    create_engine, Column, Integer, String, DateTime, func, Text,
    Boolean, Numeric, ForeignKey, CheckConstraint, Enum as SQLEnum
)
from sqlalchemy.dialects.postgresql import UUID, ARRAY
from sqlalchemy.orm.attributes import flag_modified
from sqlalchemy.orm import sessionmaker, declarative_base, relationship

# =============================================================================
# LOGGING con filtro PII
#
# ✅ FIX 2.1: las regex anteriores tenían doble escape (`\\d`, `\\+`) que las
# convertía en cadenas literales en lugar de metacaracteres, así que NUNCA
# matcheaban. Ahora los teléfonos sí se enmascaran en los logs.
# =============================================================================
def mask_phone(phone: str) -> str:
    """Enmascara un teléfono: +573001234567 -> +57***4567"""
    if not phone:
        return "***"
    phone = str(phone)
    if len(phone) < 6:
        return "***"
    return f"{phone[:3]}***{phone[-4:]}"


class PIIFilter(logging.Filter):
    """Enmascara tokens, secrets y números de teléfono en los logs."""
    # Bearer tokens largos
    _TOKEN_RE = re.compile(r'(Bearer\s+)([A-Za-z0-9_\-\.]{20,})')
    # Teléfonos: 10-15 dígitos, opcionalmente con + delante, con boundary numérica
    _PHONE_RE = re.compile(r'(?<!\d)(\+?\d{10,15})(?!\d)')

    def filter(self, record: logging.LogRecord) -> bool:
        if isinstance(record.msg, str):
            msg = record.msg
            msg = self._TOKEN_RE.sub(r'\1***REDACTED***', msg)
            msg = self._PHONE_RE.sub(lambda m: mask_phone(m.group(1)), msg)
            record.msg = msg
        # También sanear args si vienen
        if record.args:
            try:
                new_args = []
                for a in record.args:
                    if isinstance(a, str):
                        a = self._TOKEN_RE.sub(r'\1***REDACTED***', a)
                        a = self._PHONE_RE.sub(lambda m: mask_phone(m.group(1)), a)
                    new_args.append(a)
                record.args = tuple(new_args)
            except Exception:
                pass
        return True


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger.addFilter(PIIFilter())
logging.getLogger().addFilter(PIIFilter())

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

print("✅ Todas las librerías importadas correctamente")


## CELDA 4: CLASES DE CONFIGURACIÓN Y UTILIDADES

In [ ]:
class MessageType(Enum):
    TEXT = "text"
    TEMPLATE = "template"
    IMAGE = "image"
    DOCUMENT = "document"

SUPPORTED_MESSAGE_TYPES = {"text"}
MAX_USER_MESSAGE_LENGTH = 1000

@dataclass
class WhatsAppConfig:
    token: str
    phone_id: str
    verify_token: str
    api_version: str = "v17.0"

    @property
    def messages_url(self) -> str:
        return f"https://graph.facebook.com/{self.api_version}/{self.phone_id}/messages"

    def validate(self) -> None:
        if not self.token or not self.phone_id:
            raise ValueError("WA_TOKEN y WA_PHONE_ID son obligatorios")

# ✅ Config Anthropic. Modelos vigentes (abril 2026):
#    - claude-opus-4-7              (más capaz)
#    - claude-opus-4-6
#    - claude-sonnet-4-6            (mejor relación calidad/costo, recomendado)
#    - claude-haiku-4-5-20251001    (más rápido y económico)
@dataclass
class AnthropicConfig:
    api_key: str
    model_name: str = "claude-sonnet-4-6"
    max_tokens: int = 120
    temperature: float = 0.7
    typo_rate: float = 0.012  # Probabilidad por carácter de error de dedo
    history_limit: int = 10   # ✅ NUEVO: cuántas interacciones recientes inyectar como contexto
    # ✅ v1.3: configuración de plantilla Meta para primer contacto (regla 24h).
    # `outreach_template_name` debe coincidir con el nombre EXACTO aprobado en
    # Meta Business Manager. `outreach_template_lang` es el código de idioma
    # también aprobado (ej. "es_CO", "es", "es_MX"). El bot no los puede inventar:
    # si no coinciden con la plantilla aprobada, Meta rechaza el envío.
    outreach_template_name: str = "saludo"
    outreach_template_lang: str = "es_CO"
    # Tope de tokens para generar SOLO el parámetro {{1}} (frase corta).
    # Es independiente de max_tokens (que aplica al chat conversacional).
    outreach_param_max_tokens: int = 60

    def validate(self) -> None:
        if not self.api_key:
            raise ValueError("ANTHROPIC_API_KEY es obligatorio")
        if not self.outreach_template_name:
            raise ValueError("outreach_template_name no puede estar vacío")
        if not self.outreach_template_lang:
            raise ValueError("outreach_template_lang no puede estar vacío")


# ✅ NUEVO: configuración de delays humanos + anti-bloqueo Meta.
# ✅ v1.4.2: rediseño de delays inbound. Ahora hay UN solo tiempo de
# "escucha activa" (listen_window_*) durante el cual el bot espera a ver si
# llegan más mensajes del MISMO cliente. Si llega uno nuevo, el timer se
# reinicia (debounce). Cuando el timer expira sin nuevos mensajes, el bot
# genera la respuesta y la envía INMEDIATAMENTE (la generación toma 2-8s,
# despreciable). Esto simula mejor a una persona: lee, espera por si el
# cliente sigue escribiendo, y cuando termina, responde.
@dataclass
class DispatcherConfig:
    """
    - listen_window_*: rango (segundos) de espera de NUEVOS mensajes del
      mismo cliente antes de generar la respuesta. Cada mensaje entrante
      del mismo número REINICIA el timer (debounce). Default: 120..300
      (2..5 min). Tras expirar, el envío es inmediato — la "sensación
      humana" la da este tiempo de escucha, no un delay extra al final.
    - outreach_delay_*: rango (segundos) que se respeta ANTES del primer
      envío de outreach (saludo proactivo). El outreach NO se coalesce —
      es proactivo, no responde a nada — pero sí queremos un delay
      aleatorio para no enviar a todas las ferreterías en el mismo
      segundo cuando dispara el cron. Default: 60..300 (1..5 min).
    - inter_chat_*: rango (segundos) que separa dos envíos físicos
      consecutivos a números DISTINTOS. Anti-bloqueo Meta.
      Default: 120..300 (2..5 min).
    """
    listen_window_min_s: int = 120
    listen_window_max_s: int = 300
    outreach_delay_min_s: int = 60
    outreach_delay_max_s: int = 300
    inter_chat_min_s: int = 120
    inter_chat_max_s: int = 300

    def validate(self) -> None:
        if not (0 <= self.listen_window_min_s <= self.listen_window_max_s):
            raise ValueError("listen_window_min/max inválidos")
        if not (0 <= self.outreach_delay_min_s <= self.outreach_delay_max_s):
            raise ValueError("outreach_delay_min/max inválidos")
        if not (0 <= self.inter_chat_min_s <= self.inter_chat_max_s):
            raise ValueError("inter_chat_min/max inválidos")


# =============================================================================
# ✅ NUEVO (v1.1): OperatingHoursGate
# Ventana horaria de operación del bot, configurable por día de la semana.
# Usado por:
#   - WebhookServer: descarta webhooks entrantes fuera de ventana (decisión 1a).
#   - BroadcastScheduler: el cron sigue agendado, pero el job aborta al inicio
#     si está fuera de ventana (decisión 2a+2c).
#   - MessageDispatcher: si el envío diferido caería fuera, hace requeue al
#     inicio de la próxima ventana (decisión 3a).
#
# ⚠️ ZONA HORARIA: este gate trabaja en hora local de BOGOTÁ (America/Bogota,
# UTC-5, sin DST). Esto es crítico porque Colab corre en UTC: si usáramos
# datetime.now() sin TZ, la ventana se desplazaría 5 horas. Las horas que
# pasa el usuario en `windows` se interpretan como hora LOCAL DE BOGOTÁ.
#
# Configuración: dict {weekday_int: (hora_apertura, hora_cierre)} donde
# weekday_int sigue la convención de Python (lunes=0 ... domingo=6).
# Si una clave no existe o vale None, ese día está cerrado.
# Las horas son enteros 0..24 (24 = medianoche siguiente, exclusivo).
# =============================================================================

# Zona horaria del bot. Bogotá no tiene DST, así que ZoneInfo es estable
# y equivale siempre a UTC-5.
BOGOTA_TZ = ZoneInfo("America/Bogota")


@dataclass
class OperatingHoursConfig:
    """
    Configura la ventana horaria del bot en hora LOCAL DE BOGOTÁ.
    `windows` mapea weekday (0=Lun..6=Dom) → (hora_apertura, hora_cierre)
    o None para indicar "cerrado todo el día".
    Ejemplo:
        OperatingHoursConfig(windows={
            0: (8, 18),   # lunes 08:00–18:00 Bogotá
            ...
            6: None,      # domingo cerrado
        })
    """
    windows: Dict[int, Optional[Tuple[int, int]]]

    def validate(self) -> None:
        for wd, w in self.windows.items():
            if not (0 <= wd <= 6):
                raise ValueError(f"weekday inválido: {wd}")
            if w is None:
                continue
            o, c = w
            if not (0 <= o < c <= 24):
                raise ValueError(f"ventana inválida en weekday {wd}: ({o}, {c})")


class OperatingHoursGate:
    """
    Decide si el bot está dentro de su ventana operativa.
    Trabaja SIEMPRE en hora local de Bogotá (America/Bogota, UTC-5).

    Métodos puros, sin estado mutable: seguros para usar desde varios threads.

    ⚠️ Convención: si se pasa un `now` naive (sin tzinfo), se ASUME que está
    en UTC (el caso normal en Colab donde datetime.now() devuelve hora del
    servidor en UTC) y se convierte a Bogotá antes de comparar. Si se pasa
    un `now` aware, se respeta su tzinfo y se convierte a Bogotá.
    """

    def __init__(self, config: OperatingHoursConfig,
                 tz: ZoneInfo = BOGOTA_TZ):
        self.config = config
        self.config.validate()
        self.tz = tz

    def _to_local(self, now: Optional[datetime]) -> datetime:
        """
        Normaliza `now` a hora local del gate (Bogotá por default).
        - None → datetime.now(tz=self.tz)  (siempre aware)
        - naive → asume UTC, convierte a self.tz
        - aware → convierte a self.tz
        Devuelve siempre un datetime AWARE en self.tz.
        """
        if now is None:
            return datetime.now(self.tz)
        if now.tzinfo is None:
            # Asumir UTC (caso típico en Colab/servidores)
            now = now.replace(tzinfo=timezone.utc)
        return now.astimezone(self.tz)

    def is_open(self, now: Optional[datetime] = None) -> bool:
        """¿El bot está dentro de la ventana operativa AHORA (hora Bogotá)?"""
        local = self._to_local(now)
        window = self.config.windows.get(local.weekday())
        if window is None:
            return False
        opens_h, closes_h = window
        # Comparar en minutos para precisión de minuto.
        local_minutes = local.hour * 60 + local.minute
        return (opens_h * 60) <= local_minutes < (closes_h * 60)

    def next_open(self, now: Optional[datetime] = None) -> datetime:
        """
        Devuelve el próximo datetime AWARE (en TZ Bogotá) en que la ventana
        se abre. Si AHORA ya está dentro, devuelve `now` convertido a Bogotá.
        Busca hasta 8 días en el futuro; si no encuentra, lanza error
        (configuración con todos los días cerrados, escenario inválido).
        """
        local = self._to_local(now)
        if self.is_open(local):
            return local

        cursor = local
        for _ in range(8):
            window = self.config.windows.get(cursor.weekday())
            if window is not None:
                opens_h, closes_h = window
                opens_at = cursor.replace(hour=opens_h, minute=0, second=0, microsecond=0)
                if cursor.date() == local.date():
                    if local.hour < opens_h:
                        # Aún no abre hoy
                        return opens_at
                    # Ya pasó la ventana de hoy → siguiente iteración
                else:
                    return opens_at
            # avanzar 1 día (a las 00:00 para empezar limpio)
            cursor = (cursor + timedelta(days=1)).replace(
                hour=0, minute=0, second=0, microsecond=0
            )
        raise RuntimeError(
            "OperatingHoursGate.next_open: no hay ventana operativa en los "
            "próximos 8 días — revisa OperatingHoursConfig.windows"
        )

    def closes_at(self, now: Optional[datetime] = None) -> Optional[datetime]:
        """
        Si AHORA está dentro de ventana, devuelve el datetime AWARE (en TZ
        Bogotá) en que cierra HOY.
        Si está fuera, devuelve None.
        """
        local = self._to_local(now)
        if not self.is_open(local):
            return None
        window = self.config.windows[local.weekday()]
        _, closes_h = window
        if closes_h == 24:
            # Cerramos a medianoche → 00:00 del día siguiente
            return (local + timedelta(days=1)).replace(
                hour=0, minute=0, second=0, microsecond=0
            )
        return local.replace(hour=closes_h, minute=0, second=0, microsecond=0)


def retry_on_failure(max_retries: int = 3, delay: float = 1.0):
    """Decorador para reintentar funciones que fallan"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1:
                        raise
                    logger.warning(f"Intento {attempt + 1} falló: {e}. Reintentando...")
                    time.sleep(delay * (2 ** attempt))
            return None
        return wrapper
    return decorator

def validate_phone_number(phone: str) -> bool:
    """Valida que el teléfono sea válido (solo dígitos, 10-15 chars)"""
    if not phone:
        return False
    return phone.isdigit() and 10 <= len(phone) <= 15

def sanitize_user_input(text: str, max_length: int = MAX_USER_MESSAGE_LENGTH) -> str:
    """
    Sanitiza el texto del usuario antes de pasarlo al LLM:
    - Elimina caracteres de control
    - Colapsa espacios múltiples
    - Trunca a max_length
    """
    if not text or not isinstance(text, str):
        return ""
    text = ''.join(ch for ch in text if ch >= ' ' or ch in ('\n', '\t'))
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    if len(text) > max_length:
        text = text[:max_length] + "…"
    return text

print("✅ Clases de configuración y utilidades definidas (incluye OperatingHoursGate v1.1)")


## CELDA 5: MODELOS DE BASE DE DATOS (sin cambios)

Reutilizamos los modelos SQLAlchemy existentes

In [ ]:
Base = declarative_base()

class EstadoFereteria(Enum):
    """
    Ciclo de vida de una ferretería frente al bot.

    Estado especial: NULL (None en Python, NULL en Postgres/Supabase)
      - Ferretería que existe en BD pero el bot AÚN NO le ha escrito.
      - ÚNICAS candidatas al broadcast inicial.
      - Tras enviar el broadcast (o tras recibir un primer mensaje espontáneo
        de la ferretería) transiciona a `primer_mensaje`.

    Estados oficiales del flujo de negocio (5):
      - primer_mensaje : el bot ya emitió el primer mensaje (broadcast) o la
                         ferretería envió su primer mensaje espontáneo. Estado
                         FUGAZ: si en el mismo turno hay respuesta del cliente,
                         transiciona inmediatamente a `inicio`.
                         NO recibe nuevos broadcasts (regla Meta 24h).
      - inicio         : la ferretería está conversando con el bot.
      - cotizacion     : ya se obtuvo precio + marca por texto.
      - cierre         : llegó imagen o PDF de cotización con líneas válidas
                         extraídas, o el cliente confirmó cierre por texto.
      - terminado      : se identificó una despedida (estado final).

    Estado interno auxiliar (NO de negocio, solo bandera del scheduler):
      - sin_respuesta  : >7 días sin interacción. Lo escribe el job de
                         mantenimiento. Solo sale cuando la ferretería nos
                         escribe de nuevo (regla Meta 24h).
    """
    primer_mensaje = "primer_mensaje"
    inicio = "inicio"
    sin_respuesta = "sin_respuesta"
    cotizacion = "cotizacion"
    cierre = "cierre"
    terminado = "terminado"

# Ferreterías con una conversación activa en cualquier punto del flujo.
# Se usa para el job nocturno que marca sin_respuesta tras 7 días de silencio.
# ⚠️ NO usar como filtro de broadcast: el broadcast solo va a `estado IS NULL`.
ESTADOS_EN_CONVERSACION = {
    EstadoFereteria.primer_mensaje,
    EstadoFereteria.inicio,
    EstadoFereteria.cotizacion,
    EstadoFereteria.cierre,
}

# Transiciones válidas (origen -> destinos permitidos).
# Cualquier transición fuera de este mapa se rechaza en transicionar_estado().
#
# ✅ Cambios respecto a la versión anterior:
#   - None ahora SOLO puede ir a primer_mensaje (eliminado el atajo a inicio).
#     El flujo unificado es: None → primer_mensaje → inicio → ...
#     tanto si el broadcast inicia la conversación como si la ferretería
#     escribe primero (en este último caso son dos transiciones encadenadas
#     en el mismo turno).
#   - inicio puede ir directo a cierre cuando llega un PDF/imagen con líneas
#     válidas extraídas. Internamente lo hacemos como inicio→cotizacion→cierre
#     encadenado para mantener trazabilidad, pero el grafo lo permite por
#     simetría.
TRANSICIONES_VALIDAS = {
    None:                              {EstadoFereteria.primer_mensaje},
    EstadoFereteria.primer_mensaje:    {EstadoFereteria.inicio, EstadoFereteria.terminado, EstadoFereteria.sin_respuesta},
    EstadoFereteria.inicio:            {EstadoFereteria.cotizacion, EstadoFereteria.cierre, EstadoFereteria.terminado, EstadoFereteria.sin_respuesta},
    EstadoFereteria.cotizacion:        {EstadoFereteria.cierre, EstadoFereteria.terminado, EstadoFereteria.sin_respuesta},
    EstadoFereteria.cierre:            {EstadoFereteria.terminado, EstadoFereteria.sin_respuesta},
    EstadoFereteria.terminado:         set(),  # estado final
    # ⚠️ sin_respuesta SOLO sale cuando la ferretería nos escribe (regla Meta 24h:
    # no podemos enviar mensajes free-form si no respondieron en 24h). Por eso
    # NO se permite la transición sin_respuesta -> primer_mensaje (ese caso lo
    # bloquearía Meta). El bot solo puede pasarla a inicio/cotizacion/cierre/terminado
    # cuando ELLA inicia la siguiente interacción.
    EstadoFereteria.sin_respuesta:     {EstadoFereteria.inicio, EstadoFereteria.cotizacion, EstadoFereteria.cierre, EstadoFereteria.terminado},
}

class Regional(Base):
    __tablename__ = "regionales"
    __table_args__ = {"schema": "public"}
    regional = Column(String(100), primary_key=True)

class Producto(Base):
    __tablename__ = "productos"
    __table_args__ = {"schema": "public"}
    id_producto = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nombre = Column(String(150), nullable=False)
    descripcion = Column(Text)
    kilogramos = Column(Numeric(10, 2))

class Geografia(Base):
    __tablename__ = "geografia"
    __table_args__ = {"schema": "public"}
    cod_municipio = Column(String(50), primary_key=True)
    cod_departamento = Column(String(50), nullable=False)
    nombre_municipio = Column(String(100), nullable=False)
    nombre_departamento = Column(String(100), nullable=False)
    regional = Column(String(100), ForeignKey("public.regionales.regional"))
    latitud = Column(Numeric(10, 8))
    longitud = Column(Numeric(10, 8))
    regional_rel = relationship("Regional", backref="geografias")

class Ferreteria(Base):
    __tablename__ = "ferreterias"
    __table_args__ = {"schema": "public"}
    id_ferreteria = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nit = Column(String(50), unique=True)
    nombre_ferreteria = Column(String(150))
    nombre_propietario = Column(String(150))
    razon_social = Column(String(150))
    correo = Column(String(150), unique=True)
    num_telefono = Column(String(20), nullable=False)
    num_vetados = Column(ARRAY(String), nullable=True, default=list)
    direccion = Column(String(200))
    cod_municipio = Column(String(50), ForeignKey("public.geografia.cod_municipio"), nullable=False)
    regional = Column(String(100), ForeignKey("public.regionales.regional"), nullable=False)
    estado = Column(
        SQLEnum(EstadoFereteria, name="estado", create_type=False,
                values_callable=lambda e: [i.value for i in e]),
        nullable=True,
        default=None
    )
    fecha_registro = Column(DateTime(timezone=True), server_default=func.now())
    latitud = Column(Numeric(10, 8))
    longitud = Column(Numeric(10, 8))
    municipio_rel = relationship("Geografia", backref="ferreterias")
    regional_rel = relationship("Regional", backref="ferreterias")

class HistorialInteraccion(Base):
    __tablename__ = "historial_interacciones"
    __table_args__ = {"schema": "public"}
    id_interaccion = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    id_ferreteria = Column(UUID(as_uuid=True), ForeignKey("public.ferreterias.id_ferreteria"), nullable=False)
    mensaje_usuario = Column(Text)
    respuesta_ia = Column(Text)
    tokens_consumidos = Column(Integer)
    fecha_registro = Column(DateTime(timezone=True), server_default=func.now())
    ferreteria_rel = relationship("Ferreteria", backref="interacciones")

class HistorialInteraccionAntiguo(Base):
    __tablename__ = "historial_interacciones_antiguos"
    __table_args__ = {"schema": "public"}
    id_interaccion = Column(UUID(as_uuid=True), primary_key=True)
    id_ferreteria = Column(UUID(as_uuid=True), ForeignKey("public.ferreterias.id_ferreteria"), nullable=False)
    mensaje_usuario = Column(Text)
    respuesta_ia = Column(Text)
    tokens_consumidos = Column(Integer)
    fecha_registro = Column(DateTime(timezone=True))
    motivo_archivo = Column(Text, default="Migración/Archivo")

class MarcaProducto(Base):
    __tablename__ = "marcas_productos"
    __table_args__ = {"schema": "public"}
    id_marca = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    nombre_marca = Column(String(150), nullable=False)
    regional = Column(String(100), ForeignKey("public.regionales.regional"))
    cod_municipio = Column(String(50), ForeignKey("public.geografia.cod_municipio"))
    regional_rel = relationship("Regional", backref="marcas")
    municipio_rel = relationship("Geografia", backref="marcas")

class Cotizacion(Base):
    __tablename__ = "cotizaciones"
    __table_args__ = {"schema": "public"}
    id_cotizacion = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    id_interaccion = Column(UUID(as_uuid=True), ForeignKey("public.historial_interacciones.id_interaccion"), nullable=False)
    id_ferreteria = Column(UUID(as_uuid=True), ForeignKey("public.ferreterias.id_ferreteria"), nullable=False)
    id_producto = Column(UUID(as_uuid=True), ForeignKey("public.productos.id_producto"), nullable=False)
    id_marca = Column(UUID(as_uuid=True), ForeignKey("public.marcas_productos.id_marca"))
    precio = Column(Numeric(15, 2), CheckConstraint("precio >= 0"), nullable=False)
    disponibilidad = Column(String(100))
    confianza_extraccion = Column(Numeric(5, 2))
    info_solicitada_ferreteria = Column(Text)
    fecha_cotizacion = Column(DateTime(timezone=True), server_default=func.now())
    regional = Column(String(100), ForeignKey("public.regionales.regional"), nullable=False)
    interaccion_rel = relationship("HistorialInteraccion", backref="cotizaciones")
    ferreteria_rel = relationship("Ferreteria", backref="cotizaciones")
    producto_rel = relationship("Producto", backref="cotizaciones")
    marca_rel = relationship("MarcaProducto", backref="cotizaciones")
    regional_rel = relationship("Regional", backref="cotizaciones")

class PreferenciasUsuario(Base):
    __tablename__ = "preferencias_usuario"
    __table_args__ = {"schema": "public"}
    id_ferreteria = Column(UUID(as_uuid=True), ForeignKey("public.ferreterias.id_ferreteria"), primary_key=True)
    tono_preferido_ia = Column(String(50), default="Profesional")
    ultima_actualizacion = Column(DateTime(timezone=True), server_default=func.now())
    ferreteria_rel = relationship("Ferreteria", backref="preferencias")

# ✅ NUEVO: tabla de idempotencia para webhooks de WhatsApp.
# Sustituye al set in-memory `processed_messages` que se perdía con cada
# reinicio del notebook (FIX 2.7). Si Meta reintenta un webhook después de
# un reinicio, esta tabla evita doble procesamiento.
#
# IMPORTANTE: para que funcione hay que crearla en Supabase con:
#
#   CREATE TABLE IF NOT EXISTS public.webhook_events_processed (
#     msg_id        text PRIMARY KEY,
#     fecha_proceso timestamptz NOT NULL DEFAULT now()
#   );
#   CREATE INDEX IF NOT EXISTS idx_webhook_events_fecha
#     ON public.webhook_events_processed(fecha_proceso);
#
# El job de mantenimiento purga registros >24h (Meta no reintenta tras eso).
class WebhookEventoProcesado(Base):
    __tablename__ = "webhook_events_processed"
    __table_args__ = {"schema": "public"}
    msg_id = Column(String(255), primary_key=True)
    fecha_proceso = Column(DateTime(timezone=True), server_default=func.now())

print("✅ Modelos SQLAlchemy definidos (incluye webhook_events_processed para idempotencia)")


## CELDA 6: DATABASE MANAGER (sin cambios significativos)

In [ ]:
# Import puntual para statements SQL crudos en idempotencia de webhooks.
from sqlalchemy import text as sa_text


class DatabaseManager:
    """Gestor centralizado de operaciones de BD con context managers"""

    def __init__(self, db_user: str, db_password: str, db_host: str, db_name: str):
        self.db_url = (
            f"postgresql://{db_user}:{db_password}@{db_host}/{db_name}"
            f"?options=-c%20search_path=public"
        )
        self.engine = create_engine(
            self.db_url,
            pool_pre_ping=True,
            pool_size=10,
            max_overflow=20,
            pool_recycle=3600
        )
        self.Session = sessionmaker(bind=self.engine)
        logger.info("✅ DatabaseManager inicializado")

    @contextmanager
    def get_session(self):
        """Context manager para sesiones de BD con rollback automático"""
        session = self.Session()
        try:
            yield session
            session.commit()
        except Exception as e:
            session.rollback()
            logger.error(f"Error en BD: {e}")
            raise
        finally:
            session.close()

    def _get_session(self):
        return self.Session()

    @staticmethod
    def _normalizar_nombre(nombre: str) -> str:
        """Normaliza un nombre para búsqueda case-insensitive y sin tildes."""
        if not nombre:
            return ""
        s = ''.join(
            c for c in unicodedata.normalize('NFD', nombre)
            if unicodedata.category(c) != 'Mn'
        )
        return re.sub(r'\s+', ' ', s).strip().lower()

    @retry_on_failure(max_retries=3)
    def obtener_ferreteria_por_telefono(self, numero: str) -> Optional[Ferreteria]:
        session = self._get_session()
        try:
            ferreteria = session.query(Ferreteria).filter(
                Ferreteria.num_telefono == numero
            ).first()
            return ferreteria
        except Exception as e:
            logger.error(f"Error buscando ferretería: {e}")
            return None
        finally:
            session.close()

    @retry_on_failure(max_retries=3)
    def crear_ferreteria_minima(self, numero: str) -> Ferreteria:
        """
        ✅ FIX 2.2: Crea una ferretería mínima cuando llega un mensaje de un
        número desconocido. El estado inicial es `primer_mensaje` (no `inicio`).

        Razón: el flujo unificado es siempre None → primer_mensaje → inicio → ...
        En el MISMO turno en que se crea, MessageProcessor llamará a
        `transicionar_estado(..., inicio)` para reflejar que la ferretería
        ya respondió. De esta forma:
          - Auditable: quedan 2 entradas de log (None→primer_mensaje y
            primer_mensaje→inicio).
          - Consistente con el grafo TRANSICIONES_VALIDAS.
          - Se trata igual una ferretería iniciada por broadcast que una
            iniciada por mensaje espontáneo.
        """
        session = self._get_session()
        try:
            regional_default = session.query(Regional).filter_by(regional="CENTRO").first()
            geografia_default = session.query(Geografia).filter_by(cod_municipio="05001").first()
            if not regional_default or not geografia_default:
                raise ValueError("Datos por defecto no encontrados.")
            nueva = Ferreteria(
                num_telefono=numero,
                num_vetados=[],
                nombre_ferreteria="Ferretería (registro automático)",
                cod_municipio=geografia_default.cod_municipio,
                regional=regional_default.regional,
                estado=EstadoFereteria.primer_mensaje  # ✅ FIX 2.2: era `inicio`
            )
            session.add(nueva)
            session.commit()
            logger.info(f"🆕 Ferretería creada en estado primer_mensaje: {nueva.id_ferreteria}")
            return nueva
        except Exception as e:
            session.rollback()
            logger.error(f"Error creando ferretería: {e}")
            raise
        finally:
            session.close()

    @retry_on_failure(max_retries=3)
    def guardar_interaccion(self, id_ferreteria: str, mensaje_usuario: str,
                            respuesta_ia: str, tokens: Optional[int] = None) -> Optional[str]:
        session = self._get_session()
        try:
            nueva = HistorialInteraccion(
                id_ferreteria=uuid.UUID(id_ferreteria),
                mensaje_usuario=mensaje_usuario,
                respuesta_ia=respuesta_ia,
                tokens_consumidos=tokens
            )
            session.add(nueva)
            session.commit()
            logger.info(f"📊 Interacción guardada: {nueva.id_interaccion}")
            return str(nueva.id_interaccion)
        except Exception as e:
            session.rollback()
            logger.error(f"Error guardando interacción: {e}")
            raise
        finally:
            session.close()

    # ------------------------------------------------------------------
    # ✅ v1.4.2: dos métodos auxiliares para el flujo inbound coalescido.
    # El mensaje del usuario se persiste inmediatamente al llegar (sin
    # respuesta_ia) para que los mensajes siguientes del mismo burst lo
    # vean en el historial. Cuando finalmente la IA genera la respuesta,
    # se hace UPDATE sobre la fila existente.
    #
    # ✅ v1.4.3:
    # - guardar_mensaje_usuario: verifica con SELECT round-trip post-commit
    #   que la fila realmente se persistió. Detecta commits zombi (típicos
    #   tras reinicio del kernel con conexiones del pool en estado raro).
    # - actualizar_respuesta_ia: si la fila no existe (caso observado en
    #   producción tras reinicio), hace INSERT con el MISMO UUID en lugar
    #   de crear duplicado con id distinto. Patrón UPSERT semántico.
    #
    # obtener_historial_reciente ya filtra correctamente las filas con
    # respuesta_ia=NULL (solo se inyecta el rol 'user', no un 'assistant'
    # vacío), así que el comportamiento de prompts NO se ve afectado.
    # ------------------------------------------------------------------
    @retry_on_failure(max_retries=3)
    def guardar_mensaje_usuario(self, id_ferreteria: str,
                                 mensaje_usuario: str) -> Optional[str]:
        """INSERT con respuesta_ia=NULL. Retorna interaction_id.

        ✅ v1.4.3: tras commit, hace SELECT de verificación en la MISMA
        sesión. Si la fila no aparece, lanza excepción (el retry decorator
        reintenta, y si todos los intentos fallan, propaga). Esto cierra
        el bug de 'UUID válido en memoria pero fila inexistente en BD'
        que producía warnings al hacer UPDATE más tarde.
        """
        session = self._get_session()
        try:
            nueva = HistorialInteraccion(
                id_ferreteria=uuid.UUID(id_ferreteria),
                mensaje_usuario=mensaje_usuario,
                respuesta_ia=None,
                tokens_consumidos=None,
            )
            session.add(nueva)
            session.commit()
            # ✅ v1.4.3: verificación explícita post-commit. Forzamos un
            # SELECT round-trip. Si la BD no tiene la fila, algo está mal
            # con la conexión y debemos saberlo AHORA, no 5 minutos después
            # cuando intentemos UPDATE.
            verificacion = (
                session.query(HistorialInteraccion)
                .filter(HistorialInteraccion.id_interaccion == nueva.id_interaccion)
                .first()
            )
            if verificacion is None:
                raise RuntimeError(
                    f"guardar_mensaje_usuario: commit retornó OK pero la fila "
                    f"{nueva.id_interaccion} no es visible en BD. Conexión "
                    f"posiblemente en estado inválido."
                )
            logger.info(
                f"📨 Mensaje usuario persistido (pendiente respuesta IA): "
                f"{nueva.id_interaccion}"
            )
            return str(nueva.id_interaccion)
        except Exception as e:
            session.rollback()
            logger.error(f"Error guardando mensaje de usuario: {e}")
            raise
        finally:
            session.close()

    @retry_on_failure(max_retries=3)
    def actualizar_respuesta_ia(self, id_interaccion: str,
                                 respuesta_ia: str,
                                 tokens: Optional[int] = None,
                                 mensaje_usuario_fallback: Optional[str] = None,
                                 id_ferreteria_fallback: Optional[str] = None) -> bool:
        """UPDATE de la fila previamente creada por guardar_mensaje_usuario.

        ✅ v1.4.3: si la fila NO existe (caso de reinicio del kernel que
        invalidó la transacción previa), y se proveen los fallbacks de
        mensaje_usuario_fallback + id_ferreteria_fallback, hacemos INSERT
        preservando el MISMO UUID. Esto evita generar una fila zombi nueva
        y mantiene coherencia con cualquier referencia que ya use ese UUID
        (p. ej., tabla cotizaciones cuya FK apunta a id_interaccion).

        Retorna True si el resultado es una fila completa con ese UUID.
        Solo retorna False si los fallbacks no se proveyeron y la fila no
        existía (el caller hará otra cosa).
        """
        session = self._get_session()
        try:
            fila = (
                session.query(HistorialInteraccion)
                .filter(HistorialInteraccion.id_interaccion == uuid.UUID(id_interaccion))
                .first()
            )
            if fila is None:
                # Fila no existe: caso de reinicio o INSERT que falló silencioso.
                # Si tenemos los datos para reconstruirla, hacemos INSERT con
                # el mismo UUID — preserva referencias y elimina ruido.
                if mensaje_usuario_fallback and id_ferreteria_fallback:
                    logger.warning(
                        f"actualizar_respuesta_ia: fila {id_interaccion} no existe; "
                        f"insertando con MISMO UUID (probable reinicio previo)"
                    )
                    nueva = HistorialInteraccion(
                        id_interaccion=uuid.UUID(id_interaccion),
                        id_ferreteria=uuid.UUID(id_ferreteria_fallback),
                        mensaje_usuario=mensaje_usuario_fallback,
                        respuesta_ia=respuesta_ia,
                        tokens_consumidos=tokens,
                    )
                    session.add(nueva)
                    session.commit()
                    logger.info(
                        f"📊 Fila recreada con UUID original: {id_interaccion}"
                    )
                    return True
                logger.warning(
                    f"actualizar_respuesta_ia: interacción {id_interaccion} "
                    f"no existe y sin fallbacks; UPDATE descartado"
                )
                return False
            fila.respuesta_ia = respuesta_ia
            if tokens is not None:
                fila.tokens_consumidos = tokens
            session.commit()
            logger.info(f"📊 Respuesta IA actualizada en {id_interaccion}")
            return True
        except Exception as e:
            session.rollback()
            logger.error(f"Error actualizando respuesta IA: {e}")
            raise
        finally:
            session.close()

    # ------------------------------------------------------------------
    # ✅ v1.4.3: rehidratación de mensajes inbound huérfanos tras reinicio
    # ------------------------------------------------------------------
    @retry_on_failure(max_retries=3)
    def obtener_inbounds_pendientes(self, ventana_minutos: int = 30) -> List[Dict]:
        """
        Lista las filas de historial con respuesta_ia=NULL de los últimos
        `ventana_minutos` minutos. Estas son los mensajes del cliente que
        ENTRARON al bot, se persistieron en BD, pero la respuesta nunca
        se generó (típicamente porque el bot se reinició durante la
        ventana de listen_window antes de que el timer venciera).

        Se usa al arranque del bot para reagendar los inbound_intent
        correspondientes — así no quedan clientes sin respuesta.

        Devuelve lista de dicts:
          [{'id_interaccion': str, 'id_ferreteria': str,
            'numero': str, 'mensaje': str, 'fecha': datetime}, ...]
        ordenados cronológicamente (más antiguo primero). Solo incluye
        ferreterías NO vetadas y con número de teléfono válido.
        """
        session = self._get_session()
        try:
            cutoff = datetime.now(timezone.utc) - timedelta(minutes=ventana_minutos)
            filas = (
                session.query(HistorialInteraccion, Ferreteria)
                .join(
                    Ferreteria,
                    Ferreteria.id_ferreteria == HistorialInteraccion.id_ferreteria,
                )
                .filter(HistorialInteraccion.respuesta_ia.is_(None))
                .filter(HistorialInteraccion.fecha_registro >= cutoff)
                .filter(Ferreteria.num_telefono.isnot(None))
                .order_by(HistorialInteraccion.fecha_registro.asc())
                .all()
            )
            # Excluir entradas de outreach (marcadas en mensaje_usuario con
            # el prefijo '[OUTREACH-TEMPLATE]'). Solo nos interesan los
            # mensajes reales del cliente.
            resultado = []
            for interaccion, ferreteria in filas:
                msg = interaccion.mensaje_usuario or ""
                if msg.startswith("[OUTREACH-TEMPLATE]"):
                    continue
                resultado.append({
                    "id_interaccion": str(interaccion.id_interaccion),
                    "id_ferreteria": str(ferreteria.id_ferreteria),
                    "numero": ferreteria.num_telefono,
                    "mensaje": msg,
                    "fecha": interaccion.fecha_registro,
                })
            logger.info(
                f"🔎 Inbounds pendientes detectados (últimos {ventana_minutos}min): "
                f"{len(resultado)}"
            )
            return resultado
        except Exception as e:
            logger.error(f"Error obteniendo inbounds pendientes: {e}")
            return []
        finally:
            session.close()

    @retry_on_failure(max_retries=3)
    def marcar_inbound_perdido(self, id_interaccion: str) -> bool:
        """
        Marca una fila huérfana antigua con un placeholder en respuesta_ia.
        Se usa para inbounds anteriores a la ventana de rehidratación: no
        tiene sentido responder a algo de hace 2 horas (el cliente
        seguramente ya se aburrió y se fue), pero tampoco queremos dejar
        la fila eternamente NULL porque ensucia las queries.

        Tras marcar, esa fila SÍ aparecerá en obtener_historial_reciente
        como turno 'assistant' con el placeholder. Si no quieres eso,
        deja respuesta_ia NULL — el filtro de historial igual lo ignora.
        """
        session = self._get_session()
        try:
            fila = (
                session.query(HistorialInteraccion)
                .filter(HistorialInteraccion.id_interaccion == uuid.UUID(id_interaccion))
                .first()
            )
            if fila is None:
                return False
            fila.respuesta_ia = "[RESPUESTA PERDIDA POR REINICIO DEL BOT]"
            session.commit()
            return True
        except Exception as e:
            session.rollback()
            logger.error(f"Error marcando inbound perdido: {e}")
            raise
        finally:
            session.close()

    @retry_on_failure(max_retries=3)
    def obtener_historial_reciente(self, id_ferreteria: str,
                                    limite: int = 10) -> List[Dict[str, str]]:
        """
        Devuelve las últimas `limite` interacciones de una ferretería ordenadas
        cronológicamente (más antigua primero), en formato listo para inyectar
        en messages.create() de Anthropic.
        """
        session = self._get_session()
        try:
            registros = (
                session.query(HistorialInteraccion)
                .filter(HistorialInteraccion.id_ferreteria == uuid.UUID(id_ferreteria))
                .order_by(HistorialInteraccion.fecha_registro.desc())
                .limit(limite)
                .all()
            )
            registros = list(reversed(registros))
            historial: List[Dict[str, str]] = []
            for r in registros:
                if r.mensaje_usuario:
                    historial.append({"role": "user", "content": r.mensaje_usuario})
                if r.respuesta_ia:
                    historial.append({"role": "assistant", "content": r.respuesta_ia})
            logger.info(f"📚 Historial recuperado: {len(registros)} interacciones "
                        f"({len(historial)} mensajes)")
            return historial
        except Exception as e:
            logger.error(f"Error obteniendo historial: {e}")
            return []
        finally:
            session.close()

    # ------------------------------------------------------------------
    # ✅ FIX 2.7: Idempotencia de webhooks persistente en BD
    # Reemplaza al set in-memory que se perdía en cada reinicio del notebook.
    # ------------------------------------------------------------------
    def msg_ya_procesado(self, msg_id: str) -> bool:
        """
        Verifica si un msg_id ya fue procesado. Si NO existe, lo inserta
        atómicamente y devuelve False. Si ya existe, devuelve True.

        Usa INSERT con ON CONFLICT DO NOTHING para que la inserción y la
        verificación sean una sola operación atómica (sin race condition
        entre dos webhooks duplicados llegando casi simultáneos).

        Si la BD falla, devuelve False (no bloqueamos al cliente por un
        problema de infra) y dejamos que el set in-memory haga de respaldo
        a nivel de MessageProcessor.
        """
        if not msg_id:
            return False
        try:
            with self.get_session() as session:
                # Postgres-specific: INSERT ... ON CONFLICT
                result = session.execute(
                    sa_text(
                        "INSERT INTO public.webhook_events_processed (msg_id) "
                        "VALUES (:mid) ON CONFLICT (msg_id) DO NOTHING "
                        "RETURNING msg_id"
                    ),
                    {"mid": msg_id}
                )
                inserted = result.first() is not None
                # Si insertó, no estaba antes → no procesado
                # Si no insertó, ya existía → ya procesado
                return not inserted
        except Exception as e:
            logger.warning(f"msg_ya_procesado falló (asumiendo no procesado): {e}")
            return False

    def purgar_webhook_events_antiguos(self, days: int = 1) -> int:
        """Limpia eventos de webhook >24h. Meta no reintenta después de eso."""
        try:
            cutoff = datetime.now(timezone.utc) - timedelta(days=days)
            with self.get_session() as session:
                eliminados = session.query(WebhookEventoProcesado).filter(
                    WebhookEventoProcesado.fecha_proceso < cutoff
                ).delete(synchronize_session=False)
                logger.info(f"🧹 Purgados {eliminados} webhook_events antiguos (>{days}d)")
                return eliminados
        except Exception as e:
            logger.error(f"Error purgando webhook_events: {e}")
            return 0

    # ------------------------------------------------------------------
    # Helpers para resolver/crear FKs del extractor
    # ------------------------------------------------------------------
    @retry_on_failure(max_retries=3)
    def obtener_producto_por_nombre(self, nombre: str) -> Optional[uuid.UUID]:
        """
        Busca un producto por nombre normalizado (sin tildes, case-insensitive).
        ⚠️ NO crea productos: el catálogo `productos` es cerrado. Si no existe,
        devuelve None y la cotización se debe descartar/loguear.
        """
        if not nombre or not nombre.strip():
            return None
        nombre_norm = self._normalizar_nombre(nombre)
        with self.get_session() as session:
            for p in session.query(Producto).all():
                if self._normalizar_nombre(p.nombre) == nombre_norm:
                    return p.id_producto
        logger.info(f"⚠️ Producto '{nombre}' no existe en catálogo, no se crea")
        return None

    @retry_on_failure(max_retries=3)
    def obtener_producto_cemento(self) -> Optional[Tuple[uuid.UUID, str]]:
        """
        Localiza el producto 'cemento' en el catálogo con MATCH FLEXIBLE:
        cualquier producto cuyo nombre normalizado CONTENGA la palabra 'cemento'.
        Ej: 'Cemento gris', 'Cemento Argos', 'CEMENTO PORTLAND' → match.

        Devuelve (id_producto, nombre_real) del primero encontrado, o None.
        Si hay varios candidatos, prioriza el más corto (más genérico) y loguea
        los demás para que se sepa cuál se usó.
        """
        with self.get_session() as session:
            candidatos = []
            for p in session.query(Producto).all():
                if "cemento" in self._normalizar_nombre(p.nombre):
                    candidatos.append((p.id_producto, p.nombre))
            if not candidatos:
                logger.warning("⚠️ No hay producto 'cemento' en el catálogo de `productos`")
                return None
            candidatos.sort(key=lambda c: len(c[1]))
            elegido = candidatos[0]
            if len(candidatos) > 1:
                otros = ", ".join(c[1] for c in candidatos[1:])
                logger.info(f"🔍 Cemento: usando '{elegido[1]}' (otros candidatos: {otros})")
            else:
                logger.info(f"🔍 Cemento: usando '{elegido[1]}'")
            return elegido

    @retry_on_failure(max_retries=3)
    def obtener_o_crear_marca(self, nombre: str, regional: Optional[str] = None,
                              cod_municipio: Optional[str] = None) -> Optional[uuid.UUID]:
        """
        Busca una marca por nombre normalizado. Si no existe la crea
        asociada (cuando es posible) a la regional/municipio recibidos.
        """
        if not nombre or not nombre.strip():
            return None
        nombre_norm = self._normalizar_nombre(nombre)
        with self.get_session() as session:
            for m in session.query(MarcaProducto).all():
                if self._normalizar_nombre(m.nombre_marca) == nombre_norm:
                    return m.id_marca
            nueva = MarcaProducto(
                nombre_marca=nombre.strip()[:150],
                regional=regional,
                cod_municipio=cod_municipio,
            )
            session.add(nueva)
            session.flush()
            logger.info(f"🆕 Marca creada: '{nueva.nombre_marca}' → {nueva.id_marca}")
            return nueva.id_marca

    @retry_on_failure(max_retries=3)
    def registrar_cotizacion(self, id_interaccion: str, id_ferreteria: str,
                             producto_nombre: str, marca_nombre: str,
                             precio: float, regional: str,
                             disponibilidad: Optional[str] = None,
                             confianza: Optional[float] = None,
                             info_solicitada: Optional[str] = None,
                             cod_municipio: Optional[str] = None,
                             id_producto: Optional[uuid.UUID] = None) -> Optional[Dict]:
        """
        Persiste una cotización resolviendo las FKs.

        - Producto: NO se crea. Si `id_producto` no se pasa, se busca por
          `producto_nombre` en el catálogo. Si tampoco existe → cotización
          rechazada.
        - Marca: SÍ se crea automáticamente si no existe.

        Devuelve un dict con TODAS las columnas insertadas (mirror de la fila
        de `cotizaciones`) — útil para anexarlo al CSV.
        Devuelve None si la inserción falla.
        """
        try:
            if id_producto is None:
                id_producto = self.obtener_producto_por_nombre(producto_nombre)
            if id_producto is None:
                logger.warning(
                    f"Cotización rechazada: producto '{producto_nombre}' "
                    f"no existe en catálogo (no se crea)"
                )
                return None

            id_marca = self.obtener_o_crear_marca(marca_nombre, regional, cod_municipio)
            if id_marca is None:
                logger.warning("Cotización rechazada: marca vacía")
                return None

            with self.get_session() as session:
                cot = Cotizacion(
                    id_interaccion=uuid.UUID(id_interaccion),
                    id_ferreteria=uuid.UUID(id_ferreteria),
                    id_producto=id_producto,
                    id_marca=id_marca,
                    precio=precio,
                    disponibilidad=(disponibilidad or "")[:100] or None,
                    confianza_extraccion=confianza,
                    info_solicitada_ferreteria=info_solicitada,
                    regional=regional,
                )
                session.add(cot)
                session.flush()
                fila = {
                    "id_cotizacion": str(cot.id_cotizacion),
                    "id_interaccion": str(cot.id_interaccion),
                    "id_ferreteria": str(cot.id_ferreteria),
                    "id_producto": str(cot.id_producto),
                    "id_marca": str(cot.id_marca),
                    "precio": float(cot.precio) if cot.precio is not None else None,
                    "disponibilidad": cot.disponibilidad,
                    "confianza_extraccion": (
                        float(cot.confianza_extraccion)
                        if cot.confianza_extraccion is not None else None
                    ),
                    "info_solicitada_ferreteria": cot.info_solicitada_ferreteria,
                    "fecha_cotizacion": (
                        cot.fecha_cotizacion.isoformat()
                        if cot.fecha_cotizacion else datetime.now(timezone.utc).isoformat()
                    ),
                    "regional": cot.regional,
                }
                logger.info(f"💰 Cotización registrada: {fila['id_cotizacion']} "
                            f"(producto={producto_nombre}, marca={marca_nombre}, precio={precio})")
                return fila
        except Exception as e:
            logger.error(f"Error registrando cotización: {e}")
            return None

    @staticmethod
    def append_cotizacion_a_csv(fila: Dict, csv_path: str) -> bool:
        """
        Anexa una fila de cotización al CSV incremental (UUIDs reales,
        mirror de la tabla `cotizaciones`). Crea el archivo con cabecera si no
        existe. Igual al patrón usado por ARGOS.
        """
        import csv
        import os
        columnas = [
            "id_cotizacion", "id_interaccion", "id_ferreteria",
            "id_producto", "id_marca", "precio", "disponibilidad",
            "confianza_extraccion", "info_solicitada_ferreteria",
            "fecha_cotizacion", "regional",
        ]
        try:
            existe = os.path.exists(csv_path)
            os.makedirs(os.path.dirname(csv_path), exist_ok=True) if os.path.dirname(csv_path) else None
            with open(csv_path, "a", encoding="utf-8", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=columnas)
                if not existe:
                    writer.writeheader()
                writer.writerow({k: fila.get(k) for k in columnas})
            logger.info(f"📝 CSV actualizado: {csv_path} (+1 fila)")
            return True
        except Exception as e:
            logger.error(f"Error escribiendo CSV: {e}")
            return False

    # ------------------------------------------------------------------
    # Máquina de estados con guard
    # ------------------------------------------------------------------
    def transicionar_estado(self, ferreteria_id: uuid.UUID,
                            nuevo_estado: EstadoFereteria,
                            forzar: bool = False,
                            session: Optional[Any] = None) -> bool:
        """
        Aplica una transición de estado validándola contra TRANSICIONES_VALIDAS.

        - Si `forzar=True`, ignora el grafo (úsalo solo para `terminado` desde
          cualquier punto, p.ej. despedida o veto).
        - Si `session` se pasa, reusa esa sesión (útil para componer transiciones
          dentro de un job más grande). Si es None, abre una propia.

        ✅ FIX: usa flag_modified sobre el campo enum para garantizar que
        SQLAlchemy detecte el cambio (algunas versiones no lo notan en
        comparaciones de enum-string).
        """
        def _do(s) -> bool:
            ferreteria = s.query(Ferreteria).filter_by(id_ferreteria=ferreteria_id).first()
            if not ferreteria:
                logger.warning(f"transicionar_estado: ferretería no encontrada")
                return False
            actual = ferreteria.estado
            if actual == nuevo_estado:
                return True  # idempotente
            if not forzar:
                permitidos = TRANSICIONES_VALIDAS.get(actual, set())
                if nuevo_estado not in permitidos:
                    logger.info(
                        f"🚫 Transición rechazada: {actual} -> {nuevo_estado.value} "
                        f"(permitidos: {[e.value for e in permitidos]})"
                    )
                    return False
            ferreteria.estado = nuevo_estado
            flag_modified(ferreteria, "estado")
            anterior = actual.value if actual else "None"
            logger.info(f"🔄 Estado: {anterior} → {nuevo_estado.value}")
            return True

        try:
            if session is not None:
                # Reusar la sesión externa: no commiteamos aquí, lo hace el caller
                return _do(session)
            # Sesión propia con commit/rollback automático
            with self.get_session() as s:
                return _do(s)
        except Exception as e:
            logger.error(f"Error en transicionar_estado: {e}")
            return False

    def is_phone_vetoed(self, ferreteria_id: uuid.UUID, num_telefono: str) -> bool:
        with self.get_session() as session:
            ferreteria = session.query(Ferreteria).filter_by(id_ferreteria=ferreteria_id).first()
            if not ferreteria:
                return False
            if ferreteria.estado == EstadoFereteria.terminado:
                return True
            vetados = ferreteria.num_vetados or []
            return (num_telefono or "").strip() in [str(v).strip() for v in vetados]

    def add_to_vetados(self, ferreteria_id: uuid.UUID, num_telefono: str,
                       set_terminado: bool = True) -> bool:
        try:
            with self.get_session() as session:
                ferreteria = session.query(Ferreteria).filter_by(id_ferreteria=ferreteria_id).first()
                if not ferreteria:
                    return False
                vetados = list(ferreteria.num_vetados or [])
                if num_telefono and num_telefono not in vetados:
                    vetados.append(num_telefono)
                    ferreteria.num_vetados = vetados
                    flag_modified(ferreteria, "num_vetados")
                if set_terminado:
                    ferreteria.estado = EstadoFereteria.terminado
                    flag_modified(ferreteria, "estado")
                logger.info(f"Número agregado a vetados ({len(vetados)} total)")
                return True
        except Exception as e:
            logger.error(f"Error agregando a vetados: {e}")
            return False

    def update_ferreteria_estado(self, ferreteria_id: uuid.UUID, nuevo_estado: str) -> bool:
        """
        ⚠️ DEPRECADO: actualización directa sin validar transición.
        Mantenido solo por compatibilidad. Para flujo nuevo, usar
        `transicionar_estado` que respeta el grafo.
        """
        try:
            with self.get_session() as session:
                ferreteria = session.query(Ferreteria).filter_by(id_ferreteria=ferreteria_id).first()
                if not ferreteria:
                    return False
                ferreteria.estado = EstadoFereteria[nuevo_estado]
                flag_modified(ferreteria, "estado")
                logger.info(f"⚠️ Estado actualizado SIN validar grafo: {nuevo_estado}")
                return True
        except Exception as e:
            logger.error(f"Error actualizando estado: {e}")
            return False

    def archive_old_interactions(self, days: int = 7,
                                 motivo: str = "Archivo automático >7d") -> int:
        try:
            archived_count = 0
            cutoff_date = datetime.now(timezone.utc) - timedelta(days=days)
            with self.get_session() as session:
                old_interactions = session.query(HistorialInteraccion).filter(
                    HistorialInteraccion.fecha_registro < cutoff_date
                ).all()
                for interaction in old_interactions:
                    old_record = HistorialInteraccionAntiguo(
                        id_interaccion=interaction.id_interaccion,
                        id_ferreteria=interaction.id_ferreteria,
                        mensaje_usuario=interaction.mensaje_usuario,
                        respuesta_ia=interaction.respuesta_ia,
                        tokens_consumidos=interaction.tokens_consumidos,
                        fecha_registro=interaction.fecha_registro,
                        motivo_archivo=motivo
                    )
                    session.add(old_record)
                    archived_count += 1
                session.query(HistorialInteraccion).filter(
                    HistorialInteraccion.fecha_registro < cutoff_date
                ).delete()
                logger.info(f"Archivadas {archived_count} interacciones antiguas (>{days}d)")
                return archived_count
        except Exception as e:
            logger.error(f"Error archivando interacciones: {e}")
            return 0

print("✅ DatabaseManager listo (idempotencia BD, crear→primer_mensaje, transicionar_estado robusto)")


## CELDA 6.5: ✅ NUEVO — SISTEMA DE IDENTIFICACIÓN DE PRODUCTOS POR SIMILITUD

Funcionalidad **agregada** sobre BotV1.3 (no modifica el comportamiento original):

1. **Normalización robusta** de nombres de productos (tildes, mayúsculas, sinónimos, stopwords).
2. **Métricas de similitud** (Levenshtein, Jaro-Winkler, Token-Sort/Set, Cosine TF-IDF, Partial, Ponderada).
3. **Búsqueda difusa escalonada** en el catálogo de `productos`: exacto normalizado → contención → difuso ponderado.
4. **Gestor integrado** con `DatabaseManager` para resolver el `id_producto` correcto a partir de un nombre con typos o variaciones.

Estas clases se instancian opcionalmente desde el bootstrap (CELDA 11). El bot mantiene **100 % de su funcionalidad original**: si no se invocan, el flujo legado sigue intacto.

### CELDA 6.5.1: Importaciones específicas para similitud de strings

In [ ]:
# Importaciones adicionales para el módulo de similitud de productos.
# Las clases base (unicodedata, re, uuid, logging, dataclass, Enum, Optional, List,
# Dict, Tuple) ya están importadas en CELDA 3, así que aquí solo agregamos
# lo específico de este módulo.
from difflib import SequenceMatcher

# Librerías de similitud
from fuzzywuzzy import fuzz
from rapidfuzz import distance, fuzz as rf_fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("✅ Importaciones de ProductSimilarity completadas")

### CELDA 6.5.2: Normalización de nombres de productos

In [ ]:
class NormalizadorNombres:
    """
    Normaliza nombres de productos eliminando variaciones comunes.
    Útil para comparar "Cemento Argos", "cemento argos", "cémento argos", etc.
    """

    # Sinónimos y abreviaciones comunes en ferretería
    SINONIMOS = {
        "kg": "kilogramo",
        "bulto": "bolsa",
        "saco": "bolsa",
        "presentación": "",
        "pres": "",
        "gris": "",
        "blanco": "",
        "portland": "",
        "estructural": "",
        "tipo i": "",
        "tipo ii": "",
        "tipo iii": "",
    }

    # Stopwords específicos de ferretería a eliminar
    STOPWORDS_FERRETERIA = {
        "de", "la", "el", "y", "un", "una", "unos", "unas",
        "en", "por", "para", "con", "sin", "a", "o", "del",
        "bolsa", "saco", "paquete", "caja",  # contenedores
    }

    @staticmethod
    def remover_tildes(texto: str) -> str:
        """
        Convierte acentos: "cémento" → "cemento"
        """
        if not texto:
            return ""
        nfd = unicodedata.normalize('NFD', texto)
        return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')

    @staticmethod
    def limpiar_espacios_puntuacion(texto: str) -> str:
        """
        Elimina puntuación y normaliza espacios.
        """
        if not texto:
            return ""
        # Remover puntuación excepto guiones (para "tipo-i")
        texto = re.sub(r'[.,;:!?"\'\'`()\[\]{}]', ' ', texto)
        # Colapsar espacios múltiples
        texto = re.sub(r'\s+', ' ', texto).strip()
        return texto

    @classmethod
    def normalizar(cls, nombre: str, eliminar_stopwords: bool = True) -> str:
        """
        Pipeline completo de normalización.

        Args:
            nombre: string a normalizar
            eliminar_stopwords: si True, elimina palabras comunes

        Returns:
            string normalizado (lowercase, sin tildes, sin stopwords)

        Ejemplo:
            "CEMENTO Argos - Bolsa 50 kg" → "cemento argos"
        """
        if not nombre or not isinstance(nombre, str):
            return ""

        # 1) Lowercase
        texto = nombre.lower()

        # 2) Remover tildes
        texto = cls.remover_tildes(texto)

        # 3) Limpiar puntuación y espacios
        texto = cls.limpiar_espacios_puntuacion(texto)

        # 4) Expandir sinónimos (antes de eliminar stopwords)
        for sino, reemplazo in cls.SINONIMOS.items():
            # Usar boundaries de palabra para no afectar substrings
            texto = re.sub(rf'\b{re.escape(sino)}\b', reemplazo, texto)

        # 5) Eliminar stopwords si se solicita
        if eliminar_stopwords:
            palabras = texto.split()
            palabras = [p for p in palabras if p not in cls.STOPWORDS_FERRETERIA]
            texto = ' '.join(palabras)

        # 6) Colapsar espacios finales
        texto = re.sub(r'\s+', ' ', texto).strip()

        return texto if texto else ""

    @classmethod
    def normalizar_preservar_orden(cls, nombre: str) -> str:
        """
        Versión que NO elimina stopwords (preserva orden de palabras).
        Útil para cuando queremos comparar exactitud semántica.
        """
        return cls.normalizar(nombre, eliminar_stopwords=False)


print("✅ NormalizadorNombres definido")

### CELDA 6.5.3: Métricas de similitud entre strings

In [ ]:
class MetricasSimilitud:
    """
    Colección de métricas para comparar strings.
    Cada métrica devuelve un score 0.0 a 1.0 (1.0 = idéntico).
    """

    @staticmethod
    def levenshtein_ratio(s1: str, s2: str) -> float:
        """
        Distancia de Levenshtein normalizada (0–1).
        Mide ediciones (inserción, eliminación, sustitución).

        Ejemplo: "cemento" vs "cemento argos" → 0.7
        """
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        try:
            # rapidfuzz es más rápido que python-Levenshtein puro
            return rf_fuzz.ratio(s1, s2) / 100.0
        except Exception:
            # Fallback a difflib
            return SequenceMatcher(None, s1, s2).ratio()

    @staticmethod
    def jaro_winkler(s1: str, s2: str) -> float:
        """
        Jaro-Winkler: favorecer prefijos iguales.
        Útil para typos al inicio: "argos" vs "arguos".

        Rango: [0, 1]
        """
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        try:
            return rf_fuzz.jaro_winkler(s1, s2) / 100.0
        except Exception:
            return 0.0

    @staticmethod
    def token_sort_ratio(s1: str, s2: str) -> float:
        """
        Ordena tokens de cada string y luego compara.
        Excelente para orden de palabras diferente:
        "Cemento Argos Gris" vs "Argos Cemento Gris" → ~1.0
        """
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        try:
            return fuzz.token_sort_ratio(s1, s2) / 100.0
        except Exception:
            return 0.0

    @staticmethod
    def token_set_ratio(s1: str, s2: str) -> float:
        """
        Compara tokens únicos (ordena + elimina duplicados).
        Útil cuando hay palabras extra:
        "Cemento Argos 50kg" vs "Argos Cemento" → ~0.9
        """
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        try:
            return fuzz.token_set_ratio(s1, s2) / 100.0
        except Exception:
            return 0.0

    @staticmethod
    def cosine_tfidf(s1: str, s2: str) -> float:
        """
        Similitud de coseno usando TF-IDF.
        Semántica: "cemento gris" vs "cemento portland" → 0.7
        Excelente para capturar significado.
        """
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        try:
            vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 3))
            matriz = vectorizer.fit_transform([s1, s2])
            sim = cosine_similarity(matriz)[0, 1]
            return float(sim)
        except Exception:
            return 0.0

    @staticmethod
    def partial_ratio(s1: str, s2: str) -> float:
        """
        Encontrar la mejor coincidencia de substring.
        "cemento argos" vs "cemento" → ~1.0
        """
        if not s1 and not s2:
            return 1.0
        if not s1 or not s2:
            return 0.0

        try:
            return fuzz.partial_ratio(s1, s2) / 100.0
        except Exception:
            return 0.0

    @classmethod
    def score_ponderado(cls, s1: str, s2: str,
                       pesos: Optional[Dict[str, float]] = None) -> float:
        """
        Combina múltiples métricas con pesos personalizables.

        Pesos default: favorecer Jaro-Winkler + Token-Sort (orden flexible)

        Args:
            s1, s2: strings a comparar
            pesos: dict con claves {"levenshtein", "jaro_winkler",
                   "token_sort", "token_set", "cosine", "partial"}

        Returns:
            Score 0.0–1.0
        """
        if pesos is None:
            # Weights que favorecen robustez a typos y orden flexible
            pesos = {
                "levenshtein": 0.15,
                "jaro_winkler": 0.25,
                "token_sort": 0.30,
                "token_set": 0.15,
                "cosine": 0.10,
                "partial": 0.05,
            }

        # Normalizar weights a suma 1.0
        suma_pesos = sum(pesos.values())
        if suma_pesos <= 0:
            return 0.0
        pesos = {k: v / suma_pesos for k, v in pesos.items()}

        scores = {
            "levenshtein": cls.levenshtein_ratio(s1, s2),
            "jaro_winkler": cls.jaro_winkler(s1, s2),
            "token_sort": cls.token_sort_ratio(s1, s2),
            "token_set": cls.token_set_ratio(s1, s2),
            "cosine": cls.cosine_tfidf(s1, s2),
            "partial": cls.partial_ratio(s1, s2),
        }

        return sum(scores.get(k, 0.0) * v for k, v in pesos.items())


print("✅ MetricasSimilitud definida")

### CELDA 6.5.4: Dataclass `ResultadoBusqueda`

In [ ]:
@dataclass
class ResultadoBusqueda:
    """
    Resultado de búsqueda difusa de un producto en el catálogo.
    """
    id_producto: Optional[uuid.UUID]
    nombre_producto: str
    nombre_buscado: str
    score_similitud: float
    metodo: str  # "exacto", "normalizado", "difuso_ponderado"
    alternativas: List[Tuple[str, float, uuid.UUID]]  # (nombre, score, id)
    confianza: str  # "ALTA", "MEDIA", "BAJA", "NO_ENCONTRADO"

    def es_valido(self, threshold: float = 0.80) -> bool:
        """Determina si el resultado es lo suficientemente confiable."""
        return self.score_similitud >= threshold

    def __str__(self) -> str:
        return (
            f"Búsqueda: {self.nombre_buscado!r}\n"
            f"  → Encontrado: {self.nombre_producto!r}\n"
            f"  → Score: {self.score_similitud:.2%}\n"
            f"  → Confianza: {self.confianza}\n"
            f"  → Método: {self.metodo}"
        )


print("✅ ResultadoBusqueda definido")

### CELDA 6.5.5: Buscador difuso de productos en catálogo

In [ ]:
class BuscadorProductos:
    """
    Busca productos en un catálogo usando similitud de nombres.

    Flujo de búsqueda escalonada (de mayor a menor confianza):
    1. Exacto normalizado (match perfecto tras normalizar)
    2. Contención (está contenido en el nombre del catálogo)
    3. Difuso ponderado (métrica combinada)
    """

    def __init__(self, catalogo: Dict[uuid.UUID, str]):
        """
        Args:
            catalogo: dict {id_producto: nombre_producto}
                     e.g., {UUID(...): "Cemento Argos", ...}
        """
        self.catalogo = catalogo
        self.catalogo_normalizado = {
            id_: NormalizadorNombres.normalizar(nombre)
            for id_, nombre in catalogo.items()
        }
        logger.info(f"🔍 Catálogo cargado: {len(catalogo)} productos")

    def actualizar_catalogo(self, catalogo: Dict[uuid.UUID, str]):
        """Actualiza el catálogo interno (p.ej., después de DB updates)."""
        self.catalogo = catalogo
        self.catalogo_normalizado = {
            id_: NormalizadorNombres.normalizar(nombre)
            for id_, nombre in catalogo.items()
        }
        logger.info(f"🔄 Catálogo actualizado: {len(catalogo)} productos")

    def buscar(self, nombre_buscado: str,
               threshold_minimo: float = 0.60,
               limitar_alternativas: int = 3) -> ResultadoBusqueda:
        """
        Busca un producto en el catálogo con búsqueda escalonada.

        Args:
            nombre_buscado: nombre a buscar (p.ej., "cemento argos gris")
            threshold_minimo: score mínimo para considerarlo válido (0–1)
            limitar_alternativas: cuántos alternativas devolver (top N)

        Returns:
            ResultadoBusqueda con id_producto, score, confianza, etc.
        """
        if not nombre_buscado or not isinstance(nombre_buscado, str):
            return ResultadoBusqueda(
                id_producto=None,
                nombre_producto="",
                nombre_buscado=nombre_buscado,
                score_similitud=0.0,
                metodo="error_input",
                alternativas=[],
                confianza="NO_ENCONTRADO",
            )

        nombre_norm = NormalizadorNombres.normalizar(nombre_buscado)

        # ─── PASO 1: Búsqueda Exacta Normalizada ───────────────────────────
        for id_, nombre_catalogo_norm in self.catalogo_normalizado.items():
            if nombre_catalogo_norm == nombre_norm:
                return ResultadoBusqueda(
                    id_producto=id_,
                    nombre_producto=self.catalogo[id_],
                    nombre_buscado=nombre_buscado,
                    score_similitud=1.0,
                    metodo="exacto_normalizado",
                    alternativas=[],
                    confianza="ALTA",
                )

        # ─── PASO 2: Búsqueda por Contención ───────────────────────────────
        # Si el nombre buscado (normalizado) está contenido en algún producto
        if nombre_norm:
            for id_, nombre_catalogo_norm in self.catalogo_normalizado.items():
                # ¿El buscado está contenido? (p.ej., "cemento" en "cemento argos")
                if nombre_norm in nombre_catalogo_norm.split():
                    score = 0.95  # Muy alta confianza
                    return ResultadoBusqueda(
                        id_producto=id_,
                        nombre_producto=self.catalogo[id_],
                        nombre_buscado=nombre_buscado,
                        score_similitud=score,
                        metodo="contenido",
                        alternativas=[],
                        confianza="ALTA",
                    )

        # ─── PASO 3: Búsqueda Difusa Ponderada ────────────────────────────
        candidatos: List[Tuple[uuid.UUID, str, float]] = []

        for id_, nombre_catalogo in self.catalogo.items():
            # Usar normalizado para la comparación
            nombre_catalogo_norm = self.catalogo_normalizado[id_]

            score = MetricasSimilitud.score_ponderado(
                nombre_norm, nombre_catalogo_norm
            )

            if score >= threshold_minimo:
                candidatos.append((id_, nombre_catalogo, score))

        if not candidatos:
            # Sin resultados
            return ResultadoBusqueda(
                id_producto=None,
                nombre_producto="",
                nombre_buscado=nombre_buscado,
                score_similitud=0.0,
                metodo="difuso_sin_resultado",
                alternativas=[],
                confianza="NO_ENCONTRADO",
            )

        # Ordenar por score descendente
        candidatos.sort(key=lambda x: x[2], reverse=True)

        # Mejora: tomar el mejor
        id_ganador, nombre_ganador, score_ganador = candidatos[0]

        # Alternativas (los otros candidatos)
        alternativas = [(nombre, score, id_) for id_, nombre, score in candidatos[1:limitar_alternativas+1]]

        # Determinar confianza según el score
        if score_ganador >= 0.90:
            confianza = "ALTA"
        elif score_ganador >= 0.75:
            confianza = "MEDIA"
        else:
            confianza = "BAJA"

        return ResultadoBusqueda(
            id_producto=id_ganador,
            nombre_producto=nombre_ganador,
            nombre_buscado=nombre_buscado,
            score_similitud=score_ganador,
            metodo="difuso_ponderado",
            alternativas=alternativas,
            confianza=confianza,
        )

    def buscar_multiples(self, nombres: List[str]) -> List[ResultadoBusqueda]:
        """
        Busca múltiples nombres de productos en batch.
        Útil para procesar listas de cotizaciones.

        Args:
            nombres: lista de strings a buscar

        Returns:
            lista de ResultadoBusqueda (uno por nombre)
        """
        return [self.buscar(nombre) for nombre in nombres]


print("✅ BuscadorProductos definido")

### CELDA 6.5.6: Gestor integrado con `DatabaseManager`

Esta clase carga el catálogo desde BD usando el `DatabaseManager` del bot, y expone métodos para buscar productos por similitud y registrar cotizaciones con resolución automática del `id_producto` correcto.

In [ ]:
class GestorBusquedaProductos:
    """
    Integra BuscadorProductos con DatabaseManager para:
    1. Cargar catálogo desde BD
    2. Buscar productos por similitud
    3. Persistir cotizaciones con producto correcto identificado
    """

    def __init__(self, db_manager):
        """
        Args:
            db_manager: instancia de DatabaseManager (del bot principal)
        """
        self.db_manager = db_manager
        self.buscador = None
        self._cargar_catalogo()

    def _cargar_catalogo(self):
        """Carga el catálogo de productos desde BD."""
        try:
            with self.db_manager.get_session() as session:
                # NOTA: El modelo Producto ya está definido globalmente en el notebook
                # (CELDA 5: MODELOS DE BASE DE DATOS). No requiere import explícito.
                productos = session.query(Producto).all()
                catalogo = {p.id_producto: p.nombre for p in productos}

                self.buscador = BuscadorProductos(catalogo)
                logger.info(f"📚 Catálogo BD cargado: {len(catalogo)} productos")
        except Exception as e:
            logger.error(f"Error cargando catálogo: {e}")
            self.buscador = BuscadorProductos({})  # catálogo vacío

    def recargar_catalogo(self):
        """Recarga el catálogo (después de cambios en BD)."""
        self._cargar_catalogo()

    def buscar_producto(self, nombre: str, threshold: float = 0.80) -> Optional[uuid.UUID]:
        """
        Busca un producto y devuelve su UUID si alcanza el threshold.

        Args:
            nombre: nombre del producto a buscar
            threshold: score mínimo para considerarlo válido (default 0.80)

        Returns:
            uuid.UUID si encontró, None si no alcanzó el threshold
        """
        if not self.buscador:
            logger.warning("Buscador no inicializado")
            return None

        resultado = self.buscador.buscar(nombre)

        if resultado.es_valido(threshold):
            logger.info(
                f"✅ Producto encontrado: {resultado.nombre_buscado!r} → "
                f"{resultado.nombre_producto!r} ({resultado.score_similitud:.2%})"
            )
            return resultado.id_producto
        else:
            logger.warning(
                f"⚠️  Score insuficiente para '{resultado.nombre_buscado}': "
                f"{resultado.score_similitud:.2%} < {threshold}"
            )
            return None

    def obtener_producto_con_alternativas(self, nombre: str) -> ResultadoBusqueda:
        """
        Devuelve el resultado completo incluyendo alternativas.
        Útil para UIs o decisiones manuales.
        """
        if not self.buscador:
            logger.warning("Buscador no inicializado")
            return ResultadoBusqueda(
                id_producto=None,
                nombre_producto="",
                nombre_buscado=nombre,
                score_similitud=0.0,
                metodo="error",
                alternativas=[],
                confianza="NO_ENCONTRADO",
            )

        return self.buscador.buscar(nombre)

    def registrar_cotizacion_con_busqueda(
        self,
        id_interaccion: str,
        id_ferreteria: str,
        producto_nombre: str,  # Nombre que viene del extractor (puede tener errores)
        marca_nombre: str,
        precio: float,
        regional: str,
        disponibilidad: Optional[str] = None,
        confianza: Optional[float] = None,
        info_solicitada: Optional[str] = None,
        cod_municipio: Optional[str] = None,
        threshold_busqueda: float = 0.75,  # threshold más bajo que validación final
    ) -> Optional[Dict]:
        """
        Versión mejorada de registrar_cotizacion que:
        1. Busca el producto por similitud (si no existe exactamente)
        2. Usa el ID correcto del catálogo
        3. Persiste la cotización

        Args:
            threshold_busqueda: score mínimo para usar el producto encontrado
            (otros args): igual que DatabaseManager.registrar_cotizacion

        Returns:
            Dict con la fila insertada, o None si falló
        """
        # 1) Buscar producto con similitud
        id_producto = self.buscar_producto(producto_nombre, threshold=threshold_busqueda)

        if id_producto is None:
            logger.warning(
                f"❌ No se encontró producto '{producto_nombre}' "
                f"(threshold={threshold_busqueda}). Cotización rechazada."
            )
            return None

        # 2) Usar DatabaseManager.registrar_cotizacion con el ID encontrado
        return self.db_manager.registrar_cotizacion(
            id_interaccion=id_interaccion,
            id_ferreteria=id_ferreteria,
            producto_nombre=producto_nombre,
            marca_nombre=marca_nombre,
            precio=precio,
            regional=regional,
            disponibilidad=disponibilidad,
            confianza=confianza,
            info_solicitada=info_solicitada,
            cod_municipio=cod_municipio,
            id_producto=id_producto,  # ← Pasar el ID encontrado
        )


print("✅ GestorBusquedaProductos definido")

### CELDA 6.5.7: Verificación rápida (no requiere BD)

Esta celda valida que las clases se cargaron correctamente con un catálogo de prueba en memoria. No toca la base de datos del bot.

In [ ]:
# ─── Verificación rápida con catálogo en memoria ──────────────────────────
try:
    _catalogo_demo = {
        uuid.uuid4(): "Cemento Argos",
        uuid.uuid4(): "Cemento Gris Portland",
        uuid.uuid4(): "Varilla 1/2 pulgada",
    }
    _buscador_demo = BuscadorProductos(_catalogo_demo)
    _r = _buscador_demo.buscar("cémento argós")
    print(f"✅ ProductSimilarity OK — match demo: {_r.nombre_producto!r} "
          f"(score={_r.score_similitud:.2%}, confianza={_r.confianza})")
    del _catalogo_demo, _buscador_demo, _r
except Exception as _e:
    print(f"⚠️  Verificación de ProductSimilarity falló: {_e}")

## CELDA 6.6: ✅ NUEVO — EXTRACTOR ACUMULATIVO DE COTIZACIONES POR TEXTO

**Problema que resuelve:** el extractor original (`AnthropicExtractionClient.extract_quote_info`)
recibe SOLO el último intercambio cliente↔bot. Cuando la información de la cotización
se reparte en varios turnos (cliente dice producto en T4, precio en T6, marca en T8),
el extractor ve cada pieza aislada, devuelve `null` en los campos faltantes, y la
transición `inicio → cotizacion` nunca se dispara. El bot termina re-preguntando
información que ya tenía.

**Solución:** un extractor que:
1. **Reconstruye estado acumulado** del historial completo de la conversación
2. **Detecta precios deterministamente** con regex (sin LLM)
3. **Empareja producto y marca contra catálogo** con fuzzy matching (sin LLM)
4. **Usa LLM solo como fallback** acotado al catálogo, cuando faltan datos
5. **Marca precios fuera de rango** como sospechosos para sanity check

Reusa `BuscadorProductos` y `MetricasSimilitud` de la celda 6.5.


### CELDA 6.6.1: Detector determinista de precios

Maneja todas las variantes vistas en conversaciones reales:
- Explícitos: `$32.000`, `32.000`, `32,000`, `3200000`
- Coloquiales: `32 mil`, `32mil`, `32k`, `32 lucas`
- Implícitos: `a 32`, `en 35`, `=32`, `queda en 32`
- Anota contexto: kg detectado (`50 kg`, `25 kg`) y unidad (`bulto`, `saco`)


In [ ]:
@dataclass
class PrecioDetectado:
    """Un candidato de precio detectado por regex en el texto."""
    valor: float                            # COP, normalizado a número
    texto_original: str                     # match crudo del regex
    contexto_unidad: Optional[str] = None   # "bulto", "saco" si se infirió
    contexto_kg: Optional[int] = None       # 50 o 25 si "50 kg" / "25 kg"
    posicion: int = 0                       # offset en el texto


class DetectorPrecios:
    """
    Detector determinista de precios en mensajes de WhatsApp en español
    colombiano. Probado contra casos reales del CSV de historial:
      $32.000, 32.000, 32 mil, 32mil, 32k, "a 32", "=32 mil", 250.000,
      3200000 (precio total), 23 mil, 32 lucas, etc.

    NO decide si el precio es válido — solo detecta candidatos.
    El sanity check de rango lo hace el llamador con `precio_unitario_plausible()`.
    """

    # Rango plausible para precio UNITARIO de un bulto de cemento en COP.
    # Fuera de este rango → marcar como sospechoso (probablemente sea total).
    PRECIO_UNITARIO_MIN = 15_000
    PRECIO_UNITARIO_MAX = 60_000

    SUFIJOS_MIL = {"mil", "k", "lucas", "luca"}

    def __init__(self):
        # Patrón: precio EXPLÍCITO con $ o con separadores ($32.000, 3200000)
        self._re_precio_completo = re.compile(
            r'\$?\s*'
            r'(\d{1,3}(?:[.,]\d{3})+|\d{4,})'
            r'(?:\s*pesos?)?',
            re.IGNORECASE
        )
        # Patrón: con sufijo coloquial (32 mil, 32k, 32 lucas)
        sufijos_pat = '|'.join(self.SUFIJOS_MIL)
        self._re_precio_sufijo = re.compile(
            r'(\d{1,3}(?:[.,]\d{1,3})?)'
            rf'\s*({sufijos_pat})\b',
            re.IGNORECASE
        )
        # Patrón: implícito ("a 32", "=32", "queda en 35")
        self._re_precio_implicito = re.compile(
            r'\b(?:a|en|por|=|queda(?:n)?\s+en)\s+'
            r'(\d{1,3})\b'
            r'(?!\s*(?:mil|k|lucas?|\.\d|,\d|\d))',
            re.IGNORECASE
        )
        # Contextos: kg y unidad
        self._re_kg = re.compile(
            r'(\d{1,3})\s*(?:kg|kilos?|kilogramos?)\b',
            re.IGNORECASE
        )
        self._re_unidad = re.compile(
            r'\b(bulto|saco|tonelada|ton|m3|metro\s+cubico)s?\b',
            re.IGNORECASE
        )

    @staticmethod
    def _normalizar_numero(texto: str) -> Optional[float]:
        """
        '32.000'    → 32000.0   (separador de miles, convención CO)
        '1,200,000' → 1200000.0
        '3200000'   → 3200000.0
        '32.5' / '32,5' → 32.5  (decimal, pero raro en precios)
        """
        t = texto.strip().replace('$', '').replace(' ', '')
        if not t:
            return None
        if re.fullmatch(r'\d{1,3}[.,]\d{1,2}', t):
            return float(t.replace(',', '.'))
        try:
            return float(t.replace('.', '').replace(',', ''))
        except ValueError:
            return None

    def detectar(self, texto: str) -> List[PrecioDetectado]:
        """Devuelve TODOS los candidatos de precio en orden de aparición."""
        if not texto:
            return []

        candidatos: List[PrecioDetectado] = []
        spans_usados: List[Tuple[int, int]] = []

        def _ocupa(start, end):
            return any(not (end <= s or start >= e) for s, e in spans_usados)

        # 1) Sufijo "mil/k/lucas" PRIMERO (más específico que dígitos solos)
        for m in self._re_precio_sufijo.finditer(texto):
            num = self._normalizar_numero(m.group(1))
            if num is None:
                continue
            candidatos.append(PrecioDetectado(
                valor=num * 1000,
                texto_original=m.group(0),
                posicion=m.start(),
            ))
            spans_usados.append(m.span())

        # 2) Precio completo con separadores ($32.000, 3200000)
        for m in self._re_precio_completo.finditer(texto):
            if _ocupa(*m.span()):
                continue
            num = self._normalizar_numero(m.group(1))
            if num is None or num < 1000:
                continue
            candidatos.append(PrecioDetectado(
                valor=num,
                texto_original=m.group(0),
                posicion=m.start(),
            ))
            spans_usados.append(m.span())

        # 3) Implícito ("a 32") — asumir miles si está en rango plausible
        for m in self._re_precio_implicito.finditer(texto):
            if _ocupa(*m.span()):
                continue
            num = self._normalizar_numero(m.group(1))
            if num is None:
                continue
            if 10 <= num <= 999:
                candidatos.append(PrecioDetectado(
                    valor=num * 1000,
                    texto_original=m.group(0),
                    posicion=m.start(),
                ))
                spans_usados.append(m.span())

        # 4) Anotar contextos por proximidad
        kg_matches = [(m.start(), m.end(), int(m.group(1)))
                      for m in self._re_kg.finditer(texto)]
        unidad_matches = [(m.start(), m.end(), m.group(1).lower())
                          for m in self._re_unidad.finditer(texto)]

        for c in candidatos:
            mejor_kg, mejor_dist_kg = None, 30
            for s, e, kg in kg_matches:
                dist = min(abs(s - c.posicion), abs(e - c.posicion))
                if dist < mejor_dist_kg:
                    mejor_dist_kg, mejor_kg = dist, kg
            c.contexto_kg = mejor_kg

            mejor_unidad, mejor_dist_u = None, 40
            for s, e, u in unidad_matches:
                dist = min(abs(s - c.posicion), abs(e - c.posicion))
                if dist < mejor_dist_u:
                    mejor_dist_u, mejor_unidad = dist, u
            c.contexto_unidad = mejor_unidad

        candidatos.sort(key=lambda x: x.posicion)
        return candidatos

    @classmethod
    def precio_unitario_plausible(cls, valor: float) -> bool:
        """¿Cae en el rango razonable para un bulto de cemento?"""
        return cls.PRECIO_UNITARIO_MIN <= valor <= cls.PRECIO_UNITARIO_MAX


print("✅ DetectorPrecios definido")


### CELDA 6.6.2: Gestor de marcas (análogo a `GestorBusquedaProductos`)

Carga las 16 marcas de `marcas_productos` y permite hacer fuzzy match contra
ellas. Reusa `BuscadorProductos` cambiando el catálogo (productos → marcas).


In [ ]:
class GestorBusquedaMarcas:
    """
    Análogo a GestorBusquedaProductos pero para la tabla `marcas_productos`.
    Carga el catálogo de marcas (filtrado por regional opcional) y permite
    resolver nombres con typos via fuzzy matching.
    """

    def __init__(self, db_manager, regional: Optional[str] = None):
        self.db_manager = db_manager
        self.regional = regional
        self.buscador = None
        self._cargar_catalogo()

    def _cargar_catalogo(self):
        try:
            with self.db_manager.get_session() as session:
                q = session.query(MarcaProducto)
                if self.regional:
                    q = q.filter(
                        (MarcaProducto.regional == self.regional)
                        | (MarcaProducto.regional.is_(None))
                    )
                marcas = q.all()
                # Dedup por nombre normalizado para no inflar el catálogo si
                # hay marcas duplicadas en distintas regionales/municipios.
                catalogo: Dict[uuid.UUID, str] = {}
                vistos: set = set()
                for m in marcas:
                    norm = NormalizadorNombres.normalizar(m.nombre_marca)
                    if norm in vistos:
                        continue
                    vistos.add(norm)
                    catalogo[m.id_marca] = m.nombre_marca
                self.buscador = BuscadorProductos(catalogo)
                logger.info(f"🏷️  Catálogo de marcas cargado: {len(catalogo)} marcas")
        except Exception as e:
            logger.error(f"Error cargando catálogo de marcas: {e}")
            self.buscador = BuscadorProductos({})

    def recargar_catalogo(self):
        self._cargar_catalogo()

    def buscar_marca(self, nombre: str,
                     threshold: float = 0.80) -> Optional[ResultadoBusqueda]:
        """
        Busca una marca por nombre. Devuelve ResultadoBusqueda completo
        (con score y alternativas) si supera el threshold, None si no.
        """
        if not self.buscador or not nombre:
            return None
        resultado = self.buscador.buscar(nombre, threshold_minimo=0.60)
        if resultado.es_valido(threshold):
            return resultado
        return None


print("✅ GestorBusquedaMarcas definido")


### CELDA 6.6.3: Estado acumulado de la cotización

Dataclass que va llenándose turno a turno. Al final de procesar el historial
completo, decimos si la cotización está "completa" (producto + marca + precio
unitario plausible) y disparamos la transición a `cotizacion`.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class EstadoExtraccionAcumulado:
    """
    Estado acumulado de una cotización a lo largo de la conversación.
    Cada campo guarda el último valor confirmado + en qué turno apareció.
    """
    producto_nombre: Optional[str] = None
    producto_id: Optional[uuid.UUID] = None
    producto_score: float = 0.0
    producto_turno: int = -1

    marca_nombre: Optional[str] = None
    marca_id: Optional[uuid.UUID] = None
    marca_score: float = 0.0
    marca_turno: int = -1

    precio_unitario: Optional[float] = None
    precio_turno: int = -1
    precio_sospechoso: bool = False  # fuera de rango plausible

    cantidad: Optional[float] = None
    unidad: Optional[str] = None
    kg_presentacion: Optional[int] = None  # 50, 25, etc.
    disponibilidad: Optional[str] = None

    # Diagnóstico: qué métodos se usaron y dónde
    fuentes: List[str] = field(default_factory=list)

    def es_completo(self) -> bool:
        """¿Tenemos los 3 campos mínimos para registrar cotización?"""
        return (
            self.producto_nombre is not None
            and self.marca_nombre is not None
            and self.precio_unitario is not None
            and self.precio_unitario > 0
        )

    def faltantes(self) -> List[str]:
        """Lista de campos faltantes para dar feedback útil al LLM fallback."""
        out = []
        if self.producto_nombre is None:
            out.append("producto")
        if self.marca_nombre is None:
            out.append("marca")
        if self.precio_unitario is None:
            out.append("precio")
        return out

    def resumen(self) -> str:
        """Línea de log compacta para diagnóstico."""
        p = self.producto_nombre or "?"
        m = self.marca_nombre or "?"
        pr = f"${self.precio_unitario:,.0f}" if self.precio_unitario else "?"
        flag = " ⚠SOSPECHOSO" if self.precio_sospechoso else ""
        return f"producto={p}, marca={m}, precio={pr}{flag}"


print("✅ EstadoExtraccionAcumulado definido")

### CELDA 6.6.4: Extractor acumulativo (orquestador)

El componente principal. Su método `extraer_de_historial(historial, mensaje_actual)`:

1. Itera turno a turno por el historial
2. Para cada mensaje del cliente, corre detección determinista:
   - Precios → `DetectorPrecios`
   - Producto → `BuscadorProductos` contra catálogo de productos (11)
   - Marca → `GestorBusquedaMarcas` contra catálogo de marcas (16)
3. Hace **merge** con el estado acumulado: lo nuevo refina lo viejo, no lo borra
4. Si al terminar faltan campos pero el último mensaje los podría tener,
   invoca al LLM **acotado al catálogo** como fallback
5. Devuelve el `EstadoExtraccionAcumulado` final

El bot conversacional sigue intacto. Esto solo cambia cómo se detectan
las transiciones de estado.


In [ ]:
class ExtractorTextoAcumulativo:
    """
    Extractor que reconstruye el estado de la cotización a partir del historial
    completo, no solo del último mensaje. Resuelve el problema de info repartida
    en varios turnos.

    Args al construir:
        gestor_productos: GestorBusquedaProductos (catálogo de los 11 productos)
        gestor_marcas: GestorBusquedaMarcas (catálogo de las 16 marcas)
        anthropic_client: AnthropicExtractionClient (solo como fallback)

    Thresholds:
        threshold_producto: score mínimo fuzzy para aceptar producto (default 0.75)
        threshold_marca: score mínimo fuzzy para aceptar marca (default 0.80)
    """

    def __init__(self,
                 gestor_productos,
                 gestor_marcas,
                 anthropic_client: Optional["AnthropicExtractionClient"] = None,
                 threshold_producto: float = 0.75,
                 threshold_marca: float = 0.80):
        self.gestor_productos = gestor_productos
        self.gestor_marcas = gestor_marcas
        self.anthropic_client = anthropic_client
        self.threshold_producto = threshold_producto
        self.threshold_marca = threshold_marca
        self.detector_precios = DetectorPrecios()

    # ------------------------------------------------------------------
    # Palabras genéricas que NO deben usarse aisladas para matching de producto.
    # "cemento" sola matchearía con CUALQUIER producto del catálogo (todos los
    # productos contienen la palabra). Necesitamos un calificador adicional.
    _PALABRAS_GENERICAS_PRODUCTO = {"cemento", "producto", "bulto", "saco", "kg", "kilos"}

    def _extraer_producto(self, texto: str) -> Optional[Tuple[uuid.UUID, str, float]]:
        """
        Busca el producto en el texto. Estrategia escalonada:
        1. Frase completa (sin filtros).
        2. Ventanas de 2-4 palabras.
        3. Palabras sueltas calificadoras (≥5 chars y NO genéricas), para
           capturar abreviaciones tipo "estruc" → "estructural".

        Filtra ventanas que tras normalizar quedan solo en palabras genéricas
        (p. ej. "cemento" sola), porque matchearían con cualquier producto.
        """
        if not texto or not self.gestor_productos or not self.gestor_productos.buscador:
            return None

        candidatos: List[ResultadoBusqueda] = []

        def _es_consulta_inutil(consulta: str) -> bool:
            """¿La consulta es solo palabras genéricas? Si sí, descartarla."""
            try:
                norm = NormalizadorNombres.normalizar(consulta)
            except Exception:
                return False
            tokens = [t for t in norm.split() if t]
            if not tokens:
                return True
            # Si todos los tokens son genéricos, la consulta no aporta señal
            return all(t in self._PALABRAS_GENERICAS_PRODUCTO for t in tokens)

        def _intentar(consulta: str):
            if not consulta or _es_consulta_inutil(consulta):
                return
            r = self.gestor_productos.buscador.buscar(consulta, threshold_minimo=0.60)
            if r.id_producto and r.score_similitud >= self.threshold_producto:
                candidatos.append(r)

        # 1) Frase completa
        _intentar(texto)

        # 2) Ventanas de 4..2 palabras
        palabras = texto.split()
        for n in (4, 3, 2):
            for i in range(len(palabras) - n + 1):
                ventana = ' '.join(palabras[i:i+n])
                if len(ventana) < 5:
                    continue
                _intentar(ventana)

        # 3) Palabras sueltas SIGNIFICATIVAS (>=5 chars, no genéricas).
        # Esto captura abreviaciones tipo "estruc" → "Cemento estructural max"
        # via partial_ratio interno del BuscadorProductos.
        for palabra in palabras:
            limpia = palabra.strip(".,;:¿?¡!").lower()
            if len(limpia) < 5:
                continue
            if limpia in self._PALABRAS_GENERICAS_PRODUCTO:
                continue
            _intentar(limpia)

        # 4) Fallback: comparar palabras significativas contra los nombres
        # ORIGINALES del catálogo (sin pasar por NormalizadorNombres). Esto
        # es necesario porque algunos catálogos tienen palabras clave que
        # el normalizador trata como sinónimos vacíos (ej. "estructural",
        # "portland"). partial_ratio encuentra abreviaciones como
        # "estruc" → "estructural" sobre el nombre crudo.
        try:
            from fuzzywuzzy import fuzz as _fz
            for palabra in palabras:
                limpia = palabra.strip(".,;:¿?¡!").lower()
                if len(limpia) < 5 or limpia in self._PALABRAS_GENERICAS_PRODUCTO:
                    continue
                for pid, pnombre in self.gestor_productos.buscador.catalogo.items():
                    pn_low = pnombre.lower()
                    pr = _fz.partial_ratio(limpia, pn_low) / 100.0
                    # Exigir alta confianza para palabra suelta sobre nombre
                    # original (es una señal débil, debe estar muy contenida).
                    if pr >= 0.90:
                        # Construir un ResultadoBusqueda manual
                        rb = ResultadoBusqueda(
                            id_producto=pid,
                            nombre_producto=pnombre,
                            nombre_buscado=limpia,
                            score_similitud=pr * 0.85,  # penalizar señal indirecta
                            metodo="partial_directo",
                            alternativas=[],
                            confianza="MEDIA" if pr >= 0.95 else "BAJA",
                        )
                        if rb.score_similitud >= self.threshold_producto:
                            candidatos.append(rb)
        except Exception as e:
            logger.debug(f"Fallback partial_ratio falló: {e}")

        if not candidatos:
            return None
        # Preferir el match con más palabras coincidentes (más específico).
        # En empate de score, gana el de nombre_producto más largo (más específico).
        candidatos.sort(
            key=lambda x: (x.score_similitud, len(x.nombre_producto)),
            reverse=True
        )
        ganador = candidatos[0]
        return (ganador.id_producto, ganador.nombre_producto, ganador.score_similitud)

    # ------------------------------------------------------------------
    def _extraer_marca(self, texto: str) -> Optional[Tuple[uuid.UUID, str, float]]:
        """Devuelve (id, nombre, score) de la marca si la encuentra."""
        if not texto or not self.gestor_marcas or not self.gestor_marcas.buscador:
            return None
        candidatos: List[ResultadoBusqueda] = []
        # Las marcas suelen ser 1-2 palabras (Argos, Cemex, San Marcos)
        palabras = texto.split()
        for n in (2, 1):
            for i in range(len(palabras) - n + 1):
                ventana = ' '.join(palabras[i:i+n])
                if len(ventana) < 3:
                    continue
                r = self.gestor_marcas.buscador.buscar(ventana, threshold_minimo=0.65)
                if r.id_producto and r.score_similitud >= self.threshold_marca:
                    candidatos.append(r)
        if not candidatos:
            return None
        candidatos.sort(key=lambda x: x.score_similitud, reverse=True)
        ganador = candidatos[0]
        return (ganador.id_producto, ganador.nombre_producto, ganador.score_similitud)

    # ------------------------------------------------------------------
    def _seleccionar_precio_unitario(self,
                                     precios: List[PrecioDetectado]
                                     ) -> Optional[PrecioDetectado]:
        """
        Dada una lista de precios detectados en un mensaje, elige el más
        probable como precio UNITARIO.

        Heurística:
        - Preferir precios en rango plausible ($15k–$60k)
        - Entre los plausibles, preferir el que tenga contexto "kg=50"
          (presentación estándar) o "bulto"
        - Si NINGUNO está en rango, devolver el menor de los grandes (sospechoso)
        """
        if not precios:
            return None

        plausibles = [p for p in precios
                      if DetectorPrecios.precio_unitario_plausible(p.valor)]

        if plausibles:
            # Preferir 50kg, luego con unidad bulto, luego el primero
            for p in plausibles:
                if p.contexto_kg == 50:
                    return p
            for p in plausibles:
                if p.contexto_unidad in ("bulto", "saco"):
                    return p
            return plausibles[0]

        # Ningún precio en rango → marcar como sospechoso pero devolverlo
        # Probablemente es precio total. El llamador decide qué hacer.
        precios_ordenados = sorted(precios, key=lambda p: p.valor)
        return precios_ordenados[0]

    # ------------------------------------------------------------------
    def _procesar_turno(self, mensaje_cliente: str, turno: int,
                        estado: EstadoExtraccionAcumulado):
        """Aplica detección determinista a un mensaje y mergea con el estado."""
        if not mensaje_cliente:
            return

        # Producto (refresca solo si encontramos uno con score ≥ al actual)
        prod = self._extraer_producto(mensaje_cliente)
        if prod:
            pid, pnombre, pscore = prod
            if pscore >= estado.producto_score:
                estado.producto_id = pid
                estado.producto_nombre = pnombre
                estado.producto_score = pscore
                estado.producto_turno = turno
                estado.fuentes.append(f"T{turno}:producto:fuzzy({pscore:.2f})")

        # Marca
        marca = self._extraer_marca(mensaje_cliente)
        if marca:
            mid, mnombre, mscore = marca
            if mscore >= estado.marca_score:
                estado.marca_id = mid
                estado.marca_nombre = mnombre
                estado.marca_score = mscore
                estado.marca_turno = turno
                estado.fuentes.append(f"T{turno}:marca:fuzzy({mscore:.2f})")

        # Precios
        precios = self.detector_precios.detectar(mensaje_cliente)
        elegido = self._seleccionar_precio_unitario(precios)
        if elegido:
            plausible = DetectorPrecios.precio_unitario_plausible(elegido.valor)
            # Si ya tenemos un precio plausible y el nuevo NO lo es, no sobrescribir.
            if estado.precio_unitario and not estado.precio_sospechoso and not plausible:
                pass  # mantener el viejo
            else:
                estado.precio_unitario = elegido.valor
                estado.precio_sospechoso = not plausible
                estado.precio_turno = turno
                if elegido.contexto_kg:
                    estado.kg_presentacion = elegido.contexto_kg
                if elegido.contexto_unidad:
                    estado.unidad = elegido.contexto_unidad
                estado.fuentes.append(
                    f"T{turno}:precio:{elegido.valor:.0f}"
                    f"{'⚠' if not plausible else ''}"
                )

    # ------------------------------------------------------------------
    def extraer_de_historial(self,
                             historial: List[Dict[str, str]],
                             mensaje_actual: str = "",
                             usar_llm_fallback: bool = True
                             ) -> EstadoExtraccionAcumulado:
        """
        Reconstruye el estado de la cotización a partir del historial completo.

        Args:
            historial: lista de dicts {role: "user"|"assistant", content: str}
                       (formato que devuelve DatabaseManager.obtener_historial_reciente)
            mensaje_actual: mensaje del cliente que está siendo procesado AHORA
                            (puede no estar todavía en `historial` si llamamos
                             ANTES de guardarlo)
            usar_llm_fallback: si True, invoca al LLM cuando faltan campos
                               tras la fase determinista

        Returns:
            EstadoExtraccionAcumulado con la mejor info disponible
        """
        estado = EstadoExtraccionAcumulado()

        # 1) Procesar TODOS los mensajes del cliente en el historial
        turno = 0
        for msg in historial or []:
            if msg.get("role") == "user":
                turno += 1
                contenido = msg.get("content", "") or ""
                # Saltar marcadores sintéticos de outreach
                if contenido.startswith("[OUTREACH") or contenido.startswith("[imagen"):
                    continue
                self._procesar_turno(contenido, turno, estado)

        # 2) Procesar el mensaje actual (que puede no estar en historial todavía)
        if mensaje_actual:
            turno += 1
            self._procesar_turno(mensaje_actual, turno, estado)

        # 3) Si está completo, listo
        if estado.es_completo():
            logger.info(f"✅ Extractor acumulativo (determinista): {estado.resumen()}")
            return estado

        # 4) Fallback a LLM SOLO si faltan campos Y hay un cliente disponible
        if usar_llm_fallback and self.anthropic_client and (mensaje_actual or historial):
            faltantes = estado.faltantes()
            logger.info(
                f"🤖 Extractor acumulativo: faltantes={faltantes} → fallback LLM"
            )
            self._fallback_llm(estado, historial, mensaje_actual)

        logger.info(f"📊 Extractor acumulativo (final): {estado.resumen()}")
        return estado

    # ------------------------------------------------------------------
    def _fallback_llm(self,
                      estado: EstadoExtraccionAcumulado,
                      historial: List[Dict[str, str]],
                      mensaje_actual: str):
        """
        Invoca al LLM SOLO para los campos que faltan, con el catálogo como
        opciones cerradas. Esto reduce alucinaciones y costo.
        """
        if not self.anthropic_client:
            return

        # Tomar últimos N mensajes del cliente para contexto
        mensajes_cliente = []
        for msg in (historial or [])[-10:]:
            if msg.get("role") == "user":
                c = msg.get("content", "")
                if c and not c.startswith("[OUTREACH") and not c.startswith("[imagen"):
                    mensajes_cliente.append(c)
        if mensaje_actual:
            mensajes_cliente.append(mensaje_actual)

        if not mensajes_cliente:
            return

        contexto = "\n".join(f"- {m}" for m in mensajes_cliente[-6:])

        # Catálogos como opciones cerradas
        productos_lista = []
        if self.gestor_productos and self.gestor_productos.buscador:
            productos_lista = list(self.gestor_productos.buscador.catalogo.values())
        marcas_lista = []
        if self.gestor_marcas and self.gestor_marcas.buscador:
            marcas_lista = list(self.gestor_marcas.buscador.catalogo.values())

        prompt = (
            "Eres un extractor de cotizaciones de cemento. Te paso una conversación "
            "de WhatsApp entre una FERRETERÍA (vendedora) y un BOT comprador. "
            "Tu trabajo es extraer SOLO los campos que aún faltan.\n\n"
            f"PRODUCTO ya identificado: {estado.producto_nombre or 'NO'}\n"
            f"MARCA ya identificada: {estado.marca_nombre or 'NO'}\n"
            f"PRECIO unitario ya identificado: {estado.precio_unitario or 'NO'}\n\n"
            "OPCIONES VÁLIDAS DE PRODUCTO (elige una EXACTAMENTE como aparece, o null):\n"
            f"{productos_lista}\n\n"
            "OPCIONES VÁLIDAS DE MARCA (elige una EXACTAMENTE como aparece, o null):\n"
            f"{marcas_lista}\n\n"
            "Devuelve JSON ESTRICTO con SOLO los campos faltantes:\n"
            '{"producto": str|null, "marca": str|null, "precio_unitario": float|null}\n\n'
            "REGLAS:\n"
            "- Si un campo ya está identificado, devuélvelo igual (no lo cambies).\n"
            "- Si NO encuentras un campo en la conversación, devuelve null.\n"
            "- precio_unitario en COP, sin separadores. '32 mil' → 32000.\n"
            "- NO inventes. Si dudas, null.\n"
            "- producto y marca DEBEN coincidir EXACTAMENTE con las opciones válidas.\n\n"
            f"CONVERSACIÓN:\n{contexto}"
        )

        try:
            response = self.anthropic_client.client.messages.create(
                model=self.anthropic_client.config.model_name,
                max_tokens=200,
                temperature=0.1,
                messages=[{"role": "user", "content": prompt}],
            )
            text = response.content[0].text.strip()
            for fence in ('```json', '```'):
                if text.startswith(fence):
                    text = text[len(fence):].lstrip()
            if text.endswith('```'):
                text = text[:-3].rstrip()
            data = json.loads(text)

            # Aplicar SOLO si llena un hueco
            if estado.producto_nombre is None:
                p = data.get("producto")
                if p and self.gestor_productos:
                    pid = self.gestor_productos.buscar_producto(p, threshold=0.85)
                    if pid:
                        estado.producto_id = pid
                        estado.producto_nombre = p
                        estado.producto_score = 0.85
                        estado.fuentes.append("LLM:producto")

            if estado.marca_nombre is None:
                m = data.get("marca")
                if m and self.gestor_marcas:
                    rb = self.gestor_marcas.buscar_marca(m, threshold=0.85)
                    if rb:
                        estado.marca_id = rb.id_producto
                        estado.marca_nombre = rb.nombre_producto
                        estado.marca_score = rb.score_similitud
                        estado.fuentes.append("LLM:marca")

            if estado.precio_unitario is None:
                pr = data.get("precio_unitario")
                try:
                    pr_f = float(pr) if pr is not None else None
                except (TypeError, ValueError):
                    pr_f = None
                if pr_f and pr_f > 0:
                    estado.precio_unitario = pr_f
                    estado.precio_sospechoso = (
                        not DetectorPrecios.precio_unitario_plausible(pr_f)
                    )
                    estado.fuentes.append("LLM:precio")

        except json.JSONDecodeError as e:
            logger.error(f"Fallback LLM: JSON inválido: {e}")
        except Exception as e:
            logger.error(f"Fallback LLM: error: {e}")


print("✅ ExtractorTextoAcumulativo definido")


## CELDA 7: WHATSAPP CLIENT (sin cambios)

In [ ]:
class WhatsAppClient:
    """Cliente para enviar mensajes por WhatsApp con rate limiting y retry"""

    def __init__(self, config: WhatsAppConfig):
        self.config = config
        self.config.validate()
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Bearer {config.token}",
            "Content-Type": "application/json"
        })
        self.last_request_time = 0.0
        self.min_request_interval = 0.05

    def _wait_for_rate_limit(self) -> None:
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_request_interval:
            time.sleep(self.min_request_interval - elapsed)
        self.last_request_time = time.time()

    @retry_on_failure(max_retries=3)
    def send_text_message(self, to: str, message: str) -> Dict:
        if not validate_phone_number(to):
            raise ValueError(f"Número inválido: {mask_phone(to)}")
        payload = {
            "messaging_product": "whatsapp",
            "to": to,
            "type": "text",
            "text": {"body": message}
        }
        logger.info(f"📤 Enviando mensaje a {mask_phone(to)}")
        self._wait_for_rate_limit()
        response = self.session.post(self.config.messages_url, json=payload, timeout=10)
        response.raise_for_status()
        result = response.json()
        msg_id = result.get('messages', [{}])[0].get('id', 'N/A')
        logger.info(f"✅ Enviado. ID: {msg_id}")
        return result

    def send_message(self, phone_number: str, message_text: str) -> bool:
        try:
            self.send_text_message(phone_number, message_text)
            return True
        except Exception as e:
            logger.error(f"Error enviando mensaje: {e}")
            return False

    @retry_on_failure(max_retries=3)
    def send_template_message(self, to: str, template_name: str,
                              language_code: str,
                              body_params: Optional[List[str]] = None) -> Dict:
        """
        ✅ v1.3: envía un mensaje basado en plantilla aprobada de Meta.

        Necesario para ABRIR conversación con un número que no ha escrito al
        bot en las últimas 24h (Meta bloquea texto libre en ese caso).

        Args:
          to:            número destino en formato sin "+", solo dígitos.
          template_name: nombre EXACTO de la plantilla aprobada en Meta.
          language_code: código de idioma EXACTO con que fue aprobada
                         (ej. "es_CO"). Si no coincide, Meta rechaza.
          body_params:   lista ordenada de strings que rellenan {{1}}, {{2}}, …
                         del cuerpo. Para la plantilla "saludo" (1 placeholder)
                         se pasa una lista de un solo string.

        Devuelve el JSON de la API; lanza excepción si la API responde error.
        """
        if not validate_phone_number(to):
            raise ValueError(f"Número inválido: {mask_phone(to)}")

        components = []
        if body_params:
            components.append({
                "type": "body",
                "parameters": [
                    {"type": "text", "text": str(p)} for p in body_params
                ],
            })

        payload = {
            "messaging_product": "whatsapp",
            "to": to,
            "type": "template",
            "template": {
                "name": template_name,
                "language": {"code": language_code},
                # `components` se omite si no hay parámetros (plantillas estáticas).
                **({"components": components} if components else {}),
            },
        }

        logger.info(
            f"📤 Enviando plantilla '{template_name}' [{language_code}] "
            f"a {mask_phone(to)} (params={len(body_params or [])})"
        )
        self._wait_for_rate_limit()
        response = self.session.post(self.config.messages_url, json=payload, timeout=10)
        # Si Meta rechaza la plantilla (no aprobada, idioma incorrecto, parámetros
        # mal contados), el cuerpo trae detalle. Lo logueamos antes de raise.
        if not response.ok:
            try:
                err_body = response.json()
            except Exception:
                err_body = response.text
            logger.error(
                f"❌ Plantilla rechazada por Meta (status={response.status_code}): {err_body}"
            )
        response.raise_for_status()
        result = response.json()
        msg_id = result.get('messages', [{}])[0].get('id', 'N/A')
        logger.info(f"✅ Plantilla enviada. ID: {msg_id}")
        return result

    @retry_on_failure(max_retries=3)
    def download_media(self, media_id: str) -> Optional[Tuple[bytes, str]]:
        """
        Descarga un archivo adjunto de WhatsApp Cloud API en 2 pasos:
          1) GET /{media_id}            → JSON con `url` temporal y `mime_type`
          2) GET <url> con Authorization → bytes del archivo
        Devuelve (bytes, mime_type) o None si falla.
        """
        if not media_id:
            return None
        try:
            meta_url = f"https://graph.facebook.com/{self.config.api_version}/{media_id}"
            self._wait_for_rate_limit()
            r = self.session.get(meta_url, timeout=10)
            r.raise_for_status()
            meta = r.json()
            file_url = meta.get("url")
            mime_type = meta.get("mime_type", "application/octet-stream")
            if not file_url:
                logger.error("download_media: respuesta sin URL")
                return None
            self._wait_for_rate_limit()
            # El bearer ya está en self.session.headers
            r2 = self.session.get(file_url, timeout=30)
            r2.raise_for_status()
            logger.info(f"📥 Descargado media {media_id} ({mime_type}, {len(r2.content)} bytes)")
            return r2.content, mime_type
        except Exception as e:
            logger.error(f"Error descargando media {media_id}: {e}")
            return None

print("✅ WhatsAppClient definido")


# =============================================================================
# ✅ NUEVO: MessageDispatcher
# Cola in-memory + worker thread daemon que maneja TRES tipos de payload:
#
#   1) kind='text'  → envío directo de texto libre. Sin delay extra: el delay
#                     ya lo absorbió la ventana de escucha del intent.
#   2) kind='template' → envío de plantilla aprobada (outreach). Aplica
#                     outreach_delay_* antes del envío físico.
#   3) kind='inbound_intent' (✅ v1.4.2) → INTENT de respuesta diferida.
#                     Cuando se desencola, llama al callback registrado
#                     (processor._procesar_inbound_intent) que internamente
#                     genera la respuesta con IA y la encola como 'text'.
#                     Si llegan más mensajes del MISMO recipient mientras
#                     este intent está pendiente, el processor lo cancela
#                     vía cancel_inbound_intent_for() y encola uno nuevo
#                     con process_at reiniciado (debounce).
#
# Esto reemplaza el viejo delay intra-chat: ya no esperamos antes Y después
# de generar la respuesta. Esperamos UNA vez (listen_window_*), durante esa
# espera coalescemos mensajes, y al final enviamos inmediatamente.
#
# Reglas globales que siguen aplicando a TODOS los envíos físicos:
#   - inter-chat (2..5 min) entre envíos a números distintos (anti-bloqueo Meta)
#   - gate horario: si el envío caería fuera de ventana, requeue al next_open()
#
# El webhook NUNCA bloquea: encola y responde 200 inmediato. Esto es esencial
# porque WhatsApp reintenta el webhook si tarda y procesaríamos varias veces.
# =============================================================================
import threading
import heapq
import random as _rnd


class MessageDispatcher:
    """Cola de envíos diferidos con delays humanos + gate horario + coalescencia inbound."""

    def __init__(self, wa_client: WhatsAppClient, config: DispatcherConfig,
                 hours_gate: Optional["OperatingHoursGate"] = None):
        self.wa_client = wa_client
        self.config = config
        self.config.validate()
        self.hours_gate = hours_gate  # ✅ v1.1: gate opcional
        # Heap de items: (send_at_epoch, seq, recipient, payload)
        # payload['kind'] ∈ {'text', 'template', 'inbound_intent'}
        self._heap: list = []
        self._seq = 0
        self._cv = threading.Condition()
        self._stop = threading.Event()
        self._last_physical_send_epoch: float = 0.0
        self._worker: Optional[threading.Thread] = None
        # ✅ v1.4.2: callback que dispara la generación + envío de la respuesta
        # cuando vence un inbound_intent. Lo registra MessageProcessor al
        # construirse. Firma: callback(recipient: str) -> None.
        self._inbound_intent_callback: Optional[callable] = None

    # ------------------------------------------------------------------
    # ✅ v1.4.2: registro del callback para inbound_intent.
    # Se llama una sola vez desde el bootstrap, tras crear el processor.
    # ------------------------------------------------------------------
    def set_inbound_intent_callback(self, callback) -> None:
        self._inbound_intent_callback = callback

    def enqueue(self, recipient: str, text: str) -> float:
        """Encola un mensaje de TEXTO LIBRE listo para enviar.

        ✅ v1.4.2: sin delay artificial. La 'sensación humana' ya la dio el
        listen_window del inbound_intent que precede a este envío. El único
        delay que aún se aplica es inter_chat entre destinatarios distintos
        (en el worker) + el gate horario.
        """
        if not validate_phone_number(recipient):
            logger.warning("Dispatcher: número inválido, mensaje descartado")
            return 0.0
        send_at = time.time()  # ya: el worker aplicará inter_chat si toca
        payload = {"kind": "text", "body": text}
        with self._cv:
            self._seq += 1
            heapq.heappush(self._heap, (send_at, self._seq, recipient, payload))
            self._cv.notify_all()
        logger.info(
            f"📨 Encolado [text] para {mask_phone(recipient)} "
            f"(envío inmediato, cola={len(self._heap)})"
        )
        return send_at

    def enqueue_template(self, recipient: str, template_name: str,
                         language_code: str,
                         body_params: Optional[List[str]] = None) -> float:
        """
        ✅ v1.3: encola un mensaje de PLANTILLA aprobada (outreach / primer
        contacto). Aplica outreach_delay_* y respeta inter_chat + gate horario.
        """
        if not validate_phone_number(recipient):
            logger.warning("Dispatcher: número inválido, plantilla descartada")
            return 0.0
        delay = _rnd.uniform(
            self.config.outreach_delay_min_s, self.config.outreach_delay_max_s
        )
        send_at = time.time() + delay
        payload = {
            "kind": "template",
            "name": template_name,
            "lang": language_code,
            "params": list(body_params or []),
        }
        with self._cv:
            self._seq += 1
            heapq.heappush(self._heap, (send_at, self._seq, recipient, payload))
            self._cv.notify_all()
        logger.info(
            f"📨 Encolado [template:{template_name}] para {mask_phone(recipient)} "
            f"en {delay:.0f}s (cola={len(self._heap)})"
        )
        return send_at

    # ------------------------------------------------------------------
    # ✅ v1.4.2: inbound_intent (coalescencia / debounce)
    # ------------------------------------------------------------------
    def enqueue_inbound_intent(self, recipient: str) -> float:
        """
        Encola un INTENT de respuesta diferida para `recipient`. Cuando venza,
        el worker invocará el callback registrado, que es quien lee el buffer
        acumulado del cliente, genera la respuesta IA y la encola como 'text'.

        El delay (listen_window_*) es la ventana durante la cual esperamos
        nuevos mensajes del mismo cliente. Si llegan, el processor llama a
        cancel_inbound_intent_for() y re-llama a este método (reinicia el timer).
        """
        if not validate_phone_number(recipient):
            logger.warning("Dispatcher: número inválido, intent descartado")
            return 0.0
        delay = _rnd.uniform(
            self.config.listen_window_min_s, self.config.listen_window_max_s
        )
        process_at = time.time() + delay
        payload = {"kind": "inbound_intent"}
        with self._cv:
            self._seq += 1
            heapq.heappush(self._heap, (process_at, self._seq, recipient, payload))
            self._cv.notify_all()
        logger.info(
            f"👂 Encolado [inbound_intent] para {mask_phone(recipient)} "
            f"escuchando {delay:.0f}s (cola={len(self._heap)})"
        )
        return process_at

    def cancel_inbound_intent_for(self, recipient: str) -> int:
        """
        Cancela los inbound_intent pendientes para `recipient`. Retorna
        cuántos canceló. Solo afecta a kind='inbound_intent' — los envíos
        físicos ya encolados (text/template) NO se cancelan, porque
        representan trabajo terminado a punto de salir.

        O(n) sobre el heap. n es pequeño (decenas) en la práctica.
        """
        with self._cv:
            before = len(self._heap)
            self._heap = [
                item for item in self._heap
                if not (
                    item[2] == recipient
                    and isinstance(item[3], dict)
                    and item[3].get("kind") == "inbound_intent"
                )
            ]
            heapq.heapify(self._heap)
            cancelled = before - len(self._heap)
            if cancelled > 0:
                self._cv.notify_all()
        if cancelled > 0:
            logger.info(
                f"🔄 Cancelados {cancelled} inbound_intent(s) pendientes para "
                f"{mask_phone(recipient)}"
            )
        return cancelled

    def start(self):
        """Arranca el worker (idempotente)."""
        if self._worker and self._worker.is_alive():
            return
        self._stop.clear()
        self._worker = threading.Thread(
            target=self._run, name="MsgDispatcher", daemon=True
        )
        self._worker.start()
        gate_info = "con gate horario" if self.hours_gate else "sin gate horario"
        logger.info(
            f"🚀 Dispatcher iniciado {gate_info} "
            f"(listen={self.config.listen_window_min_s}..{self.config.listen_window_max_s}s, "
            f"outreach_delay={self.config.outreach_delay_min_s}..{self.config.outreach_delay_max_s}s, "
            f"inter={self.config.inter_chat_min_s}..{self.config.inter_chat_max_s}s)"
        )

    def stop(self, timeout: float = 5.0):
        self._stop.set()
        with self._cv:
            self._cv.notify_all()
        if self._worker:
            self._worker.join(timeout=timeout)
        logger.info("🛑 Dispatcher detenido")

    def pending(self) -> int:
        with self._cv:
            return len(self._heap)

    def _seconds_until_next_open(self) -> float:
        """Devuelve segundos hasta el próximo `next_open` del gate. Solo se
        llama cuando ya sabemos que estamos fuera de ventana.

        ⚠️ TZ-aware: pasamos `now` aware en UTC; el gate lo convierte a
        Bogotá internamente y devuelve un datetime aware. La resta entre
        dos aware es correcta (no requiere conversión adicional).
        """
        try:
            now_dt = datetime.now(timezone.utc)
            next_open_dt = self.hours_gate.next_open(now_dt)
            secs = (next_open_dt - now_dt).total_seconds()
            return max(secs, 1.0)
        except Exception as e:
            logger.error(f"Dispatcher: error consultando next_open: {e}")
            return 300.0

    def _run(self):
        while not self._stop.is_set():
            with self._cv:
                while not self._heap and not self._stop.is_set():
                    self._cv.wait(timeout=30)
                if self._stop.is_set():
                    return

                send_at, seq, recipient, payload = self._heap[0]
                now = time.time()
                kind = payload.get("kind") if isinstance(payload, dict) else "text"

                # ✅ v1.4.2: los inbound_intent NO consumen el slot inter_chat.
                # No son envíos físicos: solo disparan generación de respuesta.
                # El envío físico ocurre cuando el callback haga enqueue() del
                # 'text' resultante; ahí sí se aplicará inter_chat.
                if kind == "inbound_intent":
                    if now < send_at:
                        self._cv.wait(timeout=send_at - now)
                        continue
                    heapq.heappop(self._heap)
                    # caer fuera del lock para invocar el callback
                else:
                    # Envío físico (text o template): respetar inter_chat
                    inter = (
                        _rnd.uniform(self.config.inter_chat_min_s, self.config.inter_chat_max_s)
                        if self._last_physical_send_epoch > 0 else 0.0
                    )
                    earliest_allowed = self._last_physical_send_epoch + inter
                    target = max(send_at, earliest_allowed)

                    if now < target:
                        self._cv.wait(timeout=target - now)
                        continue

                    # ✅ v1.1: gate horario (decisión 3a). Solo aplica a envíos físicos.
                    if self.hours_gate is not None and not self.hours_gate.is_open():
                        secs_until = self._seconds_until_next_open()
                        new_send_at = time.time() + secs_until
                        heapq.heapreplace(
                            self._heap, (new_send_at, seq, recipient, payload)
                        )
                        logger.info(
                            f"⏸️  Fuera de ventana: requeue mensaje para "
                            f"{mask_phone(recipient)} en {secs_until:.0f}s"
                        )
                        continue

                    heapq.heappop(self._heap)

            # ─── Fuera del lock ───────────────────────────────────────────
            if kind == "inbound_intent":
                # Disparar el callback que genera + encola la respuesta.
                # NO actualiza _last_physical_send_epoch: aún no hay envío real.
                cb = self._inbound_intent_callback
                if cb is None:
                    logger.error(
                        f"inbound_intent venció para {mask_phone(recipient)} "
                        f"pero no hay callback registrado — descartado"
                    )
                    continue
                try:
                    cb(recipient)
                except Exception as e:
                    logger.error(
                        f"Error procesando inbound_intent para "
                        f"{mask_phone(recipient)}: {e}"
                    )
                continue

            # kind in {text, template}: enviar físicamente
            try:
                if kind == "template":
                    self.wa_client.send_template_message(
                        to=recipient,
                        template_name=payload["name"],
                        language_code=payload["lang"],
                        body_params=payload.get("params") or [],
                    )
                else:
                    body = payload["body"] if isinstance(payload, dict) else payload
                    self.wa_client.send_text_message(recipient, body)
                self._last_physical_send_epoch = time.time()
                logger.info(
                    f"✉️  Enviado a {mask_phone(recipient)} "
                    f"(kind={kind}, restantes={self.pending()})"
                )
            except Exception as e:
                logger.error(f"Error en envío diferido a {mask_phone(recipient)}: {e}")


print("✅ MessageDispatcher definido (v1.4.2: coalescencia inbound + gate horario)")


## CELDA 8: ✅ NUEVO - ANTHROPIC AI CLIENT (Reemplaza a Gemini)

In [ ]:
# =============================================================================
# ESTRUCTURA IA CONTEXTO (según diagrama actualizado)
#
#                       [ BASE PROMPT ]
#                              |
#                  -------------------------
#                  |                       |
#         [ REGION PROMPT ]        [ ESTADO PROMPT ]
#                  |                       |
#                  -------- COMBINACIÓN ----
#                              |
#               [ HISTORIAL CONVERSACIÓN ]
#               (últimas N interacciones BD)
#                              |
#                    [ CONTEXTO DINÁMICO ]
#                              |
#                      [ PROMPT FINAL ]
#                              |
#                      [ MENSAJE OUTPUT ]
#
# Reglas aplicadas al MENSAJE OUTPUT (lo que llega al usuario):
#   1. Solo la primera letra tiene mayúscula
#   2. Errores de dedo casuales (no en última letra antes de '?')
#   3. Solo `?` como signo de puntuación, y solo si es necesario
#   4. Conciso: si se dice lo mismo en menos palabras, hazlo
#
# ✅ Cambios en esta celda respecto a la versión previa:
#   - FIX 2.9: prompt del extractor con roles correctos (la FERRETERÍA vende,
#     el BOT compra). Variables del payload renombradas para claridad.
#   - FIX 2.10: typos no se aplican en la última letra antes de un `?` final.
#   - FIX 2.11: si se pasó xml_path y el parseo falla, se levanta excepción
#     en lugar de caer silenciosamente a defaults.
#   - NUEVO: extract_quote_from_image() para fotos de cotización (JPG/PNG).
# =============================================================================

import random

# Vecinos de teclado QWERTY para errores de dedo plausibles
_TYPO_NEIGHBORS = {
    'a': ['s', 'q'], 'e': ['r', 'w'], 'i': ['o', 'u'], 'o': ['i', 'p'],
    'u': ['i', 'y'], 's': ['a', 'd'], 'n': ['m', 'b'], 'm': ['n'],
    'r': ['t', 'e'], 't': ['r', 'y'], 'l': ['k'], 'c': ['v', 'x'],
}

# Muletillas a recortar para hacer el mensaje conciso
_FILLERS_RE = [
    re.compile(r'\bpor favor\b', re.IGNORECASE),
    re.compile(r'\bme podri[ai]s?\b', re.IGNORECASE),
    re.compile(r'\bsi es posible\b', re.IGNORECASE),
    re.compile(r'\bsi me hace el favor\b', re.IGNORECASE),
    re.compile(r'\bquisiera saber\b', re.IGNORECASE),
    re.compile(r'\bme gustaria saber\b', re.IGNORECASE),
    re.compile(r'\bquisiera consultar\b', re.IGNORECASE),
    re.compile(r'\bnecesitaria\b', re.IGNORECASE),
    re.compile(r'\bdisculpe la molestia\b', re.IGNORECASE),
    re.compile(r'\bno se si\b', re.IGNORECASE),
    re.compile(r'\bla verdad es que\b', re.IGNORECASE),
    re.compile(r'\bestaba pensando que\b', re.IGNORECASE),
]


class AnthropicAIClient:
    """Cliente Anthropic Claude con composición de contexto dinámico."""

    def __init__(self, config: AnthropicConfig, xml_path: Optional[str] = None):
        self.config = config
        self.config.validate()
        self.client = anthropic.Anthropic(api_key=config.api_key)
        self.prompts: Dict[str, Any] = self._default_prompts()

        if xml_path:
            # ✅ FIX 2.11: si el caller pasó un path explícito, fallar ruidosamente
            # si el XML no se puede cargar. Antes caía silenciosamente a defaults
            # con un warning que nadie veía.
            try:
                self.prompts = self._load_xml_prompts(xml_path)
                logger.info(f"✅ XML de prompts cargado: {xml_path}")
                logger.info(f"   Regiones: {list(self.prompts['region'].keys())}")
                logger.info(f"   Estados: {list(self.prompts['estado'].keys())}")
            except FileNotFoundError as e:
                raise RuntimeError(
                    f"❌ XML de prompts no encontrado en {xml_path}. "
                    f"Si quieres usar prompts por defecto, NO pases xml_path."
                ) from e
            except ET.ParseError as e:
                raise RuntimeError(
                    f"❌ XML de prompts malformado en {xml_path}: {e}"
                ) from e

    # ------------------------------------------------------------------
    # CARGA DEL XML (BASE + REGION + ESTADO)
    # ------------------------------------------------------------------
    @staticmethod
    def _xml_block_to_text(elem) -> str:
        """Serializa un nodo XML como [TAG]\\ncontenido para cada hijo."""
        parts = []
        for child in elem:
            tag = child.tag.upper()
            inner = ''.join(child.itertext()).strip()
            if inner:
                parts.append(f"[{tag}]\n{inner}")
        return "\n\n".join(parts)

    def _load_xml_prompts(self, xml_path: str) -> Dict[str, Any]:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        prompts = {'base': '', 'region': {}, 'estado': {}}

        bp = root.find('base_prompt')
        if bp is not None:
            prompts['base'] = self._xml_block_to_text(bp)

        for r in root.findall('.//region'):
            name = (r.get('name') or '').upper()
            if name:
                prompts['region'][name] = self._xml_block_to_text(r)

        for s in root.findall('.//state'):
            name = (s.get('name') or '').lower()
            if name:
                prompts['estado'][name] = self._xml_block_to_text(s)

        if not prompts['base']:
            prompts['base'] = self._default_prompts()['base']

        return prompts

    def _default_prompts(self) -> Dict[str, Any]:
        return {
            'base': (
                "Eres el encargado de compras de la constructora MR. "
                "Hablas como colombiano real, no como IA. "
                "Mensajes cortos tipo whatsapp."
            ),
            'region': {},
            'estado': {},
        }

    # ------------------------------------------------------------------
    # COMBINACIÓN -> CONTEXTO DINÁMICO -> PROMPT FINAL
    # ------------------------------------------------------------------
    def _build_dynamic_context(self, region: str, estado: str,
                                historial: Optional[List[Dict[str, str]]] = None) -> str:
        """
        Combina BASE + REGION + ESTADO + HISTORIAL en un solo system prompt.
        """
        partes = []

        base = self.prompts.get('base', '').strip()
        if base:
            partes.append("===== BASE =====\n" + base)

        p_region = self.prompts.get('region', {}).get((region or '').upper(), '').strip()
        if p_region:
            partes.append(f"===== REGION ({region.upper()}) =====\n" + p_region)

        p_estado = self.prompts.get('estado', {}).get((estado or '').lower(), '').strip()
        if p_estado:
            partes.append(f"===== ESTADO ({estado.lower()}) =====\n" + p_estado)

        if historial:
            lineas = []
            for m in historial:
                # Roles correctos: el bot es la constructora MR (yo, comprador);
                # la ferretería (vendedor) es el "Cliente" en el sentido de
                # interlocutor del bot.
                rol = "Ferretería" if m.get("role") == "user" else "Yo (MR)"
                contenido = (m.get("content") or "").strip()
                if contenido:
                    lineas.append(f"{rol}: {contenido}")
            if lineas:
                partes.append(
                    "===== HISTORIAL DE CONVERSACIÓN =====\n"
                    "Lo siguiente es lo que ya hemos hablado con esta ferretería. "
                    "No vuelvas a preguntar datos que ya están confirmados aquí. "
                    "Si la ferretería pregunta '¿qué hemos hablado?' o "
                    "'¿qué no debes volver a preguntar?', resúmelo a partir de esto:\n"
                    + "\n".join(lineas)
                )

        # Refuerzo del estilo de salida (instrucciones para Claude)
        partes.append(
            "===== ESTILO DE SALIDA OBLIGATORIO =====\n"
            "- Responde como mensaje corto de whatsapp.\n"
            "- Una sola idea por mensaje.\n"
            "- No uses tildes ni emojis.\n"
            "- No expliques quién eres ni que eres una IA.\n"
            "- No saludes en cada turno, solo si recién inicia la conversación.\n"
            "- Devuelve solo el texto del mensaje y usa ?"
        )

        return "\n\n".join(partes)

    # ------------------------------------------------------------------
    # MENSAJE OUTPUT: post-procesado con las 4 reglas
    # ------------------------------------------------------------------
    @staticmethod
    def _strip_accents(text: str) -> str:
        return ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )

    def _inject_typos(self, text: str, rate: Optional[float] = None,
                      seed: Optional[int] = None) -> str:
        """
        ✅ FIX 2.10: errores de dedo casuales, pero NO en la última letra
        si el texto termina en '?'. Esto evita outputs feos como
        "cuanto cuesa?" donde la 't' final desaparece.
        """
        if rate is None:
            rate = self.config.typo_rate
        rng = random.Random(seed)

        # Detectar si el texto termina en '?' y proteger la última letra
        ends_with_q = text.rstrip().endswith('?')
        # Índice de la última letra alfabética antes del '?' (o del fin)
        last_alpha_idx = -1
        for i in range(len(text) - 1, -1, -1):
            if text[i].isalpha():
                last_alpha_idx = i
                break

        out = []
        for i, ch in enumerate(text):
            # Proteger la última letra antes del '?' final
            protected = (ends_with_q and i == last_alpha_idx)
            if ch.isalpha() and not protected and rng.random() < rate:
                roll = rng.random()
                low = ch.lower()
                if roll < 0.5 and low in _TYPO_NEIGHBORS:
                    repl = rng.choice(_TYPO_NEIGHBORS[low])
                    out.append(repl.upper() if ch.isupper() else repl)
                elif roll < 0.8:
                    out.append(ch)
                    out.append(ch)  # duplicada
                else:
                    continue  # omitida
            else:
                out.append(ch)
        return ''.join(out)

    @staticmethod
    def _make_concise(text: str) -> str:
        for pat in _FILLERS_RE:
            text = pat.sub('', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _format_output(self, text: str) -> str:
        """
        Aplica las 4 reglas al mensaje final:
        1) solo primera letra mayúscula
        2) errores de dedo casuales (no en última letra antes de '?')
        3) solo `?` permitido y solo si es necesario
        4) conciso
        """
        if not text:
            return ""

        # 0) sin tildes
        text = self._strip_accents(text)

        # 1) eliminar todos los signos excepto '?'
        text = text.replace('¿', '').replace('¡', '')
        text = re.sub(r"[.,;:!\"'`()\[\]{}\-—–/\\*_~|]", ' ', text)

        # 2) normalizar `?`: máximo uno al final, sin espacios previos
        text = re.sub(r'\?+', '?', text)
        text = re.sub(r'\s+\?', '?', text)
        text = text.lstrip('?').strip()

        # 3) colapsar espacios
        text = re.sub(r'\s+', ' ', text).strip()

        # 4) conciso (elimina muletillas)
        text = self._make_concise(text)

        # 4b) re-normalizar '?': la eliminación de muletillas puede haber
        # dejado espacio antes del signo (p.ej. "el precio , por favor ?" →
        # "el precio  ?"). Volver a colapsar contra el '?'.
        text = re.sub(r'\s+\?', '?', text)
        text = re.sub(r'\s+', ' ', text).strip()

        # 5) solo primera letra mayúscula
        text = text.lower()
        if text:
            text = text[0].upper() + text[1:]

        # 6) errores de dedo (al final, sobre el texto ya normalizado)
        text = self._inject_typos(text)

        return text

    # ------------------------------------------------------------------
    # ENTRADA PRINCIPAL
    # ------------------------------------------------------------------
    @retry_on_failure(max_retries=2)
    def get_response(self, user_message: str, region: str = "CENTRO",
                     estado: str = "inicio",
                     historial: Optional[List[Dict[str, str]]] = None) -> str:
        """
        El historial se usa de DOS formas complementarias:
          1) Como resumen textual dentro del system prompt
             (vía _build_dynamic_context)
          2) Como mensajes estructurados antes del mensaje actual
        """
        safe_message = sanitize_user_input(user_message)
        if not safe_message:
            return "Perdon no entendi"

        hist = historial or []
        if self.config.history_limit and len(hist) > self.config.history_limit * 2:
            hist = hist[-self.config.history_limit * 2:]

        system_prompt = self._build_dynamic_context(region, estado, hist)

        mensajes = list(hist) + [{"role": "user", "content": safe_message}]

        try:
            logger.info(
                f"🤖 Claude → región={region}, estado={estado}, "
                f"modelo={self.config.model_name}, "
                f"historial={len(hist)} mensajes"
            )
            response = self.client.messages.create(
                model=self.config.model_name,
                max_tokens=self.config.max_tokens,
                temperature=self.config.temperature,
                system=system_prompt,
                messages=mensajes,
            )
            raw = response.content[0].text.strip()
            logger.info(
                f"✅ tokens in={response.usage.input_tokens} "
                f"out={response.usage.output_tokens}"
            )
            return self._format_output(raw)
        except anthropic.APIError as e:
            logger.error(f"Error API Anthropic: {e}")
            return "Perdon estoy con problemas"
        except Exception as e:
            logger.error(f"Error inesperado: {e}")
            return "Perdon no entendi"

    # ------------------------------------------------------------------
    # ✅ v1.3: generación del parámetro {{1}} de la plantilla "saludo".
    # ------------------------------------------------------------------
    def generate_outreach_param(self,
                                productos_disponibles: Optional[List[str]] = None,
                                region: Optional[str] = None) -> str:
        """
        Genera el contenido CORTO que rellena {{1}} de la plantilla "saludo":

            "Hola, buen día. Quisiera consultar si tienen disponibilidad de:
             {{1}}. Además, ¿me podrían confirmar el precio? Quedo atento,
             muchas gracias."

        El resultado debe ser una frase MUY breve mencionando 2–4 productos
        de cemento del catálogo (ej. "cemento argos o portland"). NO debe
        repetir el saludo ni el cierre — esos ya los provee la plantilla.

        Args:
          productos_disponibles: lista de nombres de producto del catálogo BD.
            Si está vacía/None, se cae a un fallback seguro.
          region: opcional, solo informativo (no se usa para filtrar aquí).

        Devuelve un string sin saltos de línea, sin signos de puntuación al final,
        listo para inyectar como parámetro de plantilla.
        """
        # Fallback determinístico si no hay catálogo o si Claude falla.
        # Lo dejamos genérico (cualquier plaza colombiana lo entiende).
        FALLBACK = "cemento gris en presentación de 50 kg"

        productos_norm: List[str] = []
        for p in (productos_disponibles or []):
            if p and isinstance(p, str):
                productos_norm.append(p.strip())
        # Limitar el catálogo a un nº razonable de opciones para que Claude no
        # se vaya por las ramas y para abaratar tokens.
        productos_norm = productos_norm[:30]

        if not productos_norm:
            logger.info("generate_outreach_param: catálogo vacío → usando fallback")
            return FALLBACK

        # Prompt: instrucciones MUY constreñidas; queremos un fragmento, no
        # un mensaje. La plantilla ya pone el saludo y el cierre.
        system = (
            "Eres un comprador colombiano que arma una frase corta mencionando "
            "2 a 4 marcas o tipos de cemento de un catálogo. La frase irá "
            "insertada DENTRO de un mensaje pre-escrito justo después de "
            "\"disponibilidad de:\" y antes de \". Además, ¿me podrían…\". "
            "Reglas estrictas:\n"
            "- Devuelve SOLO la frase a insertar, nada más.\n"
            "- 3 a 12 palabras. Sin tildes, sin emojis, sin signos finales.\n"
            "- No saludes, no te despidas, no expliques nada.\n"
            "- Usa minúsculas excepto nombres propios de marca.\n"
            "- Conecta con \"o\" o con comas (ej. \"cemento argos o portland\").\n"
            "- Si en el catálogo hay marca y tipo, prioriza marcas reconocibles."
        )
        user = (
            "Catálogo disponible (escoge 2 a 4): "
            + " | ".join(productos_norm)
        )

        try:
            resp = self.client.messages.create(
                model=self.config.model_name,
                max_tokens=self.config.outreach_param_max_tokens,
                temperature=0.4,  # más bajo que el chat: queremos consistencia
                system=system,
                messages=[{"role": "user", "content": user}],
            )
            raw = (resp.content[0].text or "").strip()
            # Saneamiento: quitar comillas envolventes, puntos finales, saltos.
            raw = raw.strip().strip('"').strip("'").strip()
            raw = raw.replace("\n", " ").replace("\r", " ")
            # Colapsar espacios.
            raw = re.sub(r"\s+", " ", raw)
            # Quitar punto/coma final que rompería la lectura ("…: cemento argos.")
            raw = raw.rstrip(".,:;")
            # Validación mínima: longitud razonable y no vacío.
            if not raw or len(raw) < 3 or len(raw) > 200:
                logger.warning(
                    f"generate_outreach_param: salida fuera de rango "
                    f"(len={len(raw)}); usando fallback"
                )
                return FALLBACK
            logger.info(f"🧩 outreach_param generado: {raw!r}")
            return raw
        except Exception as e:
            logger.error(f"generate_outreach_param falló: {e}; usando fallback")
            return FALLBACK


class AnthropicExtractionClient(AnthropicAIClient):
    """
    Cliente especializado en EXTRACCIÓN de señales del mensaje de la ferretería.

    ✅ FIX 2.9: el system prompt y los nombres de variables ahora reflejan
    correctamente los roles:
      - LA FERRETERÍA es la que vende cemento y emite los precios
      - EL BOT (constructora MR) es el comprador, hace preguntas y agradece

    NO post-procesa el mensaje (no aplica las reglas de estilo de WhatsApp).
    """

    def __init__(self, config: AnthropicConfig):
        # super().__init__ no debe cargar XML aquí: el extractor usa su propio prompt
        super().__init__(config, xml_path=None)
        self.extraction_prompt = (
            "Eres un experto en extracción de datos de cotizaciones de ferretería en Colombia.\n\n"
            "CONTEXTO DE LA CONVERSACIÓN:\n"
            "- LA FERRETERÍA es la que VENDE cemento (y otros productos).\n"
            "- EL BOT (constructora MR) es el COMPRADOR que pregunta precios.\n"
            "- Vas a recibir DOS textos: lo que escribió la FERRETERÍA y lo que respondió el BOT.\n"
            "- Tu trabajo es extraer la información comercial que la FERRETERÍA aportó.\n\n"
            "Devuelve un JSON ESTRICTO con estas claves exactas:\n"
            "{\n"
            '  "producto": str|null,           // ej: "cemento", "varilla 1/2", "arena"\n'
            '  "marca": str|null,              // ej: "argos", "cemex", "diamante"\n'
            '  "cantidad": float|null,\n'
            '  "unidad": str|null,             // "bulto", "kg", "m", "ton", "saco"\n'
            '  "precio_unitario": float|null,  // en COP, sin separadores\n'
            '  "disponibilidad": str|null,     // "disponible", "agotado", "pedido"\n'
            '  "observaciones": str|null,\n'
            '  "es_despedida": bool,           // true si la FERRETERÍA cierra la conversación\n'
            '  "confirma_cierre": bool,        // true si la FERRETERÍA va a enviar la cotización formal o confirma el pedido\n'
            '  "confianza": float              // 0.0 a 1.0\n'
            "}\n\n"
            "REGLAS:\n"
            "- Devuelve SOLO JSON válido, sin markdown, sin texto antes ni después.\n"
            "- Usa null cuando el dato no esté presente. NO inventes.\n"
            "- 'es_despedida' es true si el mensaje de la FERRETERÍA termina la conversación: "
            "'gracias', 'hasta luego', 'no necesito nada más', 'no me interesa', 'ya no'. "
            "Un simple 'gracias' en medio de una negociación NO es despedida.\n"
            "- 'confirma_cierre' es true cuando la FERRETERÍA confirma que enviará la "
            "cotización formal o que procede con el pedido. Frases típicas: "
            "'ya le mando la cotización', 'le envío el pdf', 'listo le confirmo el pedido', "
            "'mando la propuesta', 'paso la cotización', 'hago la orden'. "
            "NO es true por simplemente dar un precio; debe haber compromiso explícito de "
            "enviar documento formal o cerrar la venta.\n"
            "- 'precio_unitario' debe ser número, no string. Convierte '$32.000' → 32000.\n"
            "- Si la FERRETERÍA solo saluda o pregunta cosas sin dar datos, todos los campos "
            "de cotización van en null y confianza baja (<0.3)."
        )

    @retry_on_failure(max_retries=3)
    def extract_quote_info(self, mensaje_ferreteria: str, respuesta_bot: str,
                           interaction_id: str,
                           historial: Optional[List[Dict[str, str]]] = None) -> Optional[Dict]:
        """
        Extrae señales del último intercambio.

        ✅ FIX 2.9: nombres de parámetros y del payload ahora reflejan roles:
        `mensaje_ferreteria` (lo que escribió la ferretería que vende) y
        `respuesta_bot` (lo que respondió el bot comprador).
        `interaction_id` se acepta por compatibilidad pero la persistencia
        la hace MessageProcessor con el DatabaseManager.
        """
        safe_ferreteria = sanitize_user_input(mensaje_ferreteria)
        safe_bot = sanitize_user_input(respuesta_bot, max_length=2000)
        try:
            logger.info("🤖 Extrayendo señales del mensaje...")
            # ✅ NUEVO: si llega historial, lo incluimos como contexto
            # para que `es_despedida` y `confirma_cierre` se evalúen sobre la
            # CONVERSACIÓN, no solo el último turno aislado.
            contexto_historial = ""
            if historial:
                ult_mensajes = []
                for h in historial[-8:]:
                    rol = "FERRETERÍA" if h.get("role") == "user" else "BOT"
                    contenido = (h.get("content") or "").strip()
                    if not contenido or contenido.startswith("[OUTREACH"):
                        continue
                    ult_mensajes.append(f"{rol}: {contenido}")
                if ult_mensajes:
                    contexto_historial = (
                        "CONTEXTO DE LA CONVERSACIÓN PREVIA (más antigua arriba):\n"
                        + "\n".join(ult_mensajes) + "\n\n"
                    )

            response = self.client.messages.create(
                model=self.config.model_name,
                max_tokens=self.config.max_tokens * 3,
                temperature=0.1,
                system=self.extraction_prompt,
                messages=[{
                    "role": "user",
                    "content": (
                        f"{contexto_historial}"
                        f"ÚLTIMO INTERCAMBIO:\n"
                        f"Mensaje de la FERRETERÍA (vendedor): {safe_ferreteria}\n\n"
                        f"Respuesta del BOT (comprador MR): {safe_bot}"
                    ),
                }],
            )
            text = response.content[0].text.strip()
            for fence in ('```json', '```'):
                if text.startswith(fence):
                    text = text[len(fence):].lstrip()
            if text.endswith('```'):
                text = text[:-3].rstrip()
            data = json.loads(text)
            data.setdefault("producto", None)
            data.setdefault("marca", None)
            data.setdefault("precio_unitario", None)
            data.setdefault("disponibilidad", None)
            data.setdefault("observaciones", None)
            data.setdefault("es_despedida", False)
            data.setdefault("confirma_cierre", False)
            data.setdefault("confianza", 0.0)
            logger.info(
                f"✅ Extracción → producto={data.get('producto')}, "
                f"marca={data.get('marca')}, precio={data.get('precio_unitario')}, "
                f"despedida={data.get('es_despedida')}, "
                f"cierre={data.get('confirma_cierre')}, "
                f"confianza={data.get('confianza')}"
            )
            return data
        except json.JSONDecodeError as e:
            logger.error(f"JSON inválido del extractor: {e}")
            return None
        except Exception as e:
            logger.error(f"Error en extracción: {e}")
            return None

    @staticmethod
    def tiene_cotizacion_completa(data: Dict) -> bool:
        """Determina si la extracción aporta precio + marca + producto usables."""
        if not data:
            return False
        precio = data.get("precio_unitario")
        marca = (data.get("marca") or "").strip()
        producto = (data.get("producto") or "").strip()
        try:
            precio_ok = precio is not None and float(precio) > 0
        except (TypeError, ValueError):
            precio_ok = False
        return precio_ok and bool(marca) and bool(producto)

    @staticmethod
    def tiene_confirmacion_cierre(data: Dict) -> bool:
        """True si la ferretería confirma cierre / envío de cotización formal por texto."""
        if not data:
            return False
        return bool(data.get("confirma_cierre"))

    # ------------------------------------------------------------------
    # PROMPT COMPARTIDO entre PDF e imagen para extraer líneas de cemento.
    # Lo separamos para que ambos flujos sean exactamente equivalentes.
    # ------------------------------------------------------------------
    _CEMENTO_EXTRACTION_PROMPT = (
        "Eres un experto en extracción de cotizaciones de ferretería. "
        "Recibirás una cotización (PDF o imagen) enviada por una ferretería "
        "colombiana. Tu tarea es localizar TODAS las líneas del producto "
        "CEMENTO (cualquier presentación: gris, blanco, portland, "
        "estructural, etc.) e ignorar TODOS los demás productos.\n\n"
        "Devuelve un JSON ESTRICTO con esta forma exacta:\n"
        "{\n"
        '  "lineas_cemento": [\n'
        "    {\n"
        '      "producto": str|null,\n'
        '      "marca": str|null,\n'
        '      "cantidad": float|null,\n'
        '      "unidad": str|null,             // "bulto", "saco", "kg", "ton"\n'
        '      "precio_unitario": float|null,  // COP, sin separadores\n'
        '      "disponibilidad": str|null,\n'
        '      "observaciones": str|null,      // notas solo de esta línea\n'
        '      "confianza": float              // 0.0 a 1.0 para esta línea\n'
        "    }\n"
        "  ],\n"
        '  "confianza_global": float           // qué tan confiado estás del documento en general\n'
        "}\n\n"
        "REGLAS CRÍTICAS:\n"
        "- Devuelve SOLO JSON válido, sin markdown ni texto adicional.\n"
        "- Incluye TODAS las líneas de cemento. Si hay 5 líneas (distintas marcas/"
        "presentaciones/precios), devuelve 5 entradas en `lineas_cemento`.\n"
        "- NO deduplicar: si el documento tiene la misma línea repetida, "
        "devuélvela repetida. La fidelidad manda.\n"
        "- Si el documento NO contiene cemento, devuelve `lineas_cemento: []` "
        "y `confianza_global < 0.3`.\n"
        "- 'precio_unitario' debe ser número, no string. '$32.000' → 32000.\n"
        "- Cada línea es independiente: NO infieras valores entre filas. "
        "Si una línea no tiene marca explícita pero otra sí, esa marca NO "
        "se copia a la primera.\n"
        "- Para imágenes de baja calidad o difíciles de leer, baja la "
        "confianza por línea pero igual extrae lo que veas."
    )

    def _parse_cemento_response(self, text: str) -> Optional[List[Dict]]:
        """Parsea la respuesta del extractor de cemento (común a PDF e imagen)."""
        for fence in ('```json', '```'):
            if text.startswith(fence):
                text = text[len(fence):].lstrip()
        if text.endswith('```'):
            text = text[:-3].rstrip()
        try:
            payload = json.loads(text)
        except json.JSONDecodeError as e:
            logger.error(f"JSON inválido en extracción de cemento: {e}")
            return None

        if isinstance(payload, list):
            lineas_raw = payload
            conf_global = None
        else:
            lineas_raw = payload.get("lineas_cemento", []) or []
            conf_global = payload.get("confianza_global")

        if not isinstance(lineas_raw, list):
            logger.warning("Extracción cemento: `lineas_cemento` no es lista, devuelvo []")
            return []

        lineas: List[Dict] = []
        for item in lineas_raw:
            if not isinstance(item, dict):
                continue
            item.setdefault("producto", None)
            item.setdefault("marca", None)
            item.setdefault("cantidad", None)
            item.setdefault("unidad", None)
            item.setdefault("precio_unitario", None)
            item.setdefault("disponibilidad", None)
            item.setdefault("observaciones", None)
            item.setdefault("es_despedida", False)
            item.setdefault("confianza", 0.0)
            lineas.append(item)

        logger.info(
            f"📄 Extracción → {len(lineas)} línea(s) de cemento "
            f"(confianza_global={conf_global})"
        )
        for i, l in enumerate(lineas, 1):
            logger.info(
                f"   [{i}/{len(lineas)}] producto={l.get('producto')}, "
                f"marca={l.get('marca')}, precio={l.get('precio_unitario')}, "
                f"confianza={l.get('confianza')}"
            )
        return lineas

    @retry_on_failure(max_retries=3)
    def extract_quote_from_pdf(self, pdf_bytes: bytes) -> Optional[List[Dict]]:
        """
        Extrae TODAS las líneas de CEMENTO desde un PDF de cotización.

        Devuelve:
          - List[Dict] (puede ser []) en caso de éxito.
          - None solo si falla la API o el JSON no se pudo parsear.
        """
        import base64
        if not pdf_bytes:
            return None
        try:
            pdf_b64 = base64.standard_b64encode(pdf_bytes).decode("utf-8")
            response = self.client.messages.create(
                model=self.config.model_name,
                max_tokens=self.config.max_tokens * 8,
                temperature=0.1,
                system=self._CEMENTO_EXTRACTION_PROMPT,
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "document",
                            "source": {
                                "type": "base64",
                                "media_type": "application/pdf",
                                "data": pdf_b64,
                            },
                        },
                        {
                            "type": "text",
                            "text": (
                                "Extrae TODAS las líneas de cemento del PDF "
                                "siguiendo las reglas del system prompt. Si hay "
                                "varias, todas; si no hay ninguna, lista vacía."
                            ),
                        },
                    ],
                }],
            )
            return self._parse_cemento_response(response.content[0].text.strip())
        except Exception as e:
            logger.error(f"Error en extract_quote_from_pdf: {e}")
            return None

    @retry_on_failure(max_retries=3)
    def extract_quote_from_image(self, image_bytes: bytes,
                                 mime_type: str = "image/jpeg") -> Optional[List[Dict]]:
        """
        ✅ NUEVO: extrae líneas de CEMENTO desde una IMAGEN (foto de cotización).

        Soporta los formatos que acepta la API de Anthropic:
        image/jpeg, image/png, image/gif, image/webp.

        Mismo contrato de retorno que extract_quote_from_pdf:
          - List[Dict] (puede ser []) en caso de éxito.
          - None si falla la API o el JSON no se pudo parsear.
        """
        import base64
        if not image_bytes:
            return None

        # Validar mime soportado
        SUPPORTED = {"image/jpeg", "image/png", "image/gif", "image/webp"}
        mime_clean = (mime_type or "").lower().split(";")[0].strip()
        if mime_clean not in SUPPORTED:
            logger.warning(
                f"Imagen con MIME no soportado por Claude vision: {mime_clean}. "
                f"Soportados: {SUPPORTED}"
            )
            return None

        try:
            img_b64 = base64.standard_b64encode(image_bytes).decode("utf-8")
            response = self.client.messages.create(
                model=self.config.model_name,
                max_tokens=self.config.max_tokens * 8,
                temperature=0.1,
                system=self._CEMENTO_EXTRACTION_PROMPT,
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": mime_clean,
                                "data": img_b64,
                            },
                        },
                        {
                            "type": "text",
                            "text": (
                                "Esta es una foto de una cotización enviada por la "
                                "ferretería. Extrae TODAS las líneas de cemento "
                                "siguiendo las reglas del system prompt. Si la "
                                "imagen no contiene cemento (es otra cosa: factura "
                                "general, foto del local, etc.) devuelve "
                                "`lineas_cemento: []` con confianza_global baja."
                            ),
                        },
                    ],
                }],
            )
            return self._parse_cemento_response(response.content[0].text.strip())
        except Exception as e:
            logger.error(f"Error en extract_quote_from_image: {e}")
            return None


print("✅ AnthropicAIClient + ExtractionClient listos (FIX roles, typos, fail-loud, +imagen)")


## CELDA 9: MESSAGE PROCESSOR Y WEBHOOK SERVER

In [ ]:
class MessageProcessor:
    """
    Procesa mensajes y coordina:
      - generación de respuesta (AnthropicAIClient)
      - extracción de señales (AnthropicExtractionClient)
      - persistencia (DatabaseManager: interacción + cotización)
      - transiciones de estado de la ferretería

    ✅ v1.1 — _handle_ai_flow polimórfico, dos modos:
      modo="outreach": itera ferreterías con `estado IS NULL` y no vetadas.
                       Construye user_message desde `topic`, ejecuta pasos 2–6
                       por cada una, y transiciona None → primer_mensaje al final.
      modo="inbound":  comportamiento original. Busca ferretería por teléfono,
                       valida veto, ejecuta pasos 2–6 con el (recipient, message)
                       que llegó del webhook.

    Reglas de transición aplicadas (modo inbound):
      None + texto del cliente              → primer_mensaje → inicio  (encadenado)
      primer_mensaje + texto del cliente    → inicio                    (cliente respondió broadcast)
      inicio + (precio + marca extraídos)   → cotizacion (+ persistir cotización)
      cotizacion + (image | document)       → cierre   (si la extracción produjo ≥1 línea)
      inicio + (image | document)           → cotizacion → cierre  (encadenado, si extracción produjo líneas)
      cualquier estado + despedida          → terminado (forzado)

    Optimización: no se ejecuta el extractor si el estado actual es
    `cierre` o `terminado` (ya no aporta señales útiles, ahorra tokens).
    """

    TIPOS_TEXTO = {"text"}
    TIPOS_ADJUNTO = {"image", "document"}

    # Estados donde NO tiene sentido correr el extractor de señales
    ESTADOS_SIN_EXTRACTOR = {EstadoFereteria.cierre, EstadoFereteria.terminado}

    def __init__(self, wa_client: WhatsAppClient, ai_client: AnthropicAIClient,
                 db_manager: DatabaseManager,
                 extraction_client: Optional[AnthropicExtractionClient] = None,
                 extractor_acumulativo: Optional["ExtractorTextoAcumulativo"] = None,
                 csv_cotizaciones_pdf: Optional[str] = None,
                 dispatcher: Optional["MessageDispatcher"] = None,
                 hours_gate: Optional["OperatingHoursGate"] = None):
        self.wa_client = wa_client
        self.ai_client = ai_client
        self.db_manager = db_manager
        self.extraction_client = extraction_client
        # ✅ NUEVO: extractor acumulativo (determinista + fallback LLM acotado).
        # Si está disponible, REEMPLAZA al extractor LLM de un solo turno para
        # detectar la transición inicio → cotizacion.
        self.extractor_acumulativo = extractor_acumulativo
        self.csv_cotizaciones_pdf = csv_cotizaciones_pdf
        self.dispatcher = dispatcher
        self.hours_gate = hours_gate  # ✅ v1.1: gate opcional
        # Cache in-memory como respaldo si la BD falla durante la verificación
        # de idempotencia. La fuente de verdad es la tabla webhook_events_processed.
        self.processed_messages: set = set()
        # ✅ v1.4.2: buffer de mensajes inbound pendientes de procesar por
        # recipient. Acumula los textos del 'burst' (mensajes en cascada del
        # mismo cliente dentro de la ventana de escucha). Cuando el dispatcher
        # dispara el inbound_intent vencido, _procesar_inbound_intent() lee y
        # vacía el buffer aquí. Protegido por _pending_lock para evitar race
        # entre el hilo del webhook y el hilo del dispatcher.
        #
        # Estructura:
        #   recipient(str) -> List[Dict] donde cada dict es
        #     {'text': str, 'interaction_id': str}
        # El interaction_id apunta a la fila ya persistida en BD (con
        # respuesta_ia=NULL). Cuando la IA finalmente genere la respuesta,
        # haremos UPDATE sobre el id de la ÚLTIMA fila — los anteriores
        # quedan como mensajes del usuario sin respuesta del bot, que es
        # semánticamente correcto: el bot solo respondió una vez, al final.
        self._pending_messages: Dict[str, List[Dict[str, str]]] = {}
        self._pending_lock = threading.Lock()

    # ------------------------------------------------------------------
    # Entrada del webhook
    # ✅ v1.1: gate horario estricto (decisión 1a). Si estamos fuera de
    #          ventana, descartamos el webhook entrante completamente.
    # ------------------------------------------------------------------
    def process_incoming(self, data: Dict):
        try:
            # ✅ v1.1: gate estricto (decisión 1a)
            if self.hours_gate is not None and not self.hours_gate.is_open():
                logger.info(
                    "🌙 Webhook entrante descartado: bot fuera de ventana operativa"
                )
                return

            entry = data.get("entry", [{}])[0]
            changes = entry.get("changes", [{}])[0]
            value = changes.get("value", {})

            if "messages" not in value:
                return

            msg = value["messages"][0]
            msg_id = msg.get("id")
            msg_type = msg.get("type")

            # ✅ FIX 2.7: idempotencia persistente
            # 1) Si ya está en el cache in-memory, descartar
            if msg_id and msg_id in self.processed_messages:
                logger.info(f"Mensaje duplicado (cache local): {msg_id}")
                return
            # 2) Verificar+insertar atómicamente en BD
            if msg_id and self.db_manager.msg_ya_procesado(msg_id):
                logger.info(f"Mensaje duplicado (BD): {msg_id}")
                self.processed_messages.add(msg_id)  # cache para acelerar próximos
                return
            # 3) Si pasó ambos, marcar en cache
            if msg_id:
                self.processed_messages.add(msg_id)
                if len(self.processed_messages) > 5000:
                    self.processed_messages = set(list(self.processed_messages)[-2500:])

            sender = msg.get("from")
            if not sender or not validate_phone_number(sender):
                logger.warning("Mensaje sin 'from' válido, descartado")
                return

            if msg_type in self.TIPOS_ADJUNTO:
                attach = msg.get(msg_type, {}) or {}
                media_id = attach.get("id")
                mime_type = attach.get("mime_type", "")
                self._handle_attachment(sender, msg_type, media_id, mime_type)
                return

            if msg_type not in self.TIPOS_TEXTO:
                logger.info(f"Tipo de mensaje no soportado: {msg_type}")
                return

            text = msg.get("text", {}).get("body", "")
            text = sanitize_user_input(text)
            if not text:
                logger.info(f"Mensaje vacío tras sanitización")
                return

            # ✅ v1.4.2: coalescencia inbound. En vez de llamar directo a
            # _handle_ai_flow (que generaba IA + encolaba envío con doble
            # delay), encolamos un inbound_intent. Si la ferretería envía
            # más mensajes durante la ventana de escucha, los acumulamos
            # en el buffer y reiniciamos el timer (debounce).
            self._encolar_mensaje_inbound(sender, text)
        except Exception as e:
            logger.error(f"Error en process_incoming: {e}")

    # ------------------------------------------------------------------
    # ✅ v1.4.2: coalescencia inbound
    # ------------------------------------------------------------------
    def _encolar_mensaje_inbound(self, recipient: str, text: str) -> None:
        """
        Acumula un mensaje inbound en el buffer del recipient y (re)agenda
        el inbound_intent en el dispatcher. Cada llamada reinicia el timer
        de escucha (debounce). Si el recipient no corresponde a ninguna
        ferretería conocida o está vetado, descarta silenciosamente.

        Persiste el mensaje del usuario en BD inmediatamente (con
        respuesta_ia=NULL) para que los mensajes siguientes del mismo burst
        lo vean al consultar el historial. Cuando finalmente la IA genere
        la respuesta, se hará UPDATE sobre la última fila pendiente.
        """
        # 1) Validar la ferretería ANTES de tocar buffer o dispatcher. Si el
        # número es desconocido o vetado, no queremos gastar slots de cola.
        try:
            ferreteria = self.db_manager.obtener_ferreteria_por_telefono(recipient)
        except Exception as e:
            logger.error(f"Error buscando ferretería en inbound: {e}")
            return
        if not ferreteria:
            logger.warning(
                f"Mensaje de número desconocido {mask_phone(recipient)}. "
                "Bot no crea ferreterías automáticamente. Ignorando."
            )
            return
        try:
            if self.db_manager.is_phone_vetoed(ferreteria.id_ferreteria, recipient):
                logger.info(f"Mensaje ignorado (número vetado)")
                return
        except Exception as e:
            logger.error(f"Error verificando veto: {e}")
            return

        # 2) Persistir el mensaje del usuario YA (con respuesta_ia=NULL) para
        # que aparezca en el historial cuando lleguen mensajes posteriores.
        # Guardamos el interaction_id para hacerle UPDATE más tarde — solo al
        # ÚLTIMO de la ráfaga, que es el que carga la respuesta del bot.
        try:
            interaction_id = self.db_manager.guardar_mensaje_usuario(
                id_ferreteria=str(ferreteria.id_ferreteria),
                mensaje_usuario=text,
            )
        except Exception as e:
            logger.error(f"Error persistiendo mensaje usuario: {e}")
            interaction_id = None  # buffer en memoria sigue salvando la respuesta

        # 3) Apilar en el buffer y (re)agendar el intent. Se hace bajo el
        # mismo lock que protege el buffer, para que no se mezcle con el
        # vaciado del buffer en _procesar_inbound_intent.
        with self._pending_lock:
            self._pending_messages.setdefault(recipient, []).append(
                {"text": text, "interaction_id": interaction_id or ""}
            )
            pendientes = len(self._pending_messages[recipient])

        if self.dispatcher is None:
            # Sin dispatcher no hay coalescencia posible: caemos al flujo viejo
            # síncronamente. Caso de bootstrap incompleto.
            logger.warning(
                "Inbound sin dispatcher configurado: ejecutando flujo síncrono"
            )
            with self._pending_lock:
                msgs = self._pending_messages.pop(recipient, [])
            self._handle_ai_flow(
                modo="inbound", recipient=recipient,
                message="\n".join(m["text"] for m in msgs) if msgs else text,
            )
            return

        # Cancelar cualquier intent pendiente y encolar uno nuevo (reinicia timer).
        cancelled = self.dispatcher.cancel_inbound_intent_for(recipient)
        self.dispatcher.enqueue_inbound_intent(recipient)
        logger.info(
            f"👂 Mensaje #{pendientes} en buffer de {mask_phone(recipient)} "
            f"(intent reagendado, prev_cancelados={cancelled})"
        )

    def _procesar_inbound_intent(self, recipient: str) -> None:
        """
        Callback que invoca el dispatcher cuando vence un inbound_intent.
        Lee y vacía el buffer del recipient, concatena los mensajes
        acumulados en uno solo (separados por \\n) y ejecuta el flujo
        clásico inbound (pasos 2–6). La respuesta final se encolará como
        kind='text' dentro de _ejecutar_pasos_2_a_6 vía dispatcher.enqueue(),
        que ya no aplica delay artificial — sale tan pronto el inter_chat
        y el gate horario lo permitan.

        IMPORTANTE: este método corre en el hilo del dispatcher, NO en el
        de Flask. Por eso es seguro hacer llamadas potencialmente lentas
        (Anthropic API, BD): no bloquea webhooks.
        """
        with self._pending_lock:
            mensajes = self._pending_messages.pop(recipient, [])

        if not mensajes:
            # Race poco probable: el intent venció pero el buffer ya está vacío
            # (¿alguien lo procesó? ¿se vació externamente?). Log y salir.
            logger.warning(
                f"inbound_intent venció para {mask_phone(recipient)} pero "
                f"el buffer está vacío — nada que procesar"
            )
            return

        # Concatenar mensajes del burst. El extractor de cotizaciones recibe
        # este texto como user_message (paso 6); inyectar todos los mensajes
        # juntos da más señal al extractor sin perder contexto. El historial
        # también lo verá: las filas con respuesta_ia=NULL aparecen como
        # turnos del usuario sueltos (obtener_historial_reciente ya filtra
        # los assistant vacíos).
        user_message_agregado = "\n".join(m["text"] for m in mensajes)
        # La fila que recibirá la respuesta del bot es la del ÚLTIMO mensaje
        # del burst (cronológicamente la última de la conversación). Los
        # mensajes previos del burst se quedan con respuesta_ia=NULL — refleja
        # la realidad: el bot esperó a que la ferretería terminara y luego
        # respondió una sola vez.
        interaction_id_pendiente = mensajes[-1].get("interaction_id") or None
        logger.info(
            f"🧠 Procesando inbound_intent de {mask_phone(recipient)}: "
            f"{len(mensajes)} mensaje(s) coalescido(s)"
        )
        # Reusamos el camino existente. _handle_ai_flow_inbound vuelve a
        # buscar la ferretería y revalidar veto — redundante pero barato y
        # blindado frente a cambios de estado durante la ventana de escucha
        # (p.ej. el operador veta el número en medio del burst).
        self._handle_ai_flow(
            modo="inbound", recipient=recipient,
            message=user_message_agregado,
            interaction_id_pendiente=interaction_id_pendiente,
        )

    # ------------------------------------------------------------------
    # ✅ v1.4.3: rehidratación al arranque
    # ------------------------------------------------------------------
    def rehidratar_inbounds_huerfanos(self,
                                       ventana_minutos_responder: int = 15,
                                       ventana_minutos_marcar: int = 120) -> Dict[str, int]:
        """
        Tras un reinicio del bot, busca en BD mensajes inbound que entraron
        pero nunca recibieron respuesta (respuesta_ia=NULL). Dos rangos:

        1) [< ventana_minutos_responder min] — los repuebla en el buffer
           y encola inbound_intent para que el dispatcher los procese
           normalmente. El cliente recibirá la respuesta diferida (con un
           pequeño atraso por el reinicio, pero la recibirá).

        2) [entre ventana_minutos_responder y ventana_minutos_marcar] —
           demasiado tarde para responder (el cliente probablemente se
           fue), pero los marca con un placeholder en respuesta_ia para
           que dejen de aparecer como NULL en queries y reportes.

        Las filas más antiguas que `ventana_minutos_marcar` se ignoran:
           son historia antigua que no nos toca limpiar.

        Devuelve un dict con las cuentas:
           {'rehidratados': N, 'marcados_perdidos': M}

        IDEMPOTENTE: se puede llamar varias veces sin duplicar trabajo.
        Las filas rehidratadas se completan al vencer su nuevo intent.
        """
        if self.dispatcher is None:
            logger.warning(
                "rehidratar_inbounds_huerfanos: sin dispatcher, no se puede "
                "reagendar. Marcando todos los pendientes como perdidos."
            )
            ventana_minutos_responder = 0

        pendientes = self.db_manager.obtener_inbounds_pendientes(
            ventana_minutos=ventana_minutos_marcar
        )
        if not pendientes:
            logger.info("✅ No hay inbounds huérfanos para rehidratar")
            return {"rehidratados": 0, "marcados_perdidos": 0}

        ahora = datetime.now(timezone.utc)
        umbral_responder = ahora - timedelta(minutes=ventana_minutos_responder)

        # Agrupar por número de teléfono — varios mensajes del mismo cliente
        # en el burst original deben rehidratarse JUNTOS, no separados.
        por_numero: Dict[str, List[Dict]] = {}
        marcar_perdidos: List[str] = []

        for p in pendientes:
            # Asegurar que fecha sea aware (Postgres devuelve aware con tz)
            fecha = p["fecha"]
            if fecha.tzinfo is None:
                fecha = fecha.replace(tzinfo=timezone.utc)
            if fecha >= umbral_responder:
                por_numero.setdefault(p["numero"], []).append(p)
            else:
                marcar_perdidos.append(p["id_interaccion"])

        # Rehidratar — repoblar buffer y encolar UN intent por número
        rehidratados = 0
        for numero, items in por_numero.items():
            with self._pending_lock:
                buffer_existente = self._pending_messages.setdefault(numero, [])
                ids_ya_en_buffer = {e.get("interaction_id") for e in buffer_existente}
                for item in items:
                    if item["id_interaccion"] in ids_ya_en_buffer:
                        continue  # idempotencia: ya estaba
                    buffer_existente.append({
                        "text": item["mensaje"],
                        "interaction_id": item["id_interaccion"],
                    })
                    rehidratados += 1
            # Reagendar intent (cancela previo si lo había)
            self.dispatcher.cancel_inbound_intent_for(numero)
            self.dispatcher.enqueue_inbound_intent(numero)
            logger.info(
                f"♻️  Rehidratados {len(items)} mensaje(s) huérfanos de "
                f"{mask_phone(numero)} → intent reagendado"
            )

        # Marcar los demasiado antiguos
        marcados = 0
        for id_int in marcar_perdidos:
            try:
                if self.db_manager.marcar_inbound_perdido(id_int):
                    marcados += 1
            except Exception as e:
                logger.error(f"Error marcando perdido {id_int}: {e}")

        if marcados:
            logger.info(
                f"🪦 {marcados} inbound(s) demasiado antiguos marcados como "
                f"'respuesta perdida' (>{ventana_minutos_responder}min de retraso)"
            )
        return {"rehidratados": rehidratados, "marcados_perdidos": marcados}

    # ------------------------------------------------------------------
    # Adjuntos (image/document):
    #   - Las IMÁGENES también se descargan y se extraen (no solo PDFs).
    #   - La transición a `cierre` solo ocurre si la extracción produjo
    #     al menos UNA línea válida persistida.
    #   - Si el estado actual era `inicio`, se hace transición encadenada
    #     `inicio → cotizacion → cierre` (porque el adjunto aporta cotización
    #     Y la cierra simultáneamente).
    # ------------------------------------------------------------------
    def _handle_attachment(self, recipient: str, tipo: str,
                           media_id: Optional[str] = None,
                           mime_type: str = ""):
        try:
            ferreteria = self.db_manager.obtener_ferreteria_por_telefono(recipient)
            if not ferreteria:
                logger.info(f"Adjunto recibido de número desconocido, ignorado")
                return
            if self.db_manager.is_phone_vetoed(ferreteria.id_ferreteria, recipient):
                logger.info(f"Adjunto ignorado (número vetado)")
                return

            estado_actual = ferreteria.estado
            es_pdf = (tipo == "document" and "pdf" in (mime_type or "").lower())
            es_imagen = (tipo == "image")

            # Capturar contexto sin riesgo de session detached
            ferreteria_id_str = str(ferreteria.id_ferreteria)
            ferreteria_id_uuid = ferreteria.id_ferreteria
            regional = ferreteria.regional
            cod_municipio = ferreteria.cod_municipio

            # Solo procesamos PDFs e imágenes con extractor disponible
            persistidas = 0
            if media_id and self.extraction_client and (es_pdf or es_imagen):
                persistidas = self._procesar_adjunto_cotizacion(
                    media_id=media_id,
                    mime_type=mime_type,
                    es_pdf=es_pdf,
                    es_imagen=es_imagen,
                    ferreteria_id_str=ferreteria_id_str,
                    regional=regional,
                    cod_municipio=cod_municipio,
                )
            else:
                logger.info(
                    f"📎 Adjunto ({tipo}, mime={mime_type}) no procesable "
                    f"(media_id={bool(media_id)}, extractor={bool(self.extraction_client)}, "
                    f"pdf={es_pdf}, img={es_imagen})"
                )

            # Transiciones SOLO si se persistió al menos una línea
            if persistidas == 0:
                logger.info(
                    f"📎 Adjunto sin líneas extraídas → no se aplica transición "
                    f"(estado sigue en {estado_actual.value if estado_actual else 'None'})"
                )
                return

            # Hubo extracción exitosa → aplicar transición(es) según estado actual
            if estado_actual == EstadoFereteria.inicio:
                ok1 = self.db_manager.transicionar_estado(
                    ferreteria_id_uuid, EstadoFereteria.cotizacion
                )
                if ok1:
                    self.db_manager.transicionar_estado(
                        ferreteria_id_uuid, EstadoFereteria.cierre
                    )
                    logger.info(f"📎 Adjunto válido → inicio → cotizacion → cierre")
                else:
                    logger.warning(f"📎 No se pudo transicionar inicio → cotizacion")
            elif estado_actual == EstadoFereteria.cotizacion:
                ok = self.db_manager.transicionar_estado(
                    ferreteria_id_uuid, EstadoFereteria.cierre
                )
                if ok:
                    logger.info(f"📎 Adjunto válido → cotizacion → cierre")
                else:
                    logger.warning(f"📎 No se pudo transicionar a cierre")
            elif estado_actual == EstadoFereteria.cierre:
                logger.info(f"📎 Adjunto válido en estado cierre (idempotente)")
            elif estado_actual == EstadoFereteria.primer_mensaje:
                ok1 = self.db_manager.transicionar_estado(
                    ferreteria_id_uuid, EstadoFereteria.inicio
                )
                if ok1:
                    ok2 = self.db_manager.transicionar_estado(
                        ferreteria_id_uuid, EstadoFereteria.cotizacion
                    )
                    if ok2:
                        self.db_manager.transicionar_estado(
                            ferreteria_id_uuid, EstadoFereteria.cierre
                        )
                        logger.info(
                            f"📎 Adjunto válido → primer_mensaje → inicio → cotizacion → cierre"
                        )
            else:
                estado_str = estado_actual.value if estado_actual else "None"
                logger.info(
                    f"📎 Adjunto válido pero estado actual '{estado_str}' "
                    f"no permite transición a cierre"
                )
        except Exception as e:
            logger.error(f"Error manejando adjunto: {e}")

    def _procesar_adjunto_cotizacion(self, media_id: str, mime_type: str,
                                     es_pdf: bool, es_imagen: bool,
                                     ferreteria_id_str: str,
                                     regional: str,
                                     cod_municipio: Optional[str]) -> int:
        """
        Pipeline UNIFICADO para adjuntos (PDF e imagen):
          1) WhatsAppClient.download_media → bytes + mime
          2) Extracción con el método correspondiente al tipo
          3) Resolver cemento en catálogo UNA sola vez (NO crearlo)
          4) Por cada línea válida (con marca + precio):
             a) crear interacción marcador propia (id_interaccion único)
             b) registrar cotización en BD (FKs reales)
             c) anexar al CSV incremental
          5) Devolver el número de líneas persistidas

        Devuelve: int = líneas persistidas (0 si no hubo extracción exitosa).
        """
        try:
            # 1) descarga
            descarga = self.wa_client.download_media(media_id)
            if not descarga:
                logger.warning("Adjunto: no se pudo descargar el media")
                return 0
            file_bytes, mime_real = descarga

            # 2) extracción según tipo
            tipo_log = "PDF" if es_pdf else "imagen"
            if es_pdf:
                if "pdf" not in (mime_real or "").lower():
                    logger.info(f"Documento descargado no es PDF (mime={mime_real}), omitido")
                    return 0
                lineas = self.extraction_client.extract_quote_from_pdf(file_bytes)
            elif es_imagen:
                lineas = self.extraction_client.extract_quote_from_image(
                    file_bytes, mime_type=mime_real or mime_type
                )
            else:
                logger.warning(f"Adjunto: tipo no soportado")
                return 0

            if lineas is None:
                logger.warning(f"{tipo_log}: extractor falló (None); no se persiste nada")
                return 0
            if len(lineas) == 0:
                logger.info(f"{tipo_log}: no contiene líneas de cemento; nada que persistir")
                return 0

            # 3) resolver cemento en catálogo UNA vez
            cemento = self.db_manager.obtener_producto_cemento()
            if cemento is None:
                logger.warning(
                    f"{tipo_log}: no hay producto 'cemento' en catálogo; "
                    f"se descartan las {len(lineas)} líneas detectadas"
                )
                return 0
            id_producto, nombre_real = cemento

            # 4) iterar y persistir
            csv_path = getattr(self, "csv_cotizaciones_pdf", None)
            persistidas = 0
            descartadas = 0

            for i, linea in enumerate(lineas, start=1):
                tag = f"[{i}/{len(lineas)}]"
                marca_nombre = (linea.get("marca") or "").strip()
                precio_raw = linea.get("precio_unitario")

                if not marca_nombre or precio_raw is None:
                    logger.info(
                        f"{tipo_log} {tag}: incompleta (marca={marca_nombre!r}, "
                        f"precio={precio_raw}); línea descartada"
                    )
                    descartadas += 1
                    continue
                try:
                    precio_f = float(precio_raw)
                    if precio_f <= 0:
                        raise ValueError("precio <= 0")
                except (TypeError, ValueError) as e:
                    logger.info(f"{tipo_log} {tag}: precio inválido ({precio_raw!r}: {e}); descartada")
                    descartadas += 1
                    continue

                # 4a) interacción marcador propia para esta línea
                interaction_id = self.db_manager.guardar_interaccion(
                    id_ferreteria=ferreteria_id_str,
                    mensaje_usuario=(
                        f"[{tipo_log} cotización media_id={media_id} línea {i}/{len(lineas)}]"
                    ),
                    respuesta_ia=(
                        f"[{tipo_log} procesado: {nombre_real} {marca_nombre} ${precio_f:.2f}]"
                    ),
                )
                if not interaction_id:
                    logger.error(f"{tipo_log} {tag}: no se pudo crear interacción marcador")
                    descartadas += 1
                    continue

                # 4b) registrar cotización en BD
                fila = self.db_manager.registrar_cotizacion(
                    id_interaccion=interaction_id,
                    id_ferreteria=ferreteria_id_str,
                    producto_nombre=nombre_real,
                    marca_nombre=marca_nombre,
                    precio=precio_f,
                    regional=regional,
                    disponibilidad=linea.get("disponibilidad"),
                    confianza=linea.get("confianza"),
                    info_solicitada=linea.get("observaciones"),
                    cod_municipio=cod_municipio,
                    id_producto=id_producto,
                )
                if not fila:
                    logger.warning(f"{tipo_log} {tag}: inserción en `cotizaciones` falló")
                    descartadas += 1
                    continue

                # 4c) anexar al CSV
                if csv_path:
                    self.db_manager.append_cotizacion_a_csv(fila, csv_path)
                persistidas += 1

            # 5) resumen
            if not csv_path:
                logger.warning(f"{tipo_log}: csv_cotizaciones_pdf no configurado; solo BD")
            logger.info(
                f"📄 {tipo_log} resumen: {len(lineas)} líneas detectadas, "
                f"{persistidas} persistidas, {descartadas} descartadas"
            )
            return persistidas
        except Exception as e:
            logger.error(f"Error procesando adjunto: {e}")
            return 0

    # ==================================================================
    # ✅ v1.1: _handle_ai_flow POLIMÓRFICO
    # Dispatcher por modo. Cada modo prepara su contexto y luego ambos
    # convergen en _ejecutar_pasos_2_a_6() que contiene los pasos compartidos.
    # ==================================================================
    def _handle_ai_flow(self, modo: str, *,
                        recipient: Optional[str] = None,
                        message: Optional[str] = None,
                        topic: Optional[str] = None,
                        interaction_id_pendiente: Optional[str] = None):
        """
        Dispatcher polimórfico (v1.1).

        modo="inbound":
            Requiere `recipient` y `message`. Comportamiento clásico: busca la
            ferretería por teléfono, valida veto y ejecuta pasos 2–6 con el
            mensaje del cliente.
            ✅ v1.4.2: si viene `interaction_id_pendiente`, NO se crea una
            nueva fila en historial_interacciones — se hace UPDATE sobre la
            fila existente (que se creó al recibir el mensaje del usuario,
            con respuesta_ia=NULL). Esto evita duplicar el mensaje del
            usuario en historial cuando hubo coalescencia.

        modo="outreach":
            Requiere `topic`. Itera ferreterías con estado=NULL no vetadas,
            construye user_message desde topic, ejecuta pasos 2–6 por cada una
            y transiciona None → primer_mensaje al final.
        """
        if modo == "inbound":
            if recipient is None or message is None:
                logger.error("_handle_ai_flow inbound: recipient y message son obligatorios")
                return
            self._handle_ai_flow_inbound(
                recipient, message,
                interaction_id_pendiente=interaction_id_pendiente,
            )
        elif modo == "outreach":
            if topic is None:
                logger.error("_handle_ai_flow outreach: topic es obligatorio")
                return
            self._handle_ai_flow_outreach(topic)
        else:
            logger.error(f"_handle_ai_flow: modo desconocido {modo!r}")

    # ------------------------------------------------------------------
    # MODO INBOUND (reactivo): mensaje entrante desde webhook
    # ------------------------------------------------------------------
    def _handle_ai_flow_inbound(self, recipient: str, message: str,
                                 interaction_id_pendiente: Optional[str] = None):
        try:
            ferreteria = self.db_manager.obtener_ferreteria_por_telefono(recipient)

            if not ferreteria:
                logger.warning(
                    f"Mensaje de número desconocido {mask_phone(recipient)}. "
                    "El bot no está configurado para crear nuevas ferreterías automáticamente. Ignorando mensaje."
                )
                return

            if self.db_manager.is_phone_vetoed(ferreteria.id_ferreteria, recipient):
                logger.info(f"Mensaje ignorado (número vetado)")
                return

            self._ejecutar_pasos_2_a_6(
                ferreteria=ferreteria,
                recipient=recipient,
                user_message=message,
                modo="inbound",
                interaction_id_pendiente=interaction_id_pendiente,
            )
        except Exception as e:
            logger.error(f"Error en flujo inbound: {e}")

    # ------------------------------------------------------------------
    # MODO OUTREACH (proactivo): saluda a ferreterías con estado=NULL
    # ------------------------------------------------------------------
    def _handle_ai_flow_outreach(self, topic: str):
        """
        Itera ferreterías candidatas (estado IS NULL y no vetadas), genera el
        primer mensaje desde `topic`, ejecuta pasos 2–6 y transiciona None →
        primer_mensaje al final de cada ferretería.

        El gate horario se chequea ANTES de invocar este método (en el cron
        del BroadcastScheduler, decisión 2a+2c). Aquí asumimos que el bot
        está dentro de ventana.
        """
        logger.info(f"🚀 Outreach iniciado — topic: {topic!r}")
        # Snapshot de candidatas en memoria para evitar mantener una sesión
        # de BD abierta durante todo el bucle.
        candidatas_data: List[Tuple[Any, str]] = []
        try:
            with self.db_manager.get_session() as session:
                candidatas = session.query(Ferreteria).filter(
                    Ferreteria.estado.is_(None)
                ).all()
                # Materializar atributos necesarios antes de cerrar la sesión
                for ferr in candidatas:
                    if ferr.num_telefono in (ferr.num_vetados or []):
                        continue
                    # Capturamos un dict-like con los campos que necesitamos
                    candidatas_data.append({
                        "id_ferreteria": ferr.id_ferreteria,
                        "num_telefono": ferr.num_telefono,
                        "regional": ferr.regional,
                        "cod_municipio": ferr.cod_municipio,
                    })
            logger.info(
                f"📋 Outreach: {len(candidatas_data)} ferreterías candidatas "
                f"(estado=NULL y no vetadas)"
            )
        except Exception as e:
            logger.error(f"Outreach: error consultando candidatas: {e}")
            return

        # user_message a partir del topic (mismo patrón que tenía
        # BroadcastScheduler antes del refactor)
        user_message = f"Genera un saludo informativo sobre: {topic}"

        enviadas = 0
        falladas = 0
        for cdata in candidatas_data:
            try:
                # Re-cargar la ferretería en una sesión nueva para evitar
                # session detached al transicionar al final.
                ferreteria = self.db_manager.obtener_ferreteria_por_telefono(
                    cdata["num_telefono"]
                )
                if not ferreteria:
                    logger.warning(
                        f"Outreach: ferretería desapareció entre snapshot "
                        f"y carga ({mask_phone(cdata['num_telefono'])}); skip"
                    )
                    falladas += 1
                    continue

                self._ejecutar_pasos_2_a_6(
                    ferreteria=ferreteria,
                    recipient=ferreteria.num_telefono,
                    user_message=user_message,
                    modo="outreach",
                )

                # Tras encolar (ya hizo el dispatcher.enqueue dentro del paso 5),
                # transicionamos None → primer_mensaje. Esto es lo que antes
                # hacía BroadcastScheduler._run_broadcast_job.
                self.db_manager.transicionar_estado(
                    ferreteria.id_ferreteria, EstadoFereteria.primer_mensaje
                )
                enviadas += 1
            except Exception as e:
                logger.error(
                    f"Outreach: error procesando "
                    f"{mask_phone(cdata['num_telefono'])}: {e}"
                )
                falladas += 1
                continue

        logger.info(
            f"🏁 Outreach finalizado: {enviadas} enviadas, {falladas} fallidas"
        )

    # ------------------------------------------------------------------
    # PASOS 2–6 COMPARTIDOS
    # Idénticos para ambos modos. La única diferencia es que en outreach el
    # `estado` que se inyecta al prompt es siempre "primer_mensaje" (porque
    # la ferretería viene de NULL), mientras que en inbound usamos el estado
    # actual.
    # ------------------------------------------------------------------
    def _ejecutar_pasos_2_a_6(self, ferreteria, recipient: str,
                              user_message: str, modo: str,
                              interaction_id_pendiente: Optional[str] = None):
        """
        Pasos 2–6 del flujo (comunes a outreach e inbound):
          2) Recuperar historial reciente
          3) get_response (BASE+REGION+ESTADO+HISTORIAL)
          4) Persistir interacción:
              - outreach: INSERT vía guardar_interaccion
              - inbound con interaction_id_pendiente: UPDATE vía
                actualizar_respuesta_ia (✅ v1.4.2: la fila ya existe,
                creada al recibir el último mensaje del burst)
              - inbound sin interaction_id_pendiente: INSERT vía
                guardar_interaccion (path legacy de seguridad)
          5) dispatcher.enqueue (sin delay; respeta solo inter_chat + gate)
          6) Si estado NO es {cierre, terminado}: extract_quote_info + transiciones
        """
        try:
            estado_actual = ferreteria.estado
            # En outreach, el estado para el prompt SIEMPRE es "primer_mensaje"
            # (la ferretería todavía no ha sido contactada, viene de NULL).
            # En inbound, usamos el estado real.
            if modo == "outreach":
                estado_nombre_prompt = "primer_mensaje"
            else:
                estado_nombre_prompt = (
                    estado_actual.value if hasattr(estado_actual, 'value')
                    else str(estado_actual)
                )
            region_nombre = ferreteria.regional

            # 2) Recuperar historial conversacional
            limite = getattr(self.ai_client.config, 'history_limit', 10)
            historial = self.db_manager.obtener_historial_reciente(
                id_ferreteria=str(ferreteria.id_ferreteria),
                limite=limite
            )

            # ✅ v1.3: bifurcación inbound vs outreach.
            # En outreach (primer contacto) Meta NO permite texto libre si el
            # cliente no ha escrito en las últimas 24h, así que usamos la
            # plantilla aprobada "saludo" con un único parámetro {{1}}
            # generado por Claude a partir del catálogo de productos de cemento.
            if modo == "outreach":
                # 3-out) Obtener catálogo de productos cemento desde BD.
                productos_catalogo: List[str] = []
                try:
                    with self.db_manager.get_session() as session:
                        for p in session.query(Producto).all():
                            nombre_norm = self.db_manager._normalizar_nombre(p.nombre)
                            if "cemento" in nombre_norm:
                                productos_catalogo.append(p.nombre)
                except Exception as e:
                    logger.warning(
                        f"Outreach: no se pudo leer catálogo de productos: {e}; "
                        f"se usará fallback en generate_outreach_param"
                    )

                # 3-out) Generar el parámetro {{1}} con la IA.
                outreach_param = self.ai_client.generate_outreach_param(
                    productos_disponibles=productos_catalogo,
                    region=region_nombre,
                )

                # Renderizar el texto que verá el cliente para guardarlo en
                # historial (la plantilla literal aprobada en Meta).
                template_rendered = (
                    "Hola, buen día. Quisiera consultar si tienen disponibilidad "
                    f"de: {outreach_param}. Además, ¿me podrían confirmar el "
                    "precio? Quedo atento, muchas gracias."
                )

                # 4-out) Persistir interacción. Marcamos con [OUTREACH-TEMPLATE]
                # para distinguir de outreach antiguo de texto libre en logs.
                interaction_id = self.db_manager.guardar_interaccion(
                    id_ferreteria=str(ferreteria.id_ferreteria),
                    mensaje_usuario=f"[OUTREACH-TEMPLATE] {user_message}",
                    respuesta_ia=template_rendered,
                )

                # 5-out) Encolar PLANTILLA (no texto libre) respetando delays + gate.
                # ✅ CORRECCIÓN: garantizar que dispatcher siempre existe en outreach
                template_name = self.ai_client.config.outreach_template_name
                template_lang = self.ai_client.config.outreach_template_lang
                if self.dispatcher is None:
                    raise RuntimeError(
                        "❌ MessageProcessor requiere dispatcher configurado para outreach. "
                        "Verifica bootstrap en CELDA 11: "
                        "dispatcher = MessageDispatcher(wa_client, dispatcher_config, hours_gate=hours_gate)"
                    )

                self.dispatcher.enqueue_template(
                    recipient=recipient,
                    template_name=template_name,
                    language_code=template_lang,
                    body_params=[outreach_param],
                )
            else:
                # ── Modo inbound (sin cambios respecto a v1.1) ──
                # 3) Generar respuesta del bot (BASE+REGION+ESTADO+HISTORIAL)
                ai_response = self.ai_client.get_response(
                    user_message=user_message,
                    region=region_nombre,
                    estado=estado_nombre_prompt,
                    historial=historial,
                )

                # 4) Persistir interacción:
                # - Si vino interaction_id_pendiente (caso coalescencia v1.4.2),
                #   UPDATE sobre la fila existente, o INSERT con MISMO UUID si
                #   la fila no existe (v1.4.3, vía fallbacks).
                # - Si no vino, fallback al INSERT clásico (path defensivo:
                #   sucede si el caller invocó el flujo sin pasar por
                #   process_incoming, p. ej. tests manuales o llamadas directas).
                if interaction_id_pendiente:
                    ok = self.db_manager.actualizar_respuesta_ia(
                        id_interaccion=interaction_id_pendiente,
                        respuesta_ia=ai_response,
                        # ✅ v1.4.3: fallbacks para UPSERT semántico. Si la fila
                        # no existe (reinicio del kernel previo), se reinsertará
                        # con el MISMO UUID en lugar de crear duplicado.
                        mensaje_usuario_fallback=user_message,
                        id_ferreteria_fallback=str(ferreteria.id_ferreteria),
                    )
                    if ok:
                        interaction_id = interaction_id_pendiente
                    else:
                        # Solo entra aquí si actualizar_respuesta_ia retorna False,
                        # lo que con los fallbacks v1.4.3 NO debería pasar nunca.
                        # Mantenemos el path por completitud defensiva.
                        logger.error(
                            f"actualizar_respuesta_ia retornó False con fallbacks "
                            f"presentes para {interaction_id_pendiente} — fallback "
                            f"a guardar_interaccion clásico"
                        )
                        interaction_id = self.db_manager.guardar_interaccion(
                            id_ferreteria=str(ferreteria.id_ferreteria),
                            mensaje_usuario=user_message,
                            respuesta_ia=ai_response,
                        )
                else:
                    interaction_id = self.db_manager.guardar_interaccion(
                        id_ferreteria=str(ferreteria.id_ferreteria),
                        mensaje_usuario=user_message,
                        respuesta_ia=ai_response,
                    )

                # 5) Encolar respuesta. ✅ v1.4.2: dispatcher.enqueue ya NO
                # aplica delay artificial — la ventana de escucha ya absorbió
                # el 'tiempo humano' antes de generar.
                if self.dispatcher is not None:
                    self.dispatcher.enqueue(recipient, ai_response)
                else:
                    self.wa_client.send_text_message(recipient, ai_response)

            # 6) Aplicar transiciones de estado
            #    En outreach NO ejecutamos el extractor: el user_message es
            #    sintético (no viene del cliente), no aporta señales reales.
            #    La transición None → primer_mensaje la hace el caller
            #    (_handle_ai_flow_outreach) tras esta función.
            if modo == "outreach":
                logger.info(
                    f"⏭️  Outreach: extractor omitido (user_message sintético)"
                )
                return

            # ── A partir de aquí: solo modo inbound ──
            if estado_actual in self.ESTADOS_SIN_EXTRACTOR:
                logger.info(
                    f"⏭️  Extractor saltado "
                    f"(estado={estado_nombre_prompt}, no aporta señales)"
                )
                return

            if self.extraction_client and interaction_id:
                self._aplicar_transiciones(
                    ferreteria=ferreteria,
                    estado_actual=estado_actual,
                    user_message=user_message,
                    ai_response=ai_response,
                    interaction_id=interaction_id,
                )
        except Exception as e:
            logger.error(f"Error en pasos 2–6 ({modo}): {e}")

    # ------------------------------------------------------------------
    # Lógica de transiciones (centralizada y testeable). Solo se invoca en
    # modo inbound; en outreach el extractor no aporta señales útiles.
    # ------------------------------------------------------------------
    def _aplicar_transiciones(self, ferreteria, estado_actual,
                              user_message: str,
                              ai_response: str, interaction_id: str):
        """
        Aplica las transiciones según las reglas de negocio.

        ✅ v1.4: ahora usa DOS extractores complementarios:
          - ExtractorTextoAcumulativo (determinista + fallback LLM acotado)
            → para detectar la transición `inicio → cotizacion`
            → reconstruye el estado del HISTORIAL completo, no solo último turno
          - AnthropicExtractionClient (LLM de un solo turno)
            → para detectar `es_despedida` y `confirma_cierre`
              (juicio semántico genuino)

        Si el extractor acumulativo no está configurado, cae al comportamiento
        anterior (LLM puro de un solo turno), preservando compatibilidad.
        """
        try:
            # Recuperar historial UNA sola vez para que ambos extractores lo usen.
            historial: List[Dict[str, str]] = []
            try:
                limite = getattr(self.ai_client.config, 'history_limit', 10)
                historial = self.db_manager.obtener_historial_reciente(
                    id_ferreteria=str(ferreteria.id_ferreteria),
                    limite=limite,
                )
            except Exception as e:
                logger.warning(f"No se pudo cargar historial para extractor: {e}")

            # ── EXTRACCIÓN A: cotización (acumulativo si disponible) ──────
            estado_acumulado: Optional[EstadoExtraccionAcumulado] = None
            if self.extractor_acumulativo is not None:
                try:
                    estado_acumulado = self.extractor_acumulativo.extraer_de_historial(
                        historial=historial,
                        mensaje_actual=user_message,
                        usar_llm_fallback=True,
                    )
                except Exception as e:
                    logger.error(f"Extractor acumulativo falló: {e}")

            # ── EXTRACCIÓN B: señales semánticas (despedida / cierre) ─────
            extraccion_llm = None
            if self.extraction_client is not None:
                extraccion_llm = self.extraction_client.extract_quote_info(
                    mensaje_ferreteria=user_message,
                    respuesta_bot=ai_response,
                    interaction_id=interaction_id,
                    historial=historial,  # ← v1.4: ahora se pasa historial
                )

            # 1) Despedida → terminado (forzado, desde cualquier estado)
            if extraccion_llm and extraccion_llm.get("es_despedida"):
                self.db_manager.transicionar_estado(
                    ferreteria.id_ferreteria,
                    EstadoFereteria.terminado,
                    forzar=True,
                )
                return

            # 2) primer_mensaje → inicio (la ferretería respondió, sea por broadcast
            #    o por mensaje espontáneo recién creada).
            if estado_actual == EstadoFereteria.primer_mensaje:
                ok = self.db_manager.transicionar_estado(
                    ferreteria.id_ferreteria, EstadoFereteria.inicio
                )
                if ok:
                    estado_actual = EstadoFereteria.inicio

            # 2b) sin_respuesta → inicio (la ferretería volvió a escribir tras
            #    el timeout 7d).
            if estado_actual == EstadoFereteria.sin_respuesta:
                ok = self.db_manager.transicionar_estado(
                    ferreteria.id_ferreteria, EstadoFereteria.inicio
                )
                if ok:
                    estado_actual = EstadoFereteria.inicio

            # 3) inicio → cotizacion. PRIORIZAR extractor acumulativo.
            if estado_actual == EstadoFereteria.inicio:
                cotizacion_lista = False
                producto_nombre = marca_nombre = None
                precio = None
                disponibilidad = None
                confianza = None
                info_solicitada = None

                if estado_acumulado and estado_acumulado.es_completo():
                    # Solo aceptar precios plausibles (no sospechosos por rango)
                    if not estado_acumulado.precio_sospechoso:
                        cotizacion_lista = True
                        producto_nombre = estado_acumulado.producto_nombre
                        marca_nombre = estado_acumulado.marca_nombre
                        precio = float(estado_acumulado.precio_unitario)
                        disponibilidad = estado_acumulado.disponibilidad
                        confianza = max(
                            estado_acumulado.producto_score,
                            estado_acumulado.marca_score,
                        )
                        info_solicitada = (
                            f"Acumulativo: {estado_acumulado.resumen()} | "
                            f"fuentes={estado_acumulado.fuentes}"
                        )
                        logger.info(
                            f"✅ Transición inicio→cotizacion vía acumulativo: "
                            f"{estado_acumulado.resumen()}"
                        )
                    else:
                        logger.warning(
                            f"⚠️  Cotización completa pero precio SOSPECHOSO: "
                            f"{estado_acumulado.resumen()} — no se transiciona"
                        )

                # Fallback al extractor LLM si el acumulativo no está disponible
                if (not cotizacion_lista
                        and self.extractor_acumulativo is None
                        and extraccion_llm
                        and AnthropicExtractionClient.tiene_cotizacion_completa(extraccion_llm)):
                    cotizacion_lista = True
                    producto_nombre = extraccion_llm.get("producto")
                    marca_nombre = extraccion_llm.get("marca")
                    precio = float(extraccion_llm.get("precio_unitario"))
                    disponibilidad = extraccion_llm.get("disponibilidad")
                    confianza = extraccion_llm.get("confianza")
                    info_solicitada = extraccion_llm.get("observaciones")
                    logger.info("Transición inicio→cotizacion vía LLM (fallback)")

                if cotizacion_lista:
                    cot_id = self.db_manager.registrar_cotizacion(
                        id_interaccion=interaction_id,
                        id_ferreteria=str(ferreteria.id_ferreteria),
                        producto_nombre=producto_nombre,
                        marca_nombre=marca_nombre,
                        precio=precio,
                        regional=ferreteria.regional,
                        disponibilidad=disponibilidad,
                        confianza=confianza,
                        info_solicitada=info_solicitada,
                        cod_municipio=ferreteria.cod_municipio,
                    )
                    if cot_id:
                        self.db_manager.transicionar_estado(
                            ferreteria.id_ferreteria, EstadoFereteria.cotizacion
                        )
                        estado_actual = EstadoFereteria.cotizacion

            # 4) cotizacion → cierre (cliente confirma envío de cotización
            #    formal o pedido por TEXTO)
            if (estado_actual == EstadoFereteria.cotizacion
                    and extraccion_llm
                    and AnthropicExtractionClient.tiene_confirmacion_cierre(extraccion_llm)):
                self.db_manager.transicionar_estado(
                    ferreteria.id_ferreteria, EstadoFereteria.cierre
                )
        except Exception as e:
            logger.error(f"Error aplicando transiciones: {e}")


class WebhookServer:
    """Servidor Flask + ngrok"""

    def __init__(self, wa_config: WhatsAppConfig, processor: MessageProcessor,
                 ngrok_token: str):
        self.wa_config = wa_config
        self.processor = processor
        self.ngrok_token = ngrok_token
        self.app = Flask(__name__)
        self._setup_routes()

    def _setup_routes(self) -> None:
        @self.app.route("/webhook", methods=["GET"])
        def verify():
            mode = request.args.get("hub.mode")
            token = request.args.get("hub.verify_token")
            challenge = request.args.get("hub.challenge")
            if mode == "subscribe" and token == self.wa_config.verify_token:
                logger.info("✅ Webhook verificado")
                return challenge, 200
            logger.warning("Intento de verificación con token inválido")
            return "Token inválido", 403

        @self.app.route("/webhook", methods=["POST"])
        def receive():
            try:
                data = request.get_json(silent=True) or {}

                # ─────────────────────────────────────────────────────────────
                # 🔒 FILTRO DE SEGURIDAD (DESACTIVADO)
                # Verifica que el phone_number_id del payload coincida con el
                # configurado. Útil para rechazar webhooks falsos en URLs
                # públicas (ngrok). Reactivar descomentando.
                # ─────────────────────────────────────────────────────────────

                # ✅ v1.1: el gate horario se aplica DENTRO de process_incoming.
                # Aquí siempre devolvemos 200 para que Meta no reintente.
                self.processor.process_incoming(data)
                return jsonify({"status": "ok"}), 200
            except Exception as e:
                logger.error(f"Error webhook: {e}")
                return jsonify({"error": "internal"}), 500

        @self.app.route("/health", methods=["GET"])
        def health():
            return jsonify({"status": "healthy"}), 200

    def start(self, port: int = 5000):
        try:
            ngrok.kill()
        except Exception:
            pass
        ngrok.set_auth_token(self.ngrok_token)
        public_url = ngrok.connect(port).public_url
        print("\n" + "=" * 60)
        print("🌐 CONFIGURACIÓN DEL WEBHOOK")
        print("=" * 60)
        print(f"URL: {public_url}/webhook")
        print(f"Token de verificación: {self.wa_config.verify_token}")
        print(f"Modelo IA: Claude (Anthropic)")
        print("=" * 60 + "\n")
        logger.info(f"🚀 Servidor en puerto {port}")
        self.app.run(port=port, debug=False, use_reloader=False)


## CELDA 10: BROADCAST SCHEDULER CON JOBS DE MANTENIMIENTO

In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger

class BroadcastScheduler:
    """
    Scheduler de jobs programados:
      - Job de outreach (broadcast semanal): delega en
        processor._handle_ai_flow(modo="outreach", topic=...).
        ✅ v1.1: chequea OperatingHoursGate al inicio del job (decisión 2a+2c).
        El cron sigue agendado siempre; el gate decide si la corrida procede.
      - Jobs de mantenimiento diario:
          * archive_old_interactions (02:00)
          * check_unresponsive_ferreterias (03:00)
          * purge_webhook_events (04:00)
    """

    def __init__(self, db_manager: DatabaseManager, wa_client: WhatsAppClient,
                 ai_client: AnthropicAIClient,
                 processor: "MessageProcessor",
                 dispatcher: Optional["MessageDispatcher"] = None,
                 hours_gate: Optional["OperatingHoursGate"] = None):
        self.db_manager = db_manager
        self.wa_client = wa_client
        self.ai_client = ai_client
        self.processor = processor          # ✅ v1.1: necesario para outreach
        self.dispatcher = dispatcher
        self.hours_gate = hours_gate        # ✅ v1.1: gate opcional
        # ⚠️ TZ Bogotá: los CronTriggers (broadcast semanal, jobs de mantenimiento)
        # se interpretan en hora LOCAL DE BOGOTÁ. Sin esto, en Colab (UTC) el
        # cron 'mon 17:08' se dispararía a las 12:08 hora Bogotá.
        self.scheduler = BackgroundScheduler(timezone=BOGOTA_TZ)
        logger.info("✅ BroadcastScheduler inicializado")

    @retry_on_failure(max_retries=3)
    def _run_broadcast_job(self, campaign_topic: str):
        """
        Wrapper delgado que delega en processor._handle_ai_flow(modo="outreach").
        ✅ v1.1: gate horario al inicio (decisión 2a+2c).
        """
        # ✅ Gate horario (decisión 2a+2c)
        if self.hours_gate is not None and not self.hours_gate.is_open():
            logger.info(
                f"🌙 Broadcast '{campaign_topic}' abortado: "
                f"bot fuera de ventana operativa"
            )
            return

        logger.info(f"🚀 Broadcast aprobado por gate — topic: {campaign_topic!r}")
        try:
            self.processor._handle_ai_flow(
                modo="outreach", topic=campaign_topic
            )
        except Exception as e:
            logger.error(f"Error en broadcast '{campaign_topic}': {e}")

    def schedule_weekly_broadcast(self, topic: str, day_of_week: str = 'mon',
                                  hour: int = 8, minute: int = 0):
        trigger = CronTrigger(
            day_of_week=day_of_week, hour=hour, minute=minute,
            timezone=BOGOTA_TZ  # ⚠️ explícito: sin esto APScheduler usa UTC en algunos casos
        )
        self.scheduler.add_job(
            self._run_broadcast_job,
            trigger=trigger,
            args=[topic],
            id=f"broadcast_{day_of_week}",
            replace_existing=True
        )
        logger.info(f"📅 Broadcast programado: {day_of_week.upper()} {hour:02d}:{minute:02d}")

    def check_unresponsive_ferreterias(self):
        """
        Marca como `sin_respuesta` ferreterías sin actividad >7 días.
        Usa transicionar_estado para validar contra TRANSICIONES_VALIDAS.
        """
        try:
            seven_days_ago = datetime.now(timezone.utc) - timedelta(days=7)
            ids_candidatos: List[uuid.UUID] = []
            with self.db_manager.get_session() as session:
                candidatas = session.query(Ferreteria).filter(
                    Ferreteria.estado.in_(list(ESTADOS_EN_CONVERSACION))
                ).all()
                for ferreteria in candidatas:
                    last = session.query(HistorialInteraccion).filter(
                        HistorialInteraccion.id_ferreteria == ferreteria.id_ferreteria
                    ).order_by(HistorialInteraccion.fecha_registro.desc()).first()
                    ref_fecha = (
                        last.fecha_registro if last and last.fecha_registro
                        else ferreteria.fecha_registro
                    )
                    if ref_fecha and ref_fecha < seven_days_ago:
                        ids_candidatos.append(ferreteria.id_ferreteria)

            transicionadas = 0
            for fid in ids_candidatos:
                if self.db_manager.transicionar_estado(
                    fid, EstadoFereteria.sin_respuesta
                ):
                    transicionadas += 1
            logger.info(
                f"🕒 Job check_unresponsive: {len(ids_candidatos)} candidatas, "
                f"{transicionadas} transicionadas a sin_respuesta"
            )
        except Exception as e:
            logger.error(f"Error check_unresponsive_ferreterias: {e}")

    def archive_old_interactions(self):
        try:
            n = self.db_manager.archive_old_interactions(days=7)
            logger.info(f"Archivado: {n} interacciones transferidas")
        except Exception as e:
            logger.error(f"Error archivando: {e}")

    def purge_webhook_events(self):
        """Job diario para purgar eventos de idempotencia >24h (FIX 2.7)."""
        try:
            n = self.db_manager.purgar_webhook_events_antiguos(days=1)
            logger.info(f"Purga webhook_events: {n} eliminados")
        except Exception as e:
            logger.error(f"Error purgando webhook_events: {e}")

    def start_scheduler(self):
        try:
            self.scheduler.add_job(
                self.check_unresponsive_ferreterias,
                CronTrigger(hour=3, minute=0, timezone=BOGOTA_TZ),
                id='check_unresponsive_ferreterias',
                replace_existing=True,
                name='Verificar ferreterías sin respuesta'
            )
            self.scheduler.add_job(
                self.archive_old_interactions,
                CronTrigger(hour=2, minute=0, timezone=BOGOTA_TZ),
                id='archive_old_interactions',
                replace_existing=True,
                name='Archivar interacciones antiguas'
            )
            self.scheduler.add_job(
                self.purge_webhook_events,
                CronTrigger(hour=4, minute=0, timezone=BOGOTA_TZ),
                id='purge_webhook_events',
                replace_existing=True,
                name='Purgar webhook_events antiguos'
            )
            if not self.scheduler.running:
                self.scheduler.start()
            logger.info("✅ Scheduler iniciado con jobs de mantenimiento")
        except Exception as e:
            logger.error(f"Error iniciando scheduler: {e}")

    def stop(self):
        self.scheduler.shutdown()
        logger.info("🛑 Scheduler detenido")

print("✅ BroadcastScheduler definido (v1.1: delega outreach al processor + gate horario)")


## CELDA 11: INICIALIZAR COMPONENTES

In [ ]:
try:
    wa_config = WhatsAppConfig(
        token=SECRETS['WA_TOKEN'],
        phone_id=SECRETS['WA_PHONE_ID'],
        verify_token=SECRETS['WEBHOOK_VERIFY_TOKEN']
    )

    # Modelo: cambiar segun necesidad
    #   - claude-opus-4-7              (más capaz)
    #   - claude-opus-4-6
    #   - claude-sonnet-4-6            (recomendado, default)
    #   - claude-haiku-4-5-20251001    (más rápido y económico)
    anthropic_config = AnthropicConfig(
        api_key=SECRETS['ANTHROPIC_API_KEY'],
        model_name="claude-haiku-4-5-20251001",
        max_tokens=120,
        temperature=0.7,
        typo_rate=0.012,
        # ✅ v1.3: plantilla Meta para primer contacto (regla 24h).
        # Estos valores DEBEN coincidir EXACTAMENTE con lo aprobado en Meta
        # Business Manager. Si la plantilla se aprueba con otro idioma
        # (ej. "es" o "es_MX"), cambiar outreach_template_lang aquí.
        outreach_template_name="saludo",
        outreach_template_lang="es_CO",
        outreach_param_max_tokens=60,
    )

    db_manager = DatabaseManager(
        SECRETS['DB_USER'],
        SECRETS['DB_PASSWORD'],
        DB_HOST_WITH_PORT,
        SECRETS['DB_NAME']
    )

    wa_client = WhatsAppClient(wa_config)
    ai_client = AnthropicAIClient(
        anthropic_config,
        '/content/drive/MyDrive/BASEPROMPT.xml'
    )
    extraction_client = AnthropicExtractionClient(anthropic_config)
    # CSV incremental para cotizaciones extraídas de PDFs (estructura mirror
    # de la tabla `cotizaciones`). Mismo path-style que ARGOS.
    CSV_COTIZACIONES_PDF = "/content/drive/MyDrive/cotizaciones_pdf.csv"

    # ✅ v1.1: OperatingHoursGate
    # Convención weekday: lunes=0, martes=1, ..., sábado=5, domingo=6.
    # Modificar este dict para ajustar el horario operativo del bot.
    # Ejemplo (lun-vie 08:00–18:00, sáb 09:00–13:00, dom cerrado):
    operating_hours_config = OperatingHoursConfig(windows={
        0: (0, 24),    # lunes
        1: (0, 24),    # martes
        2: (0, 24),    # miércoles
        3: (0, 24),    # jueves
        4: (0, 24),    # viernes
        5: (0, 24),    # sábado
        6: (0, 24),    # domingo (cerrado)
    })
    hours_gate = OperatingHoursGate(operating_hours_config)

    # ✅ Dispatcher con delays humanos + anti-bloqueo Meta + gate horario.
    # ✅ v1.4.2: rediseño de delays:
    #   listen_window:  120..300s → tiempo de escucha activa (debounce)
    #                   antes de responder al mismo chat. Cada mensaje
    #                   nuevo de la misma ferretería REINICIA este timer.
    #   outreach_delay: 60..300s → delay aleatorio antes del primer envío
    #                   proactivo (saludo). Evita que todas las ferreterías
    #                   reciban el mensaje en el mismo segundo cuando el
    #                   cron dispara el broadcast.
    #   inter_chat:     120..300s → entre envíos físicos a números distintos.
    dispatcher_config = DispatcherConfig(
        listen_window_min_s=120, listen_window_max_s=300,
        outreach_delay_min_s=60, outreach_delay_max_s=300,
        inter_chat_min_s=120, inter_chat_max_s=300,
    )
    dispatcher = MessageDispatcher(
        wa_client, dispatcher_config, hours_gate=hours_gate
    )

    # ✅ v1.4: el extractor acumulativo se construye DESPUÉS, una vez
    # que tengamos los gestores de productos y marcas inicializados.
    # Por ahora dejamos extractor_acumulativo=None y lo asignamos abajo.
    processor = MessageProcessor(
        wa_client, ai_client, db_manager,
        extraction_client=extraction_client,
        extractor_acumulativo=None,  # se asigna al final del bloque
        csv_cotizaciones_pdf=CSV_COTIZACIONES_PDF,
        dispatcher=dispatcher,
        hours_gate=hours_gate,
    )
    # ✅ v1.4.2: registrar el callback que el dispatcher invocará cuando
    # venza un inbound_intent. Es el puente entre la cola (que solo sabe
    # de tiempos) y el processor (que sabe cómo generar la respuesta).
    dispatcher.set_inbound_intent_callback(processor._procesar_inbound_intent)

    broadcast_scheduler = BroadcastScheduler(
        db_manager, wa_client, ai_client,
        processor=processor,
        dispatcher=dispatcher,
        hours_gate=hours_gate,
    )

    # ─────────────────────────────────────────────────────────────────────
    # ✅ NUEVO — Sistema de identificación de productos por similitud
    # (definido en CELDA 6.5). Carga el catálogo de `productos` desde BD
    # y queda disponible como `gestor_busqueda` (variable global) y
    # también como atributo `processor.gestor_busqueda` para uso futuro.
    #
    # No modifica el flujo existente del bot: el `MessageProcessor` y el
    # `DatabaseManager` siguen funcionando exactamente igual que antes.
    # Esta funcionalidad queda lista para invocarse explícitamente cuando
    # se necesite resolver un nombre de producto con typos/variaciones.
    # ─────────────────────────────────────────────────────────────────────
    try:
        gestor_busqueda = GestorBusquedaProductos(db_manager)
        # Adjuntar al processor como atributo sin tocar su __init__:
        # cualquier código futuro puede acceder vía `processor.gestor_busqueda`.
        processor.gestor_busqueda = gestor_busqueda
        _gb_ok = True
    except Exception as _gb_e:
        gestor_busqueda = None
        if hasattr(processor, "__dict__"):
            processor.gestor_busqueda = None
        _gb_ok = False
        logger.warning(f"⚠️  GestorBusquedaProductos no disponible: {_gb_e}")

    # ─────────────────────────────────────────────────────────────────────
    # ✅ v1.4 — Extractor acumulativo de cotizaciones por TEXTO
    # Reemplaza al extractor de un solo turno para detectar `inicio→cotizacion`.
    # Reconstruye el estado de la cotización a partir del historial completo,
    # detecta precios deterministamente, empareja producto/marca contra
    # catálogos con fuzzy matching, y solo invoca al LLM como fallback acotado.
    # ─────────────────────────────────────────────────────────────────────
    try:
        gestor_marcas = GestorBusquedaMarcas(db_manager)
        extractor_acumulativo = ExtractorTextoAcumulativo(
            gestor_productos=gestor_busqueda,
            gestor_marcas=gestor_marcas,
            anthropic_client=extraction_client,
            threshold_producto=0.75,
            threshold_marca=0.80,
        )
        processor.extractor_acumulativo = extractor_acumulativo
        processor.gestor_marcas = gestor_marcas
        _ea_ok = True
    except Exception as _ea_e:
        gestor_marcas = None
        extractor_acumulativo = None
        processor.extractor_acumulativo = None
        processor.gestor_marcas = None
        _ea_ok = False
        logger.warning(f"⚠️  ExtractorTextoAcumulativo no disponible: {_ea_e}")

    print("✅ Componentes inicializados correctamente")
    print(f"   • Modelo IA: {anthropic_config.model_name}")
    print(f"   • Regiones cargadas: {list(ai_client.prompts['region'].keys())}")
    print(f"   • Estados cargados: {list(ai_client.prompts['estado'].keys())}")
    print(f"   • Ventana operativa: configurada por día de semana")
    print(f"     (lun-vie 8-18, sáb 9-13, dom cerrado — editar en celda 11)")
    if _gb_ok:
        print(f"   • ProductSimilarity: {len(gestor_busqueda.buscador.catalogo)} productos en catálogo")
    else:
        print(f"   • ProductSimilarity: NO disponible (revisar logs)")
    if _ea_ok:
        n_marcas = len(gestor_marcas.buscador.catalogo) if gestor_marcas and gestor_marcas.buscador else 0
        print(f"   • Extractor Acumulativo: ACTIVO ({n_marcas} marcas en catálogo)")
    else:
        print(f"   • Extractor Acumulativo: NO disponible (usando solo LLM)")
except Exception as e:
    print(f"❌ Error durante inicialización: {e}")
    raise


##CELDA DE PRUEBA

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# 1) Reiniciar agendamiento
broadcast_scheduler.start_scheduler()  # idempotente, no pasa nada si ya estaba

# 2) Calcular una hora 3 minutos en el futuro
BOG = ZoneInfo("America/Bogota")
now = datetime.now(BOG)
target_hour = now.hour
target_min = now.minute + 2
if target_min >= 60:
    target_min -= 60
    target_hour = (target_hour + 1) % 24

print(f"Ahora: {now.strftime('%A %H:%M')}")
print(f"Agendando broadcast para hoy a las {target_hour:01d}:{target_min:02d}")

# 3) Agendar para HOY (todos los días) en esa hora
broadcast_scheduler.schedule_weekly_broadcast(
    topic="Test cron a 3 minutos",
    day_of_week='*',  # cualquier día
    hour=target_hour,
    minute=target_min,
)

# 4) Verificar que el job está agendado
for j in broadcast_scheduler.scheduler.get_jobs():
    print(f"  • {j.id}: próximo disparo → {j.next_run_time}")

## CELDA 12: INICIAR SCHEDULER Y CONFIGURAR BROADCASTS

In [ ]:
try:
    broadcast_scheduler.start_scheduler()

    broadcast_scheduler.schedule_weekly_broadcast(
        topic="Buenos días, ¿cómo están los precios de cemento esta semana?",
        day_of_week='sun',
        hour=19,
        minute=5

    )

    print("✅ Scheduler activo:")
    print("   - archive_old_interactions → diario 02:00")
    print("   - check_unresponsive_ferreterias → diario 03:00")
    print(f"   - broadcast_sat → {day_of_week}:{hour}:{minute}")
except Exception as e:
    print(f"❌ Error al configurar scheduler: {e}")

## CELDA 12.5: API HTTP PARA EL DASHBOARD WEB

Agrega un mini-API REST encima del WebhookServer existente. El frontend (Alpha Bot Web) consume estos endpoints vía la misma URL pública de ngrok que ya se usa para `/webhook` de WhatsApp.

Está dividida en 2 celdas:
- **12.5.1** define estado global (config en memoria, ring buffer de logs, broadcasts dinámicos, loader Argos).
- **12.5.2** define `WebhookServerExtendido` con todas las rutas `/api/*` y los builders Plotly de Argos.

La celda 13 ya está modificada para usar `WebhookServerExtendido`.


In [ ]:
# =============================================================================
# CELDA 12.5.1 — ESTADO GLOBAL DE LA API + HELPERS
#
# Objetivos:
#   - Definir un store en memoria con las 5 secciones de configuración
#     que el dashboard consume (dispatcher, anthropic, operating_hours,
#     webhook, extras). El bot original ya tiene estos valores
#     repartidos por dataclasses; aquí los exponemos vía un wrapper para
#     que el dashboard pueda leerlos y actualizarlos en caliente.
#   - Conectar un handler de logging a un ring buffer in-memory para
#     servir /api/bot/logs.
#   - Manejar broadcasts dinámicos (CRUD desde el dashboard) sin tocar
#     el broadcast cron que ya está hardcodeado en la celda 12.
#   - Loader perezoso de los CSV Argos para los 4 endpoints /api/argos/*.
#
# Esta celda NO arranca nada, solo define el estado. La celda 12.5.2
# extiende WebhookServer para usar este estado.
# =============================================================================

import threading
import collections
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from typing import Any, Dict, List, Optional
import uuid as _uuid

# ─── 1. Ring buffer de logs ───────────────────────────────────────────────
class _RingBufferLogHandler(logging.Handler):
    """
    Handler de logging que guarda las últimas N entradas en memoria.
    El dashboard hace polling cada 5s a /api/bot/logs y va viendo lo
    nuevo. NO persiste: si reinicias el kernel, el buffer queda vacío.
    """
    def __init__(self, maxlen: int = 1000):
        super().__init__()
        self._buf = collections.deque(maxlen=maxlen)
        self._lock = threading.Lock()

    def emit(self, record: logging.LogRecord) -> None:
        try:
            msg = record.getMessage()
        except Exception:
            msg = str(record.msg)
        entry = {
            "timestamp": datetime.fromtimestamp(
                record.created, tz=timezone.utc
            ).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": msg,
        }
        with self._lock:
            self._buf.append(entry)

    def get(self, level: Optional[str] = None, limit: int = 200,
            since: Optional[str] = None) -> List[Dict[str, Any]]:
        with self._lock:
            items = list(self._buf)
        if level:
            level = level.upper().strip()
            items = [x for x in items if x["level"] == level]
        if since:
            items = [x for x in items if x["timestamp"] > since]
        # Devolver los últimos `limit` (más recientes al final)
        return items[-limit:]


LOG_BUFFER = _RingBufferLogHandler(maxlen=1000)
# Adjuntamos al logger raíz para capturar TODO el logging del bot.
# Aplicamos también el PIIFilter que ya está definido en la CELDA 3,
# para no exponer teléfonos crudos por la API.
LOG_BUFFER.setLevel(logging.DEBUG)
LOG_BUFFER.addFilter(PIIFilter())
logging.getLogger().addHandler(LOG_BUFFER)


# ─── 2. Lock global y configuración mutable ───────────────────────────────
CONFIG_LOCK = threading.RLock()


def _operating_hours_to_dict() -> Dict[str, Any]:
    """
    Serializa el dict del OperatingHoursGate al formato que espera el
    dashboard:
        { "windows": {"0": [8,18], "1": [8,18], ..., "6": null} }
    El front usa claves string ("0".."6") y valor `null` para "cerrado".
    """
    raw = hours_gate.config.windows  # dict[int, tuple[int,int] | None]
    out: Dict[str, Any] = {}
    for k, v in raw.items():
        if v is None:
            out[str(k)] = None
        else:
            opens, closes = v
            out[str(k)] = [int(opens), int(closes)]
    return {"windows": out}


def _operating_hours_from_dict(payload: Dict[str, Any]) -> Dict[int, Any]:
    """Inversa: dict del dashboard → formato que entiende OperatingHoursConfig."""
    windows = payload.get("windows", {})
    out: Dict[int, Any] = {}
    for k, v in windows.items():
        day = int(k)
        if v is None:
            out[day] = None
        else:
            opens, closes = int(v[0]), int(v[1])
            if not (0 <= opens < closes <= 24):
                raise ValueError(
                    f"Ventana inválida para día {day}: opens={opens}, closes={closes}"
                )
            out[day] = (opens, closes)
    return out


def cfg_get_all() -> Dict[str, Any]:
    """Snapshot de las 5 secciones de configuración."""
    with CONFIG_LOCK:
        return {
            "dispatcher": cfg_get("dispatcher"),
            "anthropic": cfg_get("anthropic"),
            "operating_hours": cfg_get("operating_hours"),
            "webhook": cfg_get("webhook"),
            "extras": cfg_get("extras"),
        }


def cfg_get(section: str) -> Dict[str, Any]:
    """Devuelve la sección pedida."""
    with CONFIG_LOCK:
        if section == "dispatcher":
            dc = dispatcher.config
            return {
                "listen_window_min_s": dc.listen_window_min_s,
                "listen_window_max_s": dc.listen_window_max_s,
                "outreach_delay_min_s": dc.outreach_delay_min_s,
                "outreach_delay_max_s": dc.outreach_delay_max_s,
                "inter_chat_min_s": dc.inter_chat_min_s,
                "inter_chat_max_s": dc.inter_chat_max_s,
            }
        if section == "anthropic":
            ac = anthropic_config
            return {
                "model_name": ac.model_name,
                "max_tokens": ac.max_tokens,
                "temperature": ac.temperature,
                "typo_rate": ac.typo_rate,
                "history_limit": getattr(ac, "history_limit", 20),
            }
        if section == "operating_hours":
            return _operating_hours_to_dict()
        if section == "webhook":
            return {
                "outreach_template_name": anthropic_config.outreach_template_name,
                "outreach_template_lang": anthropic_config.outreach_template_lang,
            }
        if section == "extras":
            return {
                "paused": EXTRAS["paused"],
                "dry_run": EXTRAS["dry_run"],
                "whitelist": list(EXTRAS["whitelist"]),
                "daily_quota": EXTRAS["daily_quota"],
                "log_level": EXTRAS["log_level"],
            }
        raise KeyError(f"sección desconocida: {section}")


def cfg_set(section: str, body: Dict[str, Any]) -> Dict[str, Any]:
    """Aplica cambios a la sección y devuelve el estado nuevo."""
    with CONFIG_LOCK:
        if section == "dispatcher":
            dc = dispatcher.config
            for k in ("listen_window_min_s", "listen_window_max_s",
                     "outreach_delay_min_s", "outreach_delay_max_s",
                     "inter_chat_min_s", "inter_chat_max_s"):
                if k in body:
                    v = int(body[k])
                    if not (0 <= v <= 3600):
                        raise ValueError(f"{k} fuera de rango [0..3600]")
                    setattr(dc, k, v)
            # Validar coherencia min <= max para los 3 pares
            for lo, hi in (("listen_window_min_s", "listen_window_max_s"),
                          ("outreach_delay_min_s", "outreach_delay_max_s"),
                          ("inter_chat_min_s", "inter_chat_max_s")):
                if getattr(dc, lo) > getattr(dc, hi):
                    raise ValueError(f"{lo} > {hi}")
            return cfg_get("dispatcher")

        if section == "anthropic":
            ac = anthropic_config
            if "model_name" in body:
                ac.model_name = str(body["model_name"]).strip()
            if "max_tokens" in body:
                v = int(body["max_tokens"])
                if not (1 <= v <= 8192):
                    raise ValueError("max_tokens debe estar en [1..8192]")
                ac.max_tokens = v
            if "temperature" in body:
                v = float(body["temperature"])
                if not (0.0 <= v <= 1.0):
                    raise ValueError("temperature debe estar en [0..1]")
                ac.temperature = v
            if "typo_rate" in body:
                v = float(body["typo_rate"])
                if not (0.0 <= v <= 0.05):
                    raise ValueError("typo_rate debe estar en [0..0.05]")
                ac.typo_rate = v
            if "history_limit" in body and hasattr(ac, "history_limit"):
                ac.history_limit = int(body["history_limit"])
            return cfg_get("anthropic")

        if section == "operating_hours":
            new_windows = _operating_hours_from_dict(body)
            hours_gate.config.windows = new_windows
            return cfg_get("operating_hours")

        if section == "webhook":
            if "outreach_template_name" in body:
                anthropic_config.outreach_template_name = str(
                    body["outreach_template_name"]
                ).strip()
            if "outreach_template_lang" in body:
                anthropic_config.outreach_template_lang = str(
                    body["outreach_template_lang"]
                ).strip()
            return cfg_get("webhook")

        if section == "extras":
            if "paused" in body:
                EXTRAS["paused"] = bool(body["paused"])
            if "dry_run" in body:
                EXTRAS["dry_run"] = bool(body["dry_run"])
            if "whitelist" in body:
                wl = body["whitelist"] or []
                if not isinstance(wl, list):
                    raise ValueError("whitelist debe ser lista")
                EXTRAS["whitelist"] = [str(x).strip() for x in wl if str(x).strip()]
            if "daily_quota" in body:
                v = int(body["daily_quota"])
                if not (0 <= v <= 100000):
                    raise ValueError("daily_quota fuera de rango")
                EXTRAS["daily_quota"] = v
            if "log_level" in body:
                lvl = str(body["log_level"]).upper().strip()
                if lvl not in ("DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"):
                    raise ValueError(f"log_level inválido: {lvl}")
                EXTRAS["log_level"] = lvl
                logging.getLogger().setLevel(lvl)
            return cfg_get("extras")

        raise KeyError(f"sección desconocida: {section}")


# Estado "extras" (no existía en el bot original, vive solo en RAM).
EXTRAS = {
    "paused": False,
    "dry_run": False,
    "whitelist": [],         # lista de phone numbers; vacía = todos
    "daily_quota": 1000,     # tope diario
    "log_level": "INFO",
    "_quota_used_today": 0,
    "_quota_day": datetime.now(ZoneInfo("America/Bogota")).date().isoformat(),
}


def _reset_daily_quota_if_needed() -> None:
    today = datetime.now(ZoneInfo("America/Bogota")).date().isoformat()
    if today != EXTRAS["_quota_day"]:
        EXTRAS["_quota_used_today"] = 0
        EXTRAS["_quota_day"] = today


# ─── 3. Broadcasts dinámicos ───────────────────────────────────────────────
# El bot original solo permite un broadcast "cableado" en la celda 12. Aquí
# montamos una capa CRUD que añade/quita jobs en APScheduler en caliente.
BROADCASTS: Dict[str, Dict[str, Any]] = {}
BROADCASTS_LOCK = threading.RLock()


def _broadcast_to_dict(b: Dict[str, Any]) -> Dict[str, Any]:
    """Forma que espera el dashboard."""
    return {
        "id": b["id"],
        "topic": b["topic"],
        "day_of_week": b["day_of_week"],
        "hour": b["hour"],
        "minute": b["minute"],
        "enabled": b["enabled"],
        "last_run_at": b.get("last_run_at"),
    }


def _broadcast_job_id(bid: str) -> str:
    return f"dyn_broadcast_{bid}"


def _broadcast_register_job(b: Dict[str, Any]) -> None:
    """(Re)agenda el job de APScheduler para este broadcast."""
    from apscheduler.triggers.cron import CronTrigger
    job_id = _broadcast_job_id(b["id"])
    # Quitar el viejo si existía
    try:
        broadcast_scheduler.scheduler.remove_job(job_id)
    except Exception:
        pass
    if not b["enabled"]:
        return

    def _run():
        try:
            with BROADCASTS_LOCK:
                if b["id"] not in BROADCASTS:
                    return
                BROADCASTS[b["id"]]["last_run_at"] = datetime.now(
                    timezone.utc
                ).isoformat()
            # Reutilizamos la lógica del scheduler original para enviar
            # outreach a todas las ferreterías con estado NULL.
            broadcast_scheduler._run_broadcast_job(b["topic"])
        except Exception as e:
            logger.error(f"Broadcast dyn {b['id']} falló: {e}")

    trigger = CronTrigger(
        day_of_week=b["day_of_week"],
        hour=b["hour"],
        minute=b["minute"],
        timezone=ZoneInfo("America/Bogota"),
    )
    broadcast_scheduler.scheduler.add_job(
        _run, trigger=trigger, id=job_id,
        name=f"broadcast_{b['day_of_week']}_{b['hour']:02d}{b['minute']:02d}",
        replace_existing=True,
    )


def broadcasts_list() -> List[Dict[str, Any]]:
    with BROADCASTS_LOCK:
        return [_broadcast_to_dict(b) for b in BROADCASTS.values()]


def broadcasts_create(body: Dict[str, Any]) -> Dict[str, Any]:
    bid = str(_uuid.uuid4())
    b = {
        "id": bid,
        "topic": str(body["topic"]).strip(),
        "day_of_week": str(body["day_of_week"]).strip(),
        "hour": int(body["hour"]),
        "minute": int(body["minute"]),
        "enabled": bool(body.get("enabled", True)),
        "last_run_at": None,
    }
    if len(b["topic"]) < 3:
        raise ValueError("topic muy corto (mínimo 3 chars)")
    if b["day_of_week"] not in ("mon", "tue", "wed", "thu", "fri", "sat", "sun"):
        raise ValueError("day_of_week inválido")
    if not (0 <= b["hour"] <= 23):
        raise ValueError("hour fuera de rango")
    if not (0 <= b["minute"] <= 59):
        raise ValueError("minute fuera de rango")
    with BROADCASTS_LOCK:
        BROADCASTS[bid] = b
        _broadcast_register_job(b)
    return _broadcast_to_dict(b)


def broadcasts_update(bid: str, body: Dict[str, Any]) -> Dict[str, Any]:
    with BROADCASTS_LOCK:
        if bid not in BROADCASTS:
            raise KeyError("broadcast no existe")
        b = BROADCASTS[bid]
        for k in ("topic", "day_of_week", "hour", "minute", "enabled"):
            if k in body:
                b[k] = body[k]
                if k == "topic":
                    b[k] = str(b[k]).strip()
                if k in ("hour", "minute"):
                    b[k] = int(b[k])
                if k == "enabled":
                    b[k] = bool(b[k])
        _broadcast_register_job(b)
        return _broadcast_to_dict(b)


def broadcasts_delete(bid: str) -> None:
    with BROADCASTS_LOCK:
        if bid not in BROADCASTS:
            raise KeyError("broadcast no existe")
        try:
            broadcast_scheduler.scheduler.remove_job(_broadcast_job_id(bid))
        except Exception:
            pass
        del BROADCASTS[bid]


def broadcasts_run_now(bid: str) -> Dict[str, Any]:
    """Ejecuta el broadcast inmediatamente, respetando gate y pause."""
    with BROADCASTS_LOCK:
        if bid not in BROADCASTS:
            raise KeyError("broadcast no existe")
        b = BROADCASTS[bid]
        topic = b["topic"]
    if EXTRAS["paused"]:
        return {"ok": False, "reason": "bot pausado"}
    if not hours_gate.is_open(datetime.now(ZoneInfo("America/Bogota"))):
        return {"ok": False, "reason": "fuera de ventana operativa"}
    try:
        result = broadcast_scheduler._run_broadcast_job(topic)
        with BROADCASTS_LOCK:
            if bid in BROADCASTS:
                BROADCASTS[bid]["last_run_at"] = datetime.now(
                    timezone.utc
                ).isoformat()
        # _run_broadcast_job del bot devuelve None en el código original;
        # devolvemos un sumario aproximado a partir de los logs.
        if isinstance(result, dict):
            return {"ok": True, **result}
        return {"ok": True, "enviadas": 0, "falladas": 0}
    except Exception as e:
        logger.error(f"run_now {bid} falló: {e}")
        return {"ok": False, "reason": str(e)}


# Si existía el broadcast cableado de la celda 12, lo registramos también
# como un broadcast "dyn" para que aparezca en el dashboard. Buscamos
# cualquier job en APScheduler cuyo nombre empiece con "broadcast_" y
# que no sea un dyn_broadcast_ ya registrado.
def _import_legacy_broadcast() -> None:
    try:
        for job in broadcast_scheduler.scheduler.get_jobs():
            if job.id.startswith("dyn_broadcast_"):
                continue
            if "broadcast" not in job.id.lower():
                continue
            # Reconstruir los campos a partir del trigger (CronTrigger)
            trig = job.trigger
            day_of_week_field = None
            hour_field = None
            minute_field = None
            for f in getattr(trig, "fields", []):
                if f.name == "day_of_week":
                    day_of_week_field = str(f).split(",")[0].lower() or "sun"
                elif f.name == "hour":
                    hour_field = int(str(f))
                elif f.name == "minute":
                    minute_field = int(str(f))
            if None in (day_of_week_field, hour_field, minute_field):
                continue
            # Mapeo APS → claves del dashboard (mon/tue/...)
            dow_map = {"0": "mon", "1": "tue", "2": "wed", "3": "thu",
                       "4": "fri", "5": "sat", "6": "sun",
                       "mon": "mon", "tue": "tue", "wed": "wed", "thu": "thu",
                       "fri": "fri", "sat": "sat", "sun": "sun"}
            day_of_week_field = dow_map.get(day_of_week_field, "sun")
            bid = str(_uuid.uuid4())
            BROADCASTS[bid] = {
                "id": bid,
                "topic": "(broadcast legacy de la celda 12)",
                "day_of_week": day_of_week_field,
                "hour": hour_field,
                "minute": minute_field,
                "enabled": True,
                "last_run_at": None,
            }
    except Exception as e:
        logger.warning(f"No se pudo importar broadcast legacy: {e}")


_import_legacy_broadcast()


# ─── 4. Loader perezoso de los CSV Argos ──────────────────────────────────
# Los 3 CSV ya existen en Drive (definidos en la celda 14). Los leemos
# bajo demanda y los transformamos a estructuras Plotly que entiende
# el dashboard. El cache se invalida con POST /api/argos/refresh.

ARGOS_PATHS = {
    "precios_regionales": "/content/drive/MyDrive/argos_precios_regionales.csv",
    "intervalos_hdi":     "/content/drive/MyDrive/argos_intervalos_hdi.csv",
    "perfiles_alertas":   "/content/drive/MyDrive/argos_perfiles_alertas.csv",
}
_ARGOS_CACHE: Dict[str, Any] = {}
ARGOS_LOCK = threading.RLock()


def _argos_load_csv(name: str):
    """Lee el CSV correspondiente y lo cachea. Devuelve DataFrame o None."""
    import pandas as pd
    path = ARGOS_PATHS[name]
    with ARGOS_LOCK:
        if name in _ARGOS_CACHE:
            return _ARGOS_CACHE[name]
        if not os.path.exists(path):
            return None
        try:
            df = pd.read_csv(path)
            _ARGOS_CACHE[name] = df
            return df
        except Exception as e:
            logger.error(f"Error leyendo Argos CSV {name}: {e}")
            return None


def argos_refresh(name: Optional[str] = None) -> Dict[str, Any]:
    """Invalida cache. Si name=None, invalida todo."""
    with ARGOS_LOCK:
        if name:
            _ARGOS_CACHE.pop(name, None)
            return {"refreshed": [name]}
        keys = list(_ARGOS_CACHE.keys())
        _ARGOS_CACHE.clear()
        return {"refreshed": keys}


def argos_files() -> Dict[str, Any]:
    """Estado de los 3 CSVs (existe / tamaño / mtime)."""
    out = []
    for name, path in ARGOS_PATHS.items():
        info = {"name": name, "path": path, "exists": os.path.exists(path)}
        if info["exists"]:
            st = os.stat(path)
            info["size_bytes"] = st.st_size
            info["modified_at"] = datetime.fromtimestamp(
                st.st_mtime, tz=timezone.utc
            ).isoformat()
        out.append(info)
    return {"files": out}


print("✅ Estado API global inicializado")
print(f"   • Ring buffer logs: {LOG_BUFFER._buf.maxlen} entradas")
print(f"   • Broadcasts cargados: {len(BROADCASTS)}")
print(f"   • Config secciones: dispatcher, anthropic, operating_hours, webhook, extras")


In [ ]:
# =============================================================================
# CELDA 12.5.2 — WEBHOOK SERVER EXTENDIDO CON LA API DEL DASHBOARD
#
# Subclasea WebhookServer y le añade todas las rutas /api/* que consume
# el dashboard (frontend en GitHub Pages). Usa Flask-CORS para permitir
# requests cross-origin desde el dominio GitHub Pages.
# =============================================================================
from flask_cors import CORS


def _bot_status_payload() -> Dict[str, Any]:
    """Snapshot que consume el Dashboard cada 5s."""
    _reset_daily_quota_if_needed()
    now = datetime.now(ZoneInfo("America/Bogota"))
    gate_open = hours_gate.is_open(now)

    # Calcular opens_at / closes_at: lo más simple es buscar el próximo
    # cambio en las próximas 7*24 horas barriendo de hora en hora.
    opens_at = None
    closes_at = None
    cursor = now
    last_state = gate_open
    for _ in range(7 * 24):
        cursor = cursor + timedelta(hours=1)
        st = hours_gate.is_open(cursor)
        if st != last_state:
            if st:  # acaba de abrir
                if opens_at is None:
                    opens_at = cursor.replace(minute=0, second=0, microsecond=0).isoformat()
            else:   # acaba de cerrar
                if closes_at is None:
                    closes_at = cursor.replace(minute=0, second=0, microsecond=0).isoformat()
            last_state = st
        if opens_at and closes_at:
            break

    # Jobs del scheduler (incluye los dyn + el cableado + los de mantenimiento)
    jobs_out = []
    try:
        for j in broadcast_scheduler.scheduler.get_jobs():
            nrt = getattr(j, "next_run_time", None)
            jobs_out.append({
                "id": j.id,
                "name": j.name or j.id,
                "next_run_time": nrt.isoformat() if nrt else None,
            })
        # Filtramos los jobs de mantenimiento internos para no agobiar
        jobs_out = [j for j in jobs_out if "broadcast" in (j["id"] + j["name"]).lower()]
        jobs_out.sort(key=lambda x: x["next_run_time"] or "")
    except Exception:
        pass

    # Cola del dispatcher
    try:
        pending = dispatcher.pending() if hasattr(dispatcher, "pending") else 0
    except Exception:
        pending = 0
    try:
        # El dispatcher original mantiene self._worker (Thread). Si está vivo,
        # está corriendo. Fallback a flags por si la implementación cambia.
        worker = getattr(dispatcher, "_worker", None)
        if worker is not None:
            running = bool(worker.is_alive())
        else:
            running = bool(getattr(dispatcher, "_running", False) or
                          getattr(dispatcher, "running", False))
    except Exception:
        running = False

    return {
        "now": now.isoformat(),
        "dispatcher": {
            "running": running,
            "pending": pending,
            "quota_used_today": EXTRAS["_quota_used_today"],
            "quota_total": EXTRAS["daily_quota"],
        },
        "scheduler": {
            "running": broadcast_scheduler.scheduler.running,
            "jobs": jobs_out,
        },
        "gate": {
            "open": gate_open,
            "opens_at": opens_at,
            "closes_at": closes_at,
        },
        "extras": {
            "paused": EXTRAS["paused"],
            "dry_run": EXTRAS["dry_run"],
            "log_level": EXTRAS["log_level"],
            "whitelist_count": len(EXTRAS["whitelist"]),
        },
    }


class WebhookServerExtendido(WebhookServer):
    """
    Extiende el WebhookServer original con todas las rutas /api/* que
    consume el dashboard. NO toca /webhook ni /health del padre.
    """

    def __init__(self, wa_config, processor, ngrok_token):
        super().__init__(wa_config, processor, ngrok_token)
        # CORS: aceptamos GitHub Pages, ngrok urls, y localhost (para dev).
        # Si tu GitHub Pages está bajo dominio propio, añádelo a la lista.
        CORS(
            self.app,
            resources={r"/api/*": {"origins": "*"},
                       r"/health":  {"origins": "*"}},
            supports_credentials=False,
        )
        self._setup_api_routes()

    def _setup_api_routes(self) -> None:
        app = self.app

        # ── Bot status & control ────────────────────────────────────
        @app.route("/api/bot/status", methods=["GET"])
        def _api_bot_status():
            try:
                return jsonify(_bot_status_payload()), 200
            except Exception as e:
                logger.error(f"/api/bot/status: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/bot/pause", methods=["POST"])
        def _api_bot_pause():
            with CONFIG_LOCK:
                EXTRAS["paused"] = True
            logger.info("Bot pausado desde dashboard")
            return jsonify({"ok": True, "paused": True}), 200

        @app.route("/api/bot/resume", methods=["POST"])
        def _api_bot_resume():
            with CONFIG_LOCK:
                EXTRAS["paused"] = False
            logger.info("Bot reanudado desde dashboard")
            return jsonify({"ok": True, "paused": False}), 200

        @app.route("/api/bot/logs", methods=["GET"])
        def _api_bot_logs():
            level = request.args.get("level")
            try:
                limit = int(request.args.get("limit", 200))
            except ValueError:
                limit = 200
            since = request.args.get("since")
            records = LOG_BUFFER.get(level=level, limit=limit, since=since)
            return jsonify({
                "records": records,
                "level_filter": level or None,
            }), 200

        # ── Config (5 secciones) ────────────────────────────────────
        @app.route("/api/config", methods=["GET"])
        def _api_config_all():
            try:
                return jsonify(cfg_get_all()), 200
            except Exception as e:
                logger.error(f"/api/config: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/config/<section>", methods=["GET"])
        def _api_config_section_get(section):
            try:
                return jsonify(cfg_get(section)), 200
            except KeyError as e:
                return jsonify({"detail": str(e)}), 404
            except Exception as e:
                logger.error(f"/api/config/{section} GET: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/config/<section>", methods=["PUT"])
        def _api_config_section_put(section):
            try:
                body = request.get_json(silent=True) or {}
                new_value = cfg_set(section, body)
                logger.info(f"Config actualizada: sección={section}")
                return jsonify(new_value), 200
            except KeyError as e:
                return jsonify({"detail": str(e)}), 404
            except ValueError as e:
                return jsonify({"detail": str(e)}), 422
            except Exception as e:
                logger.error(f"/api/config/{section} PUT: {e}")
                return jsonify({"detail": str(e)}), 500

        # ── Broadcasts CRUD ──────────────────────────────────────────
        @app.route("/api/broadcasts", methods=["GET"])
        def _api_broadcasts_list():
            return jsonify(broadcasts_list()), 200

        @app.route("/api/broadcasts", methods=["POST"])
        def _api_broadcasts_create():
            try:
                body = request.get_json(silent=True) or {}
                b = broadcasts_create(body)
                return jsonify(b), 201
            except (ValueError, KeyError) as e:
                return jsonify({"detail": str(e)}), 422
            except Exception as e:
                logger.error(f"POST /api/broadcasts: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/broadcasts/<bid>", methods=["PUT"])
        def _api_broadcasts_update(bid):
            try:
                body = request.get_json(silent=True) or {}
                b = broadcasts_update(bid, body)
                return jsonify(b), 200
            except KeyError:
                return jsonify({"detail": "broadcast no existe"}), 404
            except ValueError as e:
                return jsonify({"detail": str(e)}), 422
            except Exception as e:
                logger.error(f"PUT /api/broadcasts/{bid}: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/broadcasts/<bid>", methods=["DELETE"])
        def _api_broadcasts_delete(bid):
            try:
                broadcasts_delete(bid)
                return jsonify({"ok": True}), 200
            except KeyError:
                return jsonify({"detail": "broadcast no existe"}), 404
            except Exception as e:
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/broadcasts/<bid>/run-now", methods=["POST"])
        def _api_broadcasts_run_now(bid):
            try:
                return jsonify(broadcasts_run_now(bid)), 200
            except KeyError:
                return jsonify({"detail": "broadcast no existe"}), 404
            except Exception as e:
                return jsonify({"detail": str(e)}), 500

        # ── Argos (4 endpoints + files + refresh) ────────────────────
        @app.route("/api/argos/precios-regionales", methods=["GET"])
        def _api_argos_precios():
            try:
                return jsonify(_argos_chart_precios_regionales()), 200
            except DriveNotConfiguredError as e:
                return jsonify({"detail": str(e)}), 503
            except FileNotFoundError as e:
                return jsonify({"detail": str(e)}), 404
            except Exception as e:
                logger.error(f"/api/argos/precios-regionales: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/argos/intervalos-hdi", methods=["GET"])
        def _api_argos_intervalos():
            cod = request.args.get("cod_municipio")
            try:
                return jsonify(_argos_chart_intervalos_hdi(cod)), 200
            except DriveNotConfiguredError as e:
                return jsonify({"detail": str(e)}), 503
            except FileNotFoundError as e:
                return jsonify({"detail": str(e)}), 404
            except Exception as e:
                logger.error(f"/api/argos/intervalos-hdi: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/argos/perfiles-alertas", methods=["GET"])
        def _api_argos_perfiles():
            try:
                return jsonify(_argos_chart_perfiles_alertas()), 200
            except DriveNotConfiguredError as e:
                return jsonify({"detail": str(e)}), 503
            except FileNotFoundError as e:
                return jsonify({"detail": str(e)}), 404
            except Exception as e:
                logger.error(f"/api/argos/perfiles-alertas: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/argos/mapa", methods=["GET"])
        def _api_argos_mapa():
            try:
                return jsonify(_argos_chart_mapa()), 200
            except DriveNotConfiguredError as e:
                return jsonify({"detail": str(e)}), 503
            except FileNotFoundError as e:
                return jsonify({"detail": str(e)}), 404
            except Exception as e:
                logger.error(f"/api/argos/mapa: {e}")
                return jsonify({"detail": str(e)}), 500

        @app.route("/api/argos/files", methods=["GET"])
        def _api_argos_files():
            return jsonify(argos_files()), 200

        @app.route("/api/argos/refresh", methods=["POST"])
        def _api_argos_refresh():
            name = request.args.get("name")
            return jsonify(argos_refresh(name)), 200


# ───────────────────────────────────────────────────────────────────────
# Builders Plotly para los 4 charts Argos.
# Cada función lee el CSV correspondiente, lo agrega, y emite el shape
# exacto que esperan los componentes React (data + layout + stats).
# ───────────────────────────────────────────────────────────────────────
class DriveNotConfiguredError(Exception):
    """Drive no montado o variables Argos no configuradas (devuelve 503)."""


def _argos_check_drive() -> None:
    """Verifica que la carpeta de Drive esté disponible."""
    if not os.path.exists("/content/drive/MyDrive"):
        raise DriveNotConfiguredError(
            "Google Drive no configurado. Ejecuta la celda 3 que monta Drive."
        )


# Layout base para Plotly compatible con el tema oscuro del dashboard
_PLOTLY_BASE_LAYOUT = {
    "paper_bgcolor": "rgba(0,0,0,0)",
    "plot_bgcolor":  "rgba(0,0,0,0)",
    "font": {"color": "#cbd5e1", "family": "Space Grotesk, sans-serif"},
    "margin": {"l": 60, "r": 20, "t": 30, "b": 50},
    "xaxis": {"gridcolor": "rgba(255,255,255,0.05)", "zerolinecolor": "rgba(255,255,255,0.1)"},
    "yaxis": {"gridcolor": "rgba(255,255,255,0.05)", "zerolinecolor": "rgba(255,255,255,0.1)"},
}


def _argos_chart_precios_regionales() -> Dict[str, Any]:
    _argos_check_drive()
    df = _argos_load_csv("precios_regionales")
    if df is None:
        raise FileNotFoundError(
            "CSV no encontrado: argos_precios_regionales.csv. "
            "Ejecuta la celda Argos del bot al menos una vez."
        )
    # Esperamos columnas: regional, mu_h (o precio_mu)
    col_precio = "mu_h" if "mu_h" in df.columns else (
        "precio_mu" if "precio_mu" in df.columns else None
    )
    col_reg = "regional" if "regional" in df.columns else (
        "region" if "region" in df.columns else None
    )
    if col_precio is None or col_reg is None:
        raise ValueError(
            f"CSV argos_precios_regionales sin columnas esperadas. "
            f"Columnas disponibles: {list(df.columns)}"
        )
    agg = df.groupby(col_reg)[col_precio].mean().sort_values(ascending=True)
    layout = dict(_PLOTLY_BASE_LAYOUT)
    layout["xaxis"] = {**_PLOTLY_BASE_LAYOUT["xaxis"], "title": "Precio medio (COP)"}
    layout["yaxis"] = {**_PLOTLY_BASE_LAYOUT["yaxis"], "title": ""}
    layout["height"] = 260
    return {
        "data": [{
            "type": "bar",
            "orientation": "h",
            "x": [float(v) for v in agg.values],
            "y": [str(k) for k in agg.index],
            "marker": {"color": "#34d399"},
            "hovertemplate": "%{y}: $%{x:,.0f}<extra></extra>",
        }],
        "layout": layout,
        "stats": {
            "n_regionales": int(len(agg)),
            "precio_promedio": float(agg.mean()) if len(agg) else 0.0,
        },
    }


def _argos_chart_intervalos_hdi(cod_municipio: Optional[str]) -> Dict[str, Any]:
    _argos_check_drive()
    df = _argos_load_csv("intervalos_hdi")
    if df is None:
        raise FileNotFoundError("CSV no encontrado: argos_intervalos_hdi.csv")
    # Columnas esperadas: cod_municipio, nombre_municipio, hdi_lower, hdi_upper, precio_mu
    needed = ["cod_municipio", "hdi_lower", "hdi_upper", "precio_mu"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(
                f"Columna '{c}' faltante en argos_intervalos_hdi.csv. "
                f"Columnas disponibles: {list(df.columns)}"
            )
    work = df.copy()
    if cod_municipio:
        work = work[work["cod_municipio"].astype(str) == str(cod_municipio)]
    else:
        # Top 30 por amplitud de intervalo (más interesante visualmente)
        work["_amp"] = work["hdi_upper"] - work["hdi_lower"]
        work = work.sort_values("_amp", ascending=False).head(30)
    # Etiquetas
    if "nombre_municipio" in work.columns:
        labels = work["nombre_municipio"].astype(str).tolist()
    else:
        labels = work["cod_municipio"].astype(str).tolist()

    layout = dict(_PLOTLY_BASE_LAYOUT)
    layout["xaxis"] = {**_PLOTLY_BASE_LAYOUT["xaxis"], "title": "Precio (COP)"}
    layout["yaxis"] = {**_PLOTLY_BASE_LAYOUT["yaxis"], "title": "", "automargin": True}
    layout["height"] = 320
    return {
        "data": [
            {
                "type": "scatter",
                "mode": "markers",
                "x": work["precio_mu"].astype(float).tolist(),
                "y": labels,
                "error_x": {
                    "type": "data",
                    "symmetric": False,
                    "array": (work["hdi_upper"] - work["precio_mu"]).astype(float).tolist(),
                    "arrayminus": (work["precio_mu"] - work["hdi_lower"]).astype(float).tolist(),
                    "color": "rgba(96, 165, 250, 0.5)",
                    "thickness": 1.5,
                    "width": 4,
                },
                "marker": {"color": "#60a5fa", "size": 8},
                "name": "HDI 94%",
                "hovertemplate": "%{y}<br>$%{x:,.0f}<extra></extra>",
            }
        ],
        "layout": layout,
    }


def _argos_chart_perfiles_alertas() -> Dict[str, Any]:
    _argos_check_drive()
    df = _argos_load_csv("perfiles_alertas")
    if df is None:
        raise FileNotFoundError("CSV no encontrado: argos_perfiles_alertas.csv")
    needed = ["cod_municipio", "perfil", "alerta"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(
                f"Columna '{c}' faltante en argos_perfiles_alertas.csv. "
                f"Columnas disponibles: {list(df.columns)}"
            )
    # Donut por perfil
    counts = df["perfil"].value_counts()
    palette = ["#34d399", "#60a5fa", "#fbbf24", "#f87171", "#a78bfa"]
    donut_layout = dict(_PLOTLY_BASE_LAYOUT)
    donut_layout["height"] = 320
    donut_layout["showlegend"] = True
    donut_layout["legend"] = {"orientation": "v", "x": 1.05, "y": 0.5}
    donut = {
        "data": [{
            "type": "pie",
            "labels": counts.index.tolist(),
            "values": [int(v) for v in counts.values],
            "hole": 0.55,
            "marker": {"colors": palette[: len(counts)]},
            "textinfo": "percent",
            "hovertemplate": "%{label}: %{value} (%{percent})<extra></extra>",
        }],
        "layout": donut_layout,
    }
    # Tabla ordenada por criticidad de alerta
    def _alerta_rank(a):
        s = str(a)
        if "🔴" in s: return 0
        if "🟡" in s: return 1
        if "🟢" in s: return 2
        return 3
    tabla_df = df.copy()
    tabla_df["_rank"] = tabla_df["alerta"].apply(_alerta_rank)
    if "nombre_municipio" not in tabla_df.columns:
        tabla_df["nombre_municipio"] = tabla_df["cod_municipio"].astype(str)
    tabla_df = tabla_df.sort_values(["_rank", "nombre_municipio"])
    tabla = tabla_df[["cod_municipio", "nombre_municipio", "perfil", "alerta"]] \
        .astype(str).to_dict(orient="records")
    # Stats
    rojas    = int((df["alerta"].astype(str).str.contains("🔴")).sum())
    amarillas = int((df["alerta"].astype(str).str.contains("🟡")).sum())
    verdes   = int((df["alerta"].astype(str).str.contains("🟢")).sum())
    return {
        "donut": donut,
        "tabla": tabla,
        "stats": {
            "n_municipios": int(len(df)),
            "rojas": rojas,
            "amarillas": amarillas,
            "verdes": verdes,
        },
    }


def _argos_chart_mapa() -> Dict[str, Any]:
    _argos_check_drive()
    df = _argos_load_csv("perfiles_alertas")
    if df is None:
        raise FileNotFoundError("CSV no encontrado: argos_perfiles_alertas.csv")
    # Necesitamos lat/lon. Si no están, devolvemos un placeholder.
    if "lat" not in df.columns or "lon" not in df.columns:
        # Mapa vacío pero válido para que el front no rompa
        layout = dict(_PLOTLY_BASE_LAYOUT)
        layout["height"] = 480
        layout["mapbox"] = {
            "style": "open-street-map",
            "center": {"lat": 4.6, "lon": -74.1},
            "zoom": 4.5,
        }
        layout["annotations"] = [{
            "text": "Sin coordenadas en CSV. Añade columnas lat/lon a perfiles_alertas.",
            "xref": "paper", "yref": "paper",
            "x": 0.5, "y": 0.5, "showarrow": False,
            "font": {"color": "#94a3b8"},
        }]
        return {
            "data": [],
            "layout": layout,
            "stats": {"n_puntos": 0, "n_sin_geocod": int(len(df))},
        }

    df_geo = df.dropna(subset=["lat", "lon"]).copy()
    color_map = {"🔴": "#ef4444", "🟡": "#eab308", "🟢": "#22c55e"}
    df_geo["_color"] = df_geo["alerta"].astype(str).apply(
        lambda s: next((v for k, v in color_map.items() if k in s), "#94a3b8")
    )
    if "nombre_municipio" not in df_geo.columns:
        df_geo["nombre_municipio"] = df_geo["cod_municipio"].astype(str)
    layout = dict(_PLOTLY_BASE_LAYOUT)
    layout["height"] = 480
    layout["mapbox"] = {
        "style": "open-street-map",
        "center": {
            "lat": float(df_geo["lat"].mean()),
            "lon": float(df_geo["lon"].mean()),
        },
        "zoom": 4.5,
    }
    layout["margin"] = {"l": 0, "r": 0, "t": 0, "b": 0}
    return {
        "data": [{
            "type": "scattermapbox",
            "lat": df_geo["lat"].astype(float).tolist(),
            "lon": df_geo["lon"].astype(float).tolist(),
            "text": df_geo["nombre_municipio"].astype(str).tolist(),
            "marker": {
                "size": 10,
                "color": df_geo["_color"].tolist(),
                "opacity": 0.85,
            },
            "hovertemplate": "%{text}<extra></extra>",
        }],
        "layout": layout,
        "stats": {
            "n_puntos": int(len(df_geo)),
            "n_sin_geocod": int(len(df) - len(df_geo)),
        },
    }


print("✅ WebhookServerExtendido y endpoints API listos")
print("   Rutas registradas: /api/bot/*, /api/config/*, /api/broadcasts/*, /api/argos/*")


## CELDA 13: INICIALIZAR Y EJECUTAR BOT (celda bloqueante)

In [59]:
try:
    # Arrancar el dispatcher ANTES del server: cualquier mensaje que llegue
    # al webhook se encolará y el worker ya estará vivo para procesarlo.
    dispatcher.start()

    # ✅ v1.4.3: rehidratar mensajes huérfanos del reinicio anterior.
    # Si el bot se reinició en medio de una ventana de listen_window, hay
    # filas en BD con respuesta_ia=NULL que nunca recibieron respuesta.
    # - Las de los últimos 15 min se reagendan (el cliente las verá llegar).
    # - Las de hace 15-120 min se marcan como '[RESPUESTA PERDIDA]'
    #   para que dejen de aparecer NULL en reportes (demasiado tarde para
    #   responder, el cliente probablemente ya se fue).
    try:
        stats = processor.rehidratar_inbounds_huerfanos(
            ventana_minutos_responder=15,
            ventana_minutos_marcar=120,
        )
        if stats["rehidratados"] or stats["marcados_perdidos"]:
            print(
                f"♻️  Rehidratación: {stats['rehidratados']} reagendados, "
                f"{stats['marcados_perdidos']} marcados como perdidos"
            )
    except Exception as _rh_err:
        # La rehidratación NUNCA debe bloquear el arranque del bot.
        logger.error(f"Rehidratación falló: {_rh_err}")

    server = WebhookServerExtendido(wa_config, processor, SECRETS['NGROK_AUTH_TOKEN'])
    print("\\n🎯 Iniciando bot + dashboard API con Claude de Anthropic...\\n")
    server.start(port=5000)
except KeyboardInterrupt:
    print("\\n👋 Bot detenido")
    try:
        dispatcher.stop()
    except Exception:
        pass
    try:
        ngrok.kill()
    except Exception:
        pass
except Exception as e:
    print(f"\\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

\n🎯 Iniciando bot + dashboard API con Claude de Anthropic...\n

🌐 CONFIGURACIÓN DEL WEBHOOK
URL: https://stunner-upside-nutty.ngrok-free.dev/webhook
Token de verificación: Eafit2026*
Modelo IA: Claude (Anthropic)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:45] "OPTIONS /api/bot/status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:45] "GET /api/bot/status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:45] "OPTIONS /api/bot/status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:45] "OPTIONS /api/bot/logs?limit=200 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:45] "GET /api/bot/status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:45] "GET /api/bot/logs?limit=200 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:57] "OPTIONS /api/bot/status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:19:57] "GET /api/bot/status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 17:20:00] "OPTIONS 

## CELDA 14: SISTEMA ARGOS (mismo del original, mantiene toda la funcionalidad de análisis de precios)

In [ ]:
# =============================================================================
# CELDA ARGOS: SISTEMA DE INTELIGENCIA COMPETITIVA DE PRECIOS
# Integrado con BotV5.2 — Usa el engine de DatabaseManager y los SECRETS de Colab
# Incluye persistencia incremental en 3 archivos CSV
# =============================================================================
import os
import uuid
import pandas as pd
import numpy as np
import pymc as pm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import sqlalchemy as sa

# ─── DEPENDENCIAS EXTRA (instalar si no están) ──────────────────────────────
try:
    import pymc as pm
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "pymc", "-q"])
    import pymc as pm

try:
    from sklearn.cluster import KMeans
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "scikit-learn", "-q"])
    from sklearn.cluster import KMeans

# ─── REUTILIZAR EL ENGINE YA CREADO POR DatabaseManager ────────────────────
try:
    engine = db_manager.engine
    print("✅ Usando engine de DatabaseManager (conexión ya establecida)")
except NameError:
    raise RuntimeError(
        "❌ \'db_manager\' no está definido. "
        "Ejecuta primero las celdas 1–12 del BotV5.2 antes de correr esta celda."
    )

# ─── RUTAS DE LOS CSV DE PERSISTENCIA ────────────────────────────────────────
CSV_PRECIOS_REGIONALES = "/content/drive/MyDrive/argos_precios_regionales.csv"   # mu_h por producto-región
CSV_INTERVALOS_HDI     = "/content/drive/MyDrive/argos_intervalos_hdi.csv"        # HDI + precios_mu por municipio
CSV_PERFILES_ALERTAS   = "/content/drive/MyDrive/argos_perfiles_alertas.csv"      # cluster + alerta por municipio


# =============================================================================
# FUNCIONES DE PERSISTENCIA INCREMENTAL
# =============================================================================
def guardar_csv_incremental(nuevos_df: pd.DataFrame, ruta: str, nombre_tabla: str):
    """
    Decide si crear el CSV (no existe) o hacer append respetando los registros previos.
    Cada registro nuevo recibe un UUID único en la columna \'id\'.
    """
    if "id" not in nuevos_df.columns:
        nuevos_df = nuevos_df.copy()
        nuevos_df.insert(0, "id", [str(uuid.uuid4()) for _ in range(len(nuevos_df))])

    if os.path.exists(ruta):
        existente = pd.read_csv(ruta)
        combinado = pd.concat([existente, nuevos_df], ignore_index=True)
        combinado.to_csv(ruta, index=False)
        print(f"   📝 {nombre_tabla}: APPEND → {len(nuevos_df)} nuevos | total: {len(combinado)} filas")
    else:
        nuevos_df.to_csv(ruta, index=False)
        print(f"   🆕 {nombre_tabla}: CREATE → {len(nuevos_df)} filas iniciales")


# =============================================================================
# SISTEMA INTEGRAL ARGOS
# =============================================================================
def ejecutar_sistema_integral_argos():
    """
    Sistema de Inteligencia Competitiva de Precios para Argos.
    Combina Muestreo Estratificado + Bayes Jerárquico + K-Means.
    Genera/actualiza 3 archivos CSV de salida.
    """
    print("🚀 Iniciando Sistema Integral Argos: Muestreo + Bayes + Clustering")

    # ─── 1. MUESTREO ESTRATIFICADO ───────────────────────────────────────────
    print("\n📌 Paso 1: Configurando Muestreo Estratificado...")
    with engine.connect() as conn:
        query_vista = """
        CREATE OR REPLACE VIEW ferreterias_muestra AS
        SELECT DISTINCT ON (g.regional) f.*
        FROM   ferreterias f
        JOIN   geografia   g ON f.cod_municipio = g.cod_municipio
        WHERE  f.estado NOT IN (\'terminado\', \'sin_respuesta\')
        ORDER  BY g.regional, RANDOM();
        """
        try:
            conn.execute(sa.text(query_vista))
            conn.commit()
            print("   ✅ Vista \'ferreterias_muestra\' creada/actualizada")
        except Exception as e:
            print(f"   ⚠️  No se pudo crear la vista: {e}")

    # ─── 2. EXTRACCIÓN DE DATOS (incluye id_producto e id_ferreteria) ──────
    print("\n📌 Paso 2: Extrayendo datos de COTIZACIONES + FERRETERIAS...")
    query_raw = """
    SELECT
        c.id_cotizacion,
        c.id_ferreteria,
        c.id_producto,
        c.precio,
        c.confianza_extraccion,
        c.disponibilidad,
        c.regional,
        f.cod_municipio,
        g.nombre_municipio AS ciudad,
        m.nombre_marca
    FROM   cotizaciones        c
    JOIN   ferreterias         f ON c.id_ferreteria = f.id_ferreteria
    JOIN   geografia           g ON f.cod_municipio = g.cod_municipio
    LEFT JOIN marcas_productos m ON c.id_marca      = m.id_marca;
    """
    df = pd.read_sql(query_raw, engine)

    if df.empty:
        print("   ⚠️  No hay datos en COTIZACIONES aún. El proceso se detiene aquí.")
        return

    tasa_exito    = df["precio"].notnull().mean()
    confianza_llm = df["confianza_extraccion"].mean()
    print(f"   📊 Tasa Éxito: {tasa_exito:.2%}  |  Confianza LLM: {confianza_llm:.2f} (meta ≥ 0.80)")

    # ─── 3. MODELO BAYESIANO JERÁRQUICO POR PRODUCTO × REGIONAL ─────────────
    print("\n📌 Paso 3: Calculando Precios Estimados (μ̂) con PyMC...")
    df_clean = df[df["confianza_extraccion"] >= 0.80].dropna(subset=["precio"]).copy()

    if df_clean.empty:
        print("   ⚠️  Sin datos con confianza ≥ 0.80. Aumenta el corpus de cotizaciones.")
        return

    # Crear índice combinado producto × regional
    df_clean["grupo"] = df_clean["id_producto"].astype(str) + "|" + df_clean["regional"]
    grupos = df_clean["grupo"].unique()
    grupo_idx = pd.Categorical(df_clean["grupo"]).codes

    with pm.Model() as modelo:
        mu_0  = pm.Normal("mu_0",  mu=30_000, sigma=5_000)
        tau   = pm.HalfNormal("tau",   sigma=5_000)
        sigma = pm.HalfNormal("sigma", sigma=2_000)
        mu_h  = pm.Normal("mu_h", mu=mu_0, sigma=tau, shape=len(grupos))
        _ = pm.Normal("y_obs", mu=mu_h[grupo_idx], sigma=sigma,
                      observed=df_clean["precio"].astype(float))
        trace = pm.sample(1000, chains=2, progressbar=True, target_accept=0.95,
                          return_inferencedata=True)

    precios_mu = trace.posterior["mu_h"].mean(dim=["chain", "draw"]).values

    # ── Extracción robusta del HDI (compatible con distintas versiones de PyMC/ArviZ)
    import arviz as az
    hdi_ds = az.hdi(trace.posterior, hdi_prob=0.95)
    # hdi_ds["mu_h"] tiene dim ("mu_h_dim_0", "hdi") con coords ["lower","higher"]
    hdi_arr = hdi_ds["mu_h"].values  # shape (n_grupos, 2)
    if hdi_arr.shape[-1] != 2:
        hdi_arr = hdi_arr.T  # fallback si quedó (2, n_grupos)
    hdi_vals = hdi_arr  # ahora siempre (n, 2)

    # ─── 4. CSV #1: PRECIOS REGIONALES (mu_h) POR PRODUCTO ──────────────────
    print("\n📌 Paso 4: Persistiendo CSV de precios regionales por producto...")
    df_precios = pd.DataFrame({
        "id_producto": [g.split("|")[0] for g in grupos],
        "mu_h":        precios_mu,
        "regional":    [g.split("|")[1] for g in grupos],
    })
    guardar_csv_incremental(df_precios, CSV_PRECIOS_REGIONALES,
                            "argos_precios_regionales")

    # ─── 5. CSV #2: HDI + PRECIOS_MU POR MUNICIPIO ───────────────────────────
    print("\n📌 Paso 5: Persistiendo CSV de intervalos HDI por municipio...")
    # cod_municipio se obtiene relacionando cotización → ferretería
    # Tomamos el cod_municipio MÁS frecuente del grupo (producto × regional)
    df_clean["grupo_idx"] = grupo_idx
    municipio_por_grupo = (df_clean
                           .groupby("grupo_idx")["cod_municipio"]
                           .agg(lambda s: s.mode().iloc[0])
                           .reindex(range(len(grupos)))
                           .values)

    df_hdi = pd.DataFrame({
        "cod_municipio": municipio_por_grupo,
        "precios_mu":    precios_mu,
        "hdi_lower":     hdi_vals[:, 0],
        "hdi_upper":     hdi_vals[:, 1],
        "hdi_vals":      [f"[{hdi_vals[i, 0]:.2f}, {hdi_vals[i, 1]:.2f}]" for i in range(len(hdi_vals))],
    })
    guardar_csv_incremental(df_hdi, CSV_INTERVALOS_HDI, "argos_intervalos_hdi")

    # ─── 6. CLUSTERING K-MEANS Y ALERTAS ─────────────────────────────────────
    print("\n📌 Paso 6: Segmentando con K-Means y generando alertas...")
    scaler       = StandardScaler()
    precios_norm = scaler.fit_transform(precios_mu.reshape(-1, 1))
    k            = min(3, len(grupos))
    kmeans       = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters     = kmeans.fit_predict(precios_norm)

    perfiles_map = {0: "Paridad", 1: "Argos Dominante", 2: "Competencia por encima"}

    def alerta_para(precio_est, lo, hi):
        if (hi - lo) > 2500:
            return "🟡 Región con datos insuficientes"
        elif precio_est > 32_500:
            return "🔴 Precio competidor por debajo de Argos"
        else:
            return "✅ Estable"

    # ─── 7. CSV #3: PERFILES + ALERTAS POR MUNICIPIO ─────────────────────────
    print("\n📌 Paso 7: Persistiendo CSV de perfiles y alertas por municipio...")
    df_perfiles = pd.DataFrame({
        "cod_municipio": municipio_por_grupo,
        "perfiles":      [perfiles_map.get(c, "Análisis Exploratorio") for c in clusters],
        "alerta":        [alerta_para(precios_mu[i], hdi_vals[i, 0], hdi_vals[i, 1])
                          for i in range(len(grupos))],
    })
    guardar_csv_incremental(df_perfiles, CSV_PERFILES_ALERTAS,
                            "argos_perfiles_alertas")

    # ─── 8. RESUMEN FINAL ────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("✅ PROCESO ARGOS COMPLETADO")
    print("="*60)
    resumen = pd.DataFrame({
        "Regional":       [g.split("|")[1] for g in grupos],
        "Producto":       [g.split("|")[0][:8] + "..." for g in grupos],
        "Municipio":      municipio_por_grupo,
        "Precio_μ̂":      precios_mu.round(2),
        "Perfil":         [perfiles_map.get(c, "?") for c in clusters],
        "Alerta":         [alerta_para(precios_mu[i], hdi_vals[i, 0], hdi_vals[i, 1])
                           for i in range(len(grupos))],
    })
    print(resumen.to_string(index=False))

    print(f"\n📂 Archivos generados/actualizados:")
    print(f"   • {CSV_PRECIOS_REGIONALES}")
    print(f"   • {CSV_INTERVALOS_HDI}")
    print(f"   • {CSV_PERFILES_ALERTAS}")

    return resumen


# ─── EJECUTAR ─────────────────────────────────────────────────────────────────
reporte_argos = ejecutar_sistema_integral_argos()


---

## 📝 NOTAS DE LA INTEGRACIÓN — v1.4.2

### ✅ Cambios v1.4.2 sobre v1.4.1 — coalescencia de mensajes inbound

**Problema resuelto.** Si una ferretería enviaba dos mensajes seguidos
(ej. "hola, necesito cemento" + "20 bultos por favor"), el bot generaba
DOS respuestas independientes, cada una sin ver la otra, y las enviaba
con pocos segundos de diferencia. Resultado: dos respuestas inconexas,
doble consumo de tokens, y conversación que sonaba robótica.

**Solución.** Ventana única de "escucha activa" (`listen_window_*`,
default 2–5 min) que simula a una persona leyendo varios mensajes antes
de responder. Cada mensaje nuevo de la misma ferretería REINICIA el
timer (debounce). Cuando la ferretería deja de escribir y vence el
timer, la IA genera UNA sola respuesta usando TODOS los mensajes del
burst como contexto, y la envía inmediatamente (sin delay adicional).

**Cambios concretos:**

1. **`DispatcherConfig` rediseñado**:
   - `intra_chat_min_s` / `intra_chat_max_s` → **eliminados**.
   - `listen_window_min_s=120` / `listen_window_max_s=300` → **nuevo**
     (ventana de debounce para inbound).
   - `outreach_delay_min_s=60` / `outreach_delay_max_s=300` → **nuevo**
     (delay aleatorio antes del primer envío proactivo, evita que
     todas las ferreterías reciban el mensaje en el mismo segundo).
   - `inter_chat_*` → sin cambios (anti-bloqueo Meta entre destinatarios).

2. **`MessageDispatcher` con tres tipos de payload**:
   - `kind='text'`: envío directo, sin delay artificial (la espera ya la
     dio la ventana de escucha).
   - `kind='template'`: outreach, aplica `outreach_delay_*`.
   - `kind='inbound_intent'` **nuevo**: dispara un callback cuando vence,
     que ejecuta la generación IA y encola el `text` resultante.
   - `cancel_inbound_intent_for(recipient)` **nuevo**: cancela intents
     pendientes para un destinatario (usado al recibir un mensaje nuevo).
   - `set_inbound_intent_callback(cb)` **nuevo**: registra el callback
     desde el bootstrap.

3. **`DatabaseManager` con dos métodos auxiliares**:
   - `guardar_mensaje_usuario(...)`: INSERT con `respuesta_ia=NULL`.
     Se llama al recibir el mensaje, para que aparezca en el historial
     que verán los mensajes siguientes del burst.
   - `actualizar_respuesta_ia(...)`: UPDATE de la respuesta cuando
     finalmente la IA genera el texto.
   - `obtener_historial_reciente` no necesitó cambios: ya filtraba
     correctamente las filas con `respuesta_ia=NULL` (omite el turno
     'assistant' sin contenido, conserva el 'user').

4. **`MessageProcessor` con buffer por recipient**:
   - `_pending_messages: Dict[str, List[{text, interaction_id}]]`
     protegido por `_pending_lock`. Acumula los mensajes del burst en
     memoria mientras la BD ya tiene cada uno persistido.
   - `_encolar_mensaje_inbound(recipient, text)` **nuevo**: valida
     ferretería, persiste mensaje, apila en buffer, cancela intent
     anterior y encola uno nuevo.
   - `_procesar_inbound_intent(recipient)` **nuevo**: callback que
     vacía el buffer al vencer la ventana y dispara el flujo IA con
     todos los mensajes concatenados.
   - `_ejecutar_pasos_2_a_6` ahora acepta `interaction_id_pendiente`:
     en lugar de hacer INSERT cuando viene este parámetro, hace UPDATE
     sobre la fila del último mensaje del burst. Sin duplicación.

5. **Adjuntos NO se coalescen.** Las imágenes y PDFs siguen yendo por
   `_handle_attachment` directamente, sin pasar por el buffer. Razón:
   los adjuntos disparan extracción de cotizaciones y transiciones de
   estado críticas (`inicio → cotizacion → cierre`); meterlos en una
   ventana de 3 min retrasaría señales operativas. Solo el texto inbound
   se coalesce.

6. **Outreach SIN cambios funcionales.** Sigue usando
   `enqueue_template`, ahora con `outreach_delay_*` (mismo rango 60–300s
   que antes tenía como `intra_chat_*`). No tiene sentido coalescer
   outreach: es proactivo unidireccional, no responde a nada.

### 🧪 Cómo probar manualmente

Desde una celda de Colab, después del bootstrap:

```python
# Simula 3 mensajes seguidos de la misma ferretería
processor._encolar_mensaje_inbound("+573001234567", "hola")
processor._encolar_mensaje_inbound("+573001234567", "necesito cemento")
processor._encolar_mensaje_inbound("+573001234567", "20 bultos Argos")
# Esperar 3-5 min. La ferretería recibirá UNA sola respuesta que
# considera los 3 mensajes.
```

Para acelerar el test, bajar temporalmente la ventana:

```python
dispatcher_config.listen_window_min_s = 5
dispatcher_config.listen_window_max_s = 5
```

---

## 📝 NOTAS DE LA INTEGRACIÓN — v1.1

### ✅ Cambios v1.1 sobre v1.0

**1. `_handle_ai_flow` polimórfico (interpretación A confirmada)**

El método ahora acepta un parámetro `modo`:

- **`modo="inbound"`**: comportamiento clásico (reactivo desde webhook).
  Recibe `(recipient, message)`, busca la ferretería por teléfono, valida veto,
  ejecuta los pasos 2–6 con el mensaje del cliente real.

- **`modo="outreach"`**: nuevo flujo proactivo. Recibe `topic`. Itera ferreterías
  con `estado IS NULL` y no vetadas, construye `user_message = f"Genera un saludo informativo sobre: {topic}"`,
  ejecuta los pasos 2–6, y al final transiciona None → primer_mensaje.
  El extractor se omite (el user_message es sintético).

Internamente ambos modos convergen en `_ejecutar_pasos_2_a_6()`, que es
el método compartido. Esto evita duplicación de código.

**2. `OperatingHoursGate` — ventana horaria configurable (TZ Bogotá)**

Nueva clase en celda 4 (configs). Ventana por día de semana usando un dict
`{weekday_int: (hora_apertura, hora_cierre)}`. Métodos:
- `is_open(now=None) -> bool`
- `next_open(now=None) -> datetime`  *(devuelve aware en TZ Bogotá)*
- `closes_at(now=None) -> Optional[datetime]`  *(devuelve aware en TZ Bogotá)*

⚠️ **Zona horaria fija: `America/Bogota` (UTC-5, sin DST).** Las horas que
configuras en `windows` se interpretan como hora LOCAL DE BOGOTÁ. Esto es
crítico: Colab corre en UTC, y sin TZ explícita el gate evaluaría 5 horas
desplazado (a las 17:00 Bogotá un mensaje sería rechazado por estar
"fuera de ventana 8-18" cuando el reloj UTC marca las 22:00).

**Convención de inputs**:
- Si pasas un `datetime` *aware*, se respeta su tzinfo y se convierte a Bogotá.
- Si pasas un `datetime` *naive* (sin tzinfo), se ASUME UTC (el caso típico de
  `datetime.now()` en Colab) y se convierte a Bogotá.
- Si pasas `None`, el gate llama `datetime.now(tz=BOGOTA_TZ)` directamente.

**Default sugerido en bootstrap**: lun–vie 8–18, sáb 9–13, dom cerrado.

**APScheduler también usa TZ Bogotá**: `BackgroundScheduler(timezone=BOGOTA_TZ)`,
así los `CronTrigger(day_of_week='mon', hour=17, minute=8)` se interpretan
como 17:08 hora Bogotá.

**3. Tres puntos de integración del gate**

| Punto | Decisión | Comportamiento |
|---|---|---|
| Webhook entrante | **1a estricto** | `process_incoming` descarta el mensaje completo si `not gate.is_open()`. Cliente escribe fuera de horario → bot no guarda interacción ni responde. |
| Cron de outreach | **2a + 2c** | El cron se sigue agendando con `schedule_weekly_broadcast()`, pero `_run_broadcast_job` chequea el gate al inicio y aborta si está fuera. |
| Dispatcher | **3a** | Antes de cada envío físico, si el bot está fuera de ventana, el mensaje se requeue con `send_at = next_open()` en lugar de descartarse. |

**4. `BroadcastScheduler._run_broadcast_job` simplificado**

Antes contenía la lógica de iterar candidatas, generar mensajes y persistir
interacciones. Ahora es un wrapper de ~10 líneas que:
1. Chequea el gate horario.
2. Delega en `processor._handle_ai_flow(modo="outreach", topic=...)`.

Toda la lógica de IA vive ahora en el processor, manteniendo la separación
de responsabilidades: el scheduler solo agenda y dispara, el processor procesa.

**5. Bootstrap (celda 11)**

Se añadió la creación de `OperatingHoursConfig` + `OperatingHoursGate`,
y se inyecta `hours_gate=` en `MessageDispatcher`, `MessageProcessor` y
`BroadcastScheduler`. El `BroadcastScheduler` también recibe `processor=`
para poder delegarle el outreach.

### 🔧 Cómo cambiar la ventana operativa

Editar el `OperatingHoursConfig` en la celda 11 (bootstrap):

```python
operating_hours_config = OperatingHoursConfig(windows={
    0: (8, 18),    # lunes 08:00–18:00
    1: (8, 18),    # martes
    2: (8, 18),    # miércoles
    3: (8, 18),    # jueves
    4: (8, 18),    # viernes
    5: (9, 13),    # sábado 09:00–13:00
    6: None,       # domingo cerrado
})
```

- Las horas son enteros 0..24.
- Para "abierto 24h": usar `(0, 24)`.
- Para "cerrado todo el día": usar `None` (no la tupla).

### 🧪 Cómo probar manualmente el outreach

Desde una celda de Colab, después del bootstrap:

```python
processor._handle_ai_flow(modo="outreach", topic="precios de cemento esta semana")
```

Esto disparará el flujo completo, IGUAL que como lo haría el cron, pero
**no chequea el gate** (el chequeo lo hace el wrapper del scheduler). Útil
para tests manuales fuera de horario.


In [ ]:
# DIAGNÓSTICO: llama al endpoint desde el propio Colab y muestra el error real
import requests, traceback

print("=" * 60)
print("Test 1: /health")
print("=" * 60)
r = requests.get("http://localhost:5000/health")
print(f"Status: {r.status_code}")
print(f"Body: {r.text[:500]}")

print("\n" + "=" * 60)
print("Test 2: /api/bot/status")
print("=" * 60)
r = requests.get("http://localhost:5000/api/bot/status")
print(f"Status: {r.status_code}")
print(f"Body: {r.text[:2000]}")

print("\n" + "=" * 60)
print("Test 3: invocar _bot_status_payload() directamente")
print("=" * 60)
try:
    result = _bot_status_payload()
    print("OK:", result)
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
    traceback.print_exc()

In [60]:
import pandas as pd
for name, path in ARGOS_PATHS.items():
    print(f"\n=== {name} ===")
    print(f"path: {path}")
    try:
        df = pd.read_csv(path)
        print(f"columnas: {list(df.columns)}")
        print(f"filas: {len(df)}")
        print("primeras 2 filas:")
        print(df.head(2).to_dict(orient='records'))
    except Exception as e:
        print(f"ERROR: {e}")


=== precios_regionales ===
path: /content/drive/MyDrive/argos_precios_regionales.csv
columnas: ['id', 'id_producto', 'mu_h', 'regional']
filas: 89
primeras 2 filas:
[{'id': 'cd23062b-e9f6-4fb7-8d16-56a518eae50c', 'id_producto': '26fbeb4c-1093-f480-f5d0-0406eda440d1', 'mu_h': 22608.852440704915, 'regional': 'centro'}, {'id': '38e6b0a9-df61-44e1-814e-0b71caf63a35', 'id_producto': 'ca802391-9ec1-9a62-bb8b-4c1410d219b2', 'mu_h': 32718.05548110693, 'regional': 'centro'}]

=== intervalos_hdi ===
path: /content/drive/MyDrive/argos_intervalos_hdi.csv
columnas: ['id', 'cod_municipio', 'precios_mu', 'hdi_lower', 'hdi_upper', 'hdi_vals']
filas: 89
primeras 2 filas:
[{'id': '0cc2206b-acdd-49b3-90b2-6934e0ebf611', 'cod_municipio': 5001, 'precios_mu': 22608.852440704915, 'hdi_lower': 17961.461934267252, 'hdi_upper': 32437.370272039043, 'hdi_vals': '[17961.46, 32437.37]'}, {'id': '2957ecde-ed5e-42a5-aa90-42d6eb35fbf2', 'cod_municipio': 11001, 'precios_mu': 32718.05548110693, 'hdi_lower': 29136.34339